# Stage 1 : Set up project and get required libraries


In [1]:
import sys
import torch
import transformers
import datasets
import sentencepiece
import accelerate
import sklearn

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Stage 2 — Wikipedia Dataset Exploration


In [2]:
# Stage 2.1 — Wikipedia dataset inspection

from datasets import load_dataset

DATASET_NAME = "wikimedia/wikipedia"
DATASET_CONFIG = "20231101.en"

# Stream records instead of downloading the entire dataset

wiki_stream = load_dataset (
    DATASET_NAME,
    DATASET_CONFIG,
    split="train",
    streaming=True
)

# Inspect only the first 3 records
sample_articles = list(wiki_stream.take(3))
print("Number of sampled articles:", len(sample_articles))

for i, article in enumerate(sample_articles, start=1):
    print(f"\n{'=' * 60}")
    print(f"Article {i} ", )
    print(f" {'=' * 60 } ", )
    print("Available fields: ",list(article.keys()) )
    print("Title: ", article.get("title"))
    print("ID: ",  article.get("id"))
    print("URL: ",  article.get("url"))

    text_content = article.get("text") or ""

    print("Text length (characters):", len(text_content))
    print("Text preview:")
    print(text_content[:500])



Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Number of sampled articles: 3

Article 1 
Available fields:  ['id', 'url', 'title', 'text']
Title:  Anarchism
ID:  12
URL:  https://en.wikipedia.org/wiki/Anarchism
Text length (characters): 46064
Text preview:
Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as

Article 2 
Available fields:  ['id', 'url', 'title', 'text']
Title:  Albedo
ID:  39
URL:  https://en.wikipedia.org/wiki/Albedo
Text length (characters): 25076
Text preview:
Albedo (; ) is the fraction of sunlight that is diffusely reflected by a body. It is measured on a scale from 0 (co

### 2.1 Wikipedia Dataset Inspection

**Objective**

To inspect the structure and textual characteristics of the Wikipedia
dataset before constructing the document retrieval system.

**Dataset**

- Source: Wikimedia/Wikipedia (Hugging Face)
- Configuration: `20231101.en`
- Language: English
- Snapshot: November 1, 2023
- Loading method: Streaming
- Initial inspection sample: 3 articles

**Methodology**

The Hugging Face Datasets library was used to access the English
Wikipedia snapshot.

Streaming was enabled to inspect a small number of records without
downloading the complete Wikipedia dataset.

The first three records were examined to identify available fields,
article titles, document identifiers, source URLs, and text lengths.

**Experimental Observations**

| Article   |  ID | Text Length (Characters) |
| --------- | --: | -----------------------: |
| Anarchism |  12 |                   46,064 |
| Albedo    |  39 |                   25,076 |
| A         | 290 |                   10,574 |

Each inspected record contained four fields:

- `id`
- `url`
- `title`
- `text`

All three sampled articles contained non-empty text.

The total text length was 81,714 characters, with a sample mean
of 27,238 characters per article.

**Interpretation**

The inspected Wikipedia articles vary substantially in length.

This suggests that article-level retrieval could provide considerably
more text than is required to answer a specific question.

A passage-based retrieval strategy will therefore be considered,
where articles are divided into smaller searchable chunks.

Article identifiers, titles, and source URLs will be retained
for traceability and source attribution.

**Limitations**

The three records were inspected only to understand the dataset schema
and obtain preliminary text-length observations.

They do not constitute a statistically representative sample of
the complete Wikipedia dataset.

**Conclusion**

The Wikipedia dataset provides the textual content and metadata
required to construct the proposed RAG knowledge base.

The next step is to examine a larger subset before selecting the
final corpus and chunking strategy.


In [3]:
# Stage 2.2 — Wikipedia sample profiling

import pandas as pd

SAMPLE_SIZE = 100

# Retrieve the first 100 articles
wiki_sample = list(wiki_stream.take(SAMPLE_SIZE))

# Create a summary of Dataframe 
wiki_metadata = pd.DataFrame([
    {
        "id": article.get("id"),
        "title": article.get("title"),
        "url": article.get("url"),
        "text_length": len(article.get("text") or ""),
        "is_empty": not bool((article.get("text") or "").strip())
    }
    for article in wiki_sample
])

print("=== WIKIPEDIA SAMPLE PROFILE ===")

print("Number of articles:", len(wiki_metadata))

print("\nMissing/empty text records:")
print(wiki_metadata["is_empty"].sum())

print("\nDuplicate article IDs:")
print(wiki_metadata["id"].duplicated().sum())

print("\nArticle length statistics in count of characters:")
print(wiki_metadata["text_length"].describe())

print("\nFive longest articles:")
print(
    wiki_metadata.nlargest(5, "text_length")[
        ["title", "text_length"]
    ]
)

print("\nFive shortest articles:")
print(
    wiki_metadata.nsmallest(5, "text_length")[
        ["title", "text_length"]
    ]
)

=== WIKIPEDIA SAMPLE PROFILE ===
Number of articles: 100

Missing/empty text records:
0

Duplicate article IDs:
0

Article length statistics in count of characters:
count       100.000000
mean      32432.850000
std       27873.097669
min         389.000000
25%        8352.000000
50%       26925.500000
75%       51424.250000
max      115512.000000
Name: text_length, dtype: float64

Five longest articles:
              title  text_length
27           Apollo       115512
59     Alkali metal       100986
92          Albania        92370
5   Abraham Lincoln        90007
91      Afghanistan        89245

Five shortest articles:
                          title  text_length
75  Asia Minor (disambiguation)          389
94         Algorithms (journal)          930
54    Argument (disambiguation)         1484
45                 Answer (law)         1649
82          Transport in Angola         1958


### 2.2 Wikipedia Corpus Profiling

**Objective**

To characterize Wikipedia article lengths, identify potential data
quality issues, and inform the selection of a manageable corpus
for offline Retrieval-Augmented Generation.

**Methodology**

The first 100 records from the English Wikipedia November 2023
snapshot were inspected.

For each record, article identifiers, titles, URLs, and text lengths
were extracted.

The analysis included descriptive statistics, empty-text checks,
duplicate-ID checks, and inspection of the shortest and longest
articles.

**Experimental Results**

| Statistic          | Character Count |
| ------------------ | --------------: |
| Sample size        |             100 |
| Mean               |       32,432.85 |
| Standard deviation |       27,873.10 |
| Minimum            |             389 |
| 25th percentile    |           8,352 |
| Median             |       26,925.50 |
| 75th percentile    |       51,424.25 |
| Maximum            |         115,512 |

No empty article texts or duplicate article identifiers were
observed in the inspected sample.

**Interpretation**

Article lengths varied substantially, from 389 to 115,512 characters.

The mean article length exceeded the median, suggesting that
relatively long articles influence the sample average.

This variability motivates a passage-based retrieval strategy
rather than passing entire articles directly to the generator.

Disambiguation pages were also observed among the shortest articles.
Such pages may provide limited explanatory evidence for question
answering and will be excluded from the initial corpus.

**Sampling Limitation**

The first 100 records were used for preliminary profiling rather
than random sampling.

Many sampled titles began with the letter A, suggesting ordering
bias. Consequently, these statistics should not be generalized
to the complete Wikipedia dataset.

**Conclusion**

The preliminary analysis supports constructing a locally stored
Wikipedia corpus and subsequently splitting articles into smaller
retrievable passages.

A seeded shuffled selection procedure will be used to improve
topic diversity and support experimental reproducibility.


In [4]:
# Step 2.3.1 — Select and inspect the corpus / # Stage 2.3.1 — Select Wikipedia articles for offline RAG
# The first run may take some time because streaming shuffle must initialize its buffer and access remote data.
# Until here in below code , our dataset has been held in Python memory after streaming it from Hugging Face.
# If you restart the Jupyter kernel, the selected_articles variable disappears. We need to persist the articles.

RANDOM_SEED = 42
TARGET_ARTICLES = 1000
SHUFFLE_BUFFER = 5000
MIN_TEXT_LENGTH = 500

# Load Wikipedia as a streaming dataset
wiki_stream = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True
)

# Shuffle with a reproducible seed
wiki_shuffled = wiki_stream.shuffle(
    seed=RANDOM_SEED,
    buffer_size=SHUFFLE_BUFFER
)

selected_articles =[]
seen_ids = set()

for article in wiki_shuffled:
    article_id = article.get("id")
    title = (article.get("title") or "").strip()
    text = (article.get("text") or "").strip()  

     # Exclude incomplete and very short records
    if not article_id or not title or len(text) < MIN_TEXT_LENGTH:
        continue

     # Exclude clearly identified disambiguation pages
    if "(disambiguation)" in title.lower():
        continue

     # Avoid duplicate article IDs
    if article_id in seen_ids:
        continue

    selected_articles.append({
        "id": article_id,
        "title": title,
        "url": article.get("url"),
        "text": text
    })    

    seen_ids.add(article_id)

    if len(selected_articles) >= TARGET_ARTICLES:
        break

print("=== SELECTED WIKIPEDIA CORPUS ===")
print("Articles selected:", len(selected_articles))
print("Unique IDs:", len(seen_ids))


if selected_articles:
    print("\nFirst 10 sleected titles:")

    for article in selected_articles[:10]:
         print("-", article["title"])

    text_lengths = [len(a["text"]) for a in selected_articles]

    print("\nMinimum article length:", min(text_lengths))
    print("Maximum article length:", max(text_lengths))
    print("Average article length:",round(sum(text_lengths) / len(text_lengths), 2))


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

=== SELECTED WIKIPEDIA CORPUS ===
Articles selected: 1000
Unique IDs: 1000

First 10 sleected titles:
- Diana Richburg
- Die-ner (Get It?)
- You and What Army
- List of Paris Saint-Germain Féminine presidents
- How to Console Widows
- Fencing at the 1952 Summer Olympics – Men's sabre
- Gale Adcock
- James O'Gwynn
- Counsel's Opinion
- Oshikundu

Minimum article length: 500
Maximum article length: 102127
Average article length: 4025.16


In [ ]:
# Stage 2.3.2 — Save Wikipedia corpus locally

from pathlib import Path
from datasets import Dataset

# Use the notebook's currnt working directory 
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "wiki_slice_1000"

print("Project working directory:", PROJECT_ROOT)
print("Target corpus directory:", CORPUS_DIR)

# Protect an existing saved dataset from accidental overwriting

# if CORPUS_DIR.exists():
#     raise FileExistsError(
#         f"Dataset directory already exists:{CORPUS_DIR}"
#     )

if CORPUS_DIR.exists():
    raise FileExistsError(
        f"Dataset directory already exists: {CORPUS_DIR}"
    )

# Ensure the parent data directpry exists 
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Convert selected records into a Hugging face dataset
wiki_dataset = Dataset.from_list(selected_articles)

# Persist the complete selected dataset
wiki_dataset.save_to_disk(str(CORPUS_DIR))

print("\n=== CORPUS SAVED ===")
print("Articles saved:", len(wiki_dataset))
print("Saved to:", CORPUS_DIR.resolve())
print("Fields:", wiki_dataset.column_names)

# output = Saving the dataset (1/1 shards)
## This is expected. It means our much smaller, locally saved dataset was written into one data shard. The original Wikipedia dataset still has its own separate shard structure


Project working directory: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG
Target corpus directory: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\data\wiki_slice_1000


In [7]:
# Stage 2.3.3 — Verify locally saved Wikipedia corpus

from datasets import load_from_disk

# Load from local storage not the Hugging face Hub
wiki_local = load_from_disk(str(CORPUS_DIR))

print("=== LOCAL CORPUS VERIFICATION ===")
print("Articles loaded:", len(wiki_local))
print("Fields:", wiki_local.column_names)

# Compare the locally loaded corpus with our original selection
assert len(wiki_local) == TARGET_ARTICLES, "Article count mismatch!"
assert len(set(wiki_local["id"])) == TARGET_ARTICLES, "Duplicate article IDs detected!"
assert wiki_local[0]["id"] == selected_articles[0]["id"], "First article ID mismatch!"
assert wiki_local[0]["text"] == selected_articles[0]["text"], "First article text mismatch!"

print("\nFirst article:", wiki_local[0]["title"])
print("Last article:", wiki_local[-1]["title"])
print("\nAll local dataset verification checks passed.")


=== LOCAL CORPUS VERIFICATION ===
Articles loaded: 1000
Fields: ['id', 'title', 'url', 'text']

First article: Diana Richburg
Last article: Amore & Vita

All local dataset verification checks passed.


### 2.3 Wikipedia Corpus Selection and Local Persistence

**Objective**

To construct a manageable, reproducible Wikipedia corpus for
offline Retrieval-Augmented Generation and verify its integrity
after local storage.

**Methodology**

The English Wikipedia November 2023 snapshot was accessed using
Hugging Face Datasets in streaming mode.

A seeded streaming shuffle was applied using:

- Random seed: 42
- Shuffle buffer: 5,000 records
- Target corpus size: 1,000 articles
- Minimum text length: 500 characters

Records containing insufficient text, missing identifiers,
duplicate article IDs, or titles explicitly identifying
disambiguation pages were excluded.

The selected articles were converted into a Hugging Face Dataset
and saved to local storage using `save_to_disk()`.

The saved corpus was subsequently reloaded using `load_from_disk()`.

**Experimental Results**

| Metric                 |               Result |
| ---------------------- | -------------------: |
| Articles selected      |                1,000 |
| Unique article IDs     |                1,000 |
| Minimum article length |       500 characters |
| Maximum article length |   102,127 characters |
| Mean article length    |  4,025.16 characters |
| Dataset fields         | id, title, url, text |
| Local dataset shards   |                    1 |

**Local Persistence Verification**

The saved corpus was successfully reloaded from disk.

The reloaded dataset contained all 1,000 records, with
1,000 unique article identifiers.

The first article was "Diana Richburg" and the final article
was "Amore & Vita".

The first article's identifier and text matched the original
in-memory selection.

All programmed integrity assertions passed.

**Interpretation**

The resulting corpus provides a compact, locally accessible
knowledge collection for the proposed RAG system.

Preserving article identifiers, titles and URLs enables
traceability of retrieved passages to their Wikipedia sources.

The local dataset can be accessed without requesting article
content from Hugging Face during subsequent processing.

**Limitations**

The streaming shuffle uses a finite buffer and does not guarantee
a uniformly random sample of the complete Wikipedia dataset.

The corpus may have limited coverage of certain topics.

Only selected integrity conditions were checked; a full
record-by-record comparison was not performed.

Offline execution has been verified for local dataset loading,
but complete offline RAG functionality remains to be tested.

**Conclusion**

A corpus of 1,000 Wikipedia articles was successfully selected,
stored locally, and reloaded.

The corpus is ready for text chunking and indexing.


In [8]:
# Stage 3.1 — Prototype word-based chunking

CHUNK_SIZE = 100
CHUNK_OVERLAP = 20

def chunk_text(text, chunk_size=100, overlap=20):
    if chunk_size <= 0:
        raise ValueError("Chunk size must be postive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError ("Overlap must be non-negative and smaller than chunk size.")
    words = text.split()

    if not words:
        return []

    stride = chunk_size - overlap
    chunks = []

    for start in range(0, len(words), stride):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

    return chunks    


# Use the first article from our locally loaded corpus
sample_article = wiki_local[0]

sample_chunks = chunk_text(
    sample_article["text"],
    chunk_size= CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

print("=== CHUNKING PROTOTYPE ===")

print("Article title:", sample_article["title"])
print("Original word count:", len(sample_article["text"].split()))
print("Chunks created:", len(sample_chunks))

print("\nFirst chunk:")
print(sample_chunks[0][:500])

print("\nLast chunk:")
print(sample_chunks[-1][:500])

print("\nFirst chunk word count:", len(sample_chunks[0].split()))
print("Last chunk word count:", len(sample_chunks[-1].split()))


=== CHUNKING PROTOTYPE ===
Article title: Diana Richburg
Original word count: 225
Chunks created: 3

First chunk:
Diana Lynn Richburg (born July 2, 1963) is an American former middle-distance runner. She competed at the 1984 Los Angeles Olympics, and won the 1985 US national title at 1500 metres. Career Born in Troy, New York, Richburg finished fourth in the 800m at the 1984 US Olympic Trials, before earning selection for the Los Angeles Olympics by finishing third in the 1500m in 4:04.07. At the Olympics, she failed to reach the final. She won the 1500m title at the 1985 USA Outdoor Track and Field Champio

Last chunk:
competitions References 1963 births American female middle-distance runners Living people Olympic track and field athletes for the United States Sportspeople from Troy, New York Track and field athletes from New York (state) Athletes (track and field) at the 1984 Summer Olympics Athletes (track and field) at the 1987 Pan American Games Pan American Games track and field

In [9]:
# Stage 3.1.1 — Verify chunk boundaries and overlap
original_words = sample_article["text"].split()
print("=== CHUNK OVERLAP VERIFICATION ===")

for i in range(len(sample_chunks)):
    chunk_words = sample_chunks[i].split()

    print(f"\nChunk {i + 1}")
    print("Word count:", len(chunk_words))

    if i > 0:
        previous_words = sample_chunks[i-1].split()

        # Last 20 words of previous chunk
        previous_overlap = previous_words[-CHUNK_OVERLAP:]

        # First 20 words of current chunk
        current_overlap = chunk_words[:CHUNK_OVERLAP]

        overlap_matches = previous_overlap == current_overlap

        print("Overlap matches:", overlap_matches)

        assert overlap_matches, (
            f"Overlap mismatch between chunks {i} and {i + 1}"
        )

# Verify that the original text's words can be reconstructed
reconstructed = sample_chunks[0].split()

for chunk in sample_chunks[1:]:
    reconstructed.extend(chunk.split()[CHUNK_OVERLAP:])

assert reconstructed == original_words, \
    "Reconstructed words do not match the original article!"

print("\nAll chunking verification checks passed.")


=== CHUNK OVERLAP VERIFICATION ===

Chunk 1
Word count: 100

Chunk 2
Word count: 100
Overlap matches: True

Chunk 3
Word count: 65
Overlap matches: True

All chunking verification checks passed.


### 3.1 Word-Based Chunking Prototype

**Objective**

To implement and evaluate a simple word-based text chunking strategy
before processing the complete Wikipedia corpus.

**Methodology**

A fixed-size sliding-window chunking algorithm was implemented
using the following parameters:

- Chunk size: 100 words
- Overlap: 20 words
- Stride: 80 words

The algorithm was tested on the Wikipedia article "Diana Richburg".

**Mathematical Formulation**

The stride is defined as:

$$
S=C-O
$$

where:

- $S$ is the stride.
- $C$ is the chunk size.
- $O$ is the overlap.

For the selected configuration:

$$
S=100-20=80
$$

For a document with $N>C$ words, the number of chunks is:

$$
M=1+\left\lceil\frac{N-C}{C-O}\right\rceil
$$

**Experimental Results**

| Parameter                |     Value |
| ------------------------ | --------: |
| Original document length | 225 words |
| Chunk size               | 100 words |
| Overlap                  |  20 words |
| Stride                   |  80 words |
| Number of chunks         |         3 |
| First chunk length       | 100 words |
| Final chunk length       |  65 words |

The observed number of chunks matched the mathematically expected
value of three.

**Interpretation**

The sliding-window approach successfully divided the article
into manageable passages.

Overlapping chunks are intended to preserve contextual information
across boundaries.

The final chunk was shorter than the configured maximum size,
which is expected when the document length is not an exact
multiple of the stride.

The article also contained reference and category-like material,
indicating that additional text-cleaning considerations may
improve the quality of retrieved evidence.

**Limitations**

Word-based chunking does not necessarily respect sentence or
paragraph boundaries.

Furthermore, word counts differ from Transformer token counts.
Consequently, model-specific token limits must be considered
when constructing prompts for Flan-T5-small.


**Chunking Integrity Verification**

The chunking implementation was further evaluated by comparing
the overlapping words between consecutive passages.

The 20-word overlaps between chunks 1–2 and 2–3 matched exactly.

The original article word sequence was reconstructed by combining
the first chunk with the non-overlapping portions of subsequent
chunks.

The reconstruction matched the original whitespace-tokenized
article text.

These results confirm the correctness of the implemented
sliding-window chunking procedure for the tested article.


In [10]:
# Stage 3.2.1 — Chunk all locally stored Wikipedia articles

all_chunks = []

for article in wiki_local:
    article_chunks = chunk_text(
        article["text"],
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )
      
    for chunk_number, chunk in enumerate(article_chunks):
        all_chunks.append({
            "chunk_id": f"{article['id']}::{chunk_number}",
            "article_id": article["id"],
            "title": article["title"],
            "url": article["url"],
            "chunk_number": chunk_number,
            "text": chunk,
            "word_count": len(chunk.split())
        })

# Convert the chunk collection into a DataFrame
chunks_df = pd.DataFrame(all_chunks)    

print("=== WIKIPEDIA CHUNKING RESULTS ===")

print("Original articles:", len(wiki_local))
print("Total chunks created:", len(chunks_df))

print("\nUnique article IDs:")
print(chunks_df["article_id"].nunique())

print("\nUnique chunk IDs:")
print(chunks_df["chunk_id"].nunique())

print("\nChunk word-count statistics:")
print(chunks_df["word_count"].describe())

print("\nChunks per article:")
chunks_per_article = chunks_df.groupby("article_id").size()
print(chunks_per_article.describe())

print("\nFirst five chunks:")
print(
    chunks_df[
        ["chunk_id", "title", "chunk_number", "word_count"]
    ].head()
)

=== WIKIPEDIA CHUNKING RESULTS ===
Original articles: 1000
Total chunks created: 8191

Unique article IDs:
1000

Unique chunk IDs:
8191

Chunk word-count statistics:
count    8191.000000
mean       95.249542
std        15.170073
min        21.000000
25%       100.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: word_count, dtype: float64

Chunks per article:
count    1000.000000
mean        8.191000
std        16.076722
min         1.000000
25%         2.000000
50%         4.000000
75%         8.000000
max       200.000000
dtype: float64

First five chunks:
      chunk_id              title  chunk_number  word_count
0  55617566::0     Diana Richburg             0         100
1  55617566::1     Diana Richburg             1         100
2  55617566::2     Diana Richburg             2          65
3  41018293::0  Die-ner (Get It?)             0         100
4  41018293::1  Die-ner (Get It?)             1         100


### 3.2 Full-Corpus Passage Generation

**Objective**

To transform the locally stored Wikipedia corpus into smaller
retrievable passages while preserving source metadata.

**Methodology**

The validated sliding-window chunking algorithm was applied to
all 1,000 Wikipedia articles.

The chunking configuration was:

- Chunk size: 100 words
- Overlap: 20 words
- Stride: 80 words

For each passage, the following metadata was preserved:

- Unique chunk identifier
- Original Wikipedia article identifier
- Article title
- Article URL
- Chunk position
- Passage text
- Word count

**Experimental Results**

| Metric                     |      Result |
| -------------------------- | ----------: |
| Original articles          |       1,000 |
| Total generated chunks     |       8,191 |
| Unique article IDs         |       1,000 |
| Unique chunk IDs           |       8,191 |
| Mean chunk length          | 95.25 words |
| Minimum chunk length       |    21 words |
| Maximum chunk length       |   100 words |
| Mean chunks per article    |       8.191 |
| Median chunks per article  |           4 |
| Maximum chunks per article |         200 |

**Interpretation**

The chunking procedure successfully transformed the original
articles into 8,191 smaller passages.

Most chunks contained 100 words, while shorter final chunks
accounted for the lower mean length.

All 1,000 source articles were represented in the generated
passage collection, and all chunk identifiers were unique.

The difference between the mean and median number of chunks
per article indicates that some longer articles contribute
substantially more passages than typical articles.

**Limitations**

The word-based approach does not preserve sentence or paragraph
boundaries.

Overlapping passages contain repeated text, increasing the total
indexed word count.

Long articles contribute more candidate passages than short
articles, which may influence retrieval results.

Token lengths may differ from word counts and will require
attention when constructing Flan-T5 input prompts.

**Conclusion**

The Wikipedia corpus has been successfully converted into
8,191 uniquely identified passages suitable for indexing.

The next step is to persist the passage collection and construct
a searchable TF-IDF representation.


In [ ]:
# Stage 3.3.1 — Save chunked Wikipedia passages

CHUNKS_DIR = DATA_DIR / "wiki_chunks_100w_20overlap"

# Avoid overwriting existing saved data
if CHUNKS_DIR.exists():
    raise FileExistsError(
        f"Chunk dataset already exists: {CHUNKS_DIR}"
    )

# Convert DataFrame into a Hugging Face Dataset
chunks_dataset = Dataset.from_pandas(
    chunks_df,
    preserve_index=False
)

# Save the processed passages locally
chunks_dataset.save_to_disk(str(CHUNKS_DIR))

print("=== CHUNK DATASET SAVED ===")

print("Saved chunks:", len(chunks_dataset))
print("Fields:", chunks_dataset.column_names)
print("Save location:", CHUNKS_DIR.resolve())

In [12]:
# Stage 3.3.2 — Verify saved Wikipedia passages

# Load locally saved passages
# The assertions are comparing the loaded dataset (chunks_local) against the original dataset in memory (chunks_df) and related source data (wiki_local)

chunks_local = load_from_disk(str(CHUNKS_DIR))

print("=== LOCAL CHUNK DATASET VERIFICATION ===")

print("Chunks loaded:", len(chunks_local))
print("Fields:", chunks_local.column_names)

# Validate record counts
assert len(chunks_local) == len(chunks_df), "Chunk count mismatch!"

# Validate unique chunk IDs
assert len(set(chunks_local["chunk_id"])) == len(chunks_local), "Duplicate chunk IDs detected!"

# Validate source article coverage
assert len(set(chunks_local["article_id"])) == len(wiki_local), "Some source articles are missing!"

# Validate original passage content and metadata
assert chunks_local[0]["text"] == chunks_df.iloc[0]["text"], "First chunk text mismatch!"

assert chunks_local[0]["title"] == chunks_df.iloc[0]["title"], "First chunk title mismatch!"

assert chunks_local[-1]["text"] == chunks_df.iloc[-1]["text"], "Last chunk text mismatch!"

print("\nFirst chunk ID:", chunks_local[0]["chunk_id"])
print("Last chunk ID:", chunks_local[-1]["chunk_id"])

print("\nAll local chunk dataset verification checks passed.")

=== LOCAL CHUNK DATASET VERIFICATION ===
Chunks loaded: 8191
Fields: ['chunk_id', 'article_id', 'title', 'url', 'chunk_number', 'text', 'word_count']

First chunk ID: 55617566::0
Last chunk ID: 28820436::15

All local chunk dataset verification checks passed.


In [13]:
# Stage 3.3.3 — Verify reusable chunking module

import sys
from pathlib import Path

# Identify our project root
PROJECT_ROOT = Path.cwd()

# Location of reusable Python modules
SRC_DIR = PROJECT_ROOT / "src"

# Confirm the expected module file exists
assert (SRC_DIR / "rag" / "chunking.py").is_file(), \
    f"Cannot find chunking.py under {SRC_DIR}"

# Make the src directory importable
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import our new reusable function
from rag.chunking import chunk_text as modular_chunk_text

print("=== MODULE IMPORT VERIFICATION ===")

print("Project root:", PROJECT_ROOT)
print("Module location:", SRC_DIR / "rag" / "chunking.py")

# Execute the modular function on our original test article
modular_chunks = modular_chunk_text(
    sample_article["text"],
    chunk_size=100,
    overlap=20
)

print("\n=== CHUNKING COMPARISON ===")

print("Original function chunks:", len(sample_chunks))
print("Modular function chunks:", len(modular_chunks))

# Compare complete chunk contents
assert modular_chunks == sample_chunks, \
    "Modular chunking output differs from the original!"

print("Original and modular outputs match:", True)

# Verify expected errors and edge cases
assert modular_chunk_text("") == []
assert modular_chunk_text("one two three", 100, 20) == [
    "one two three"
]

try:
    modular_chunk_text("test", chunk_size=10, overlap=10)
except ValueError:
    print("Invalid overlap validation: Passed")
else:
    raise AssertionError("Invalid overlap was not rejected!")

print("\nAll reusable module verification checks passed.")

=== MODULE IMPORT VERIFICATION ===
Project root: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG
Module location: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\src\rag\chunking.py

=== CHUNKING COMPARISON ===
Original function chunks: 3
Modular function chunks: 3
Original and modular outputs match: True
Invalid overlap validation: Passed

All reusable module verification checks passed.


In [14]:
# Stage 3.3.3 — Verify reusable data loader

from rag.data_loader import load_local_dataset

print("=== DATA LOADER MODULE VERIFICATION ===")

# Load original Wikipedia articles
wiki_modular = load_local_dataset(
    PROJECT_ROOT,
    "wiki_slice_1000"
)

# Load processed Wikipedia passages
chunks_modular = load_local_dataset(
    PROJECT_ROOT,
    "wiki_chunks_100w_20overlap"
)

print("Original articles loaded:", len(wiki_modular))
print("Processed chunks loaded:", len(chunks_modular))

print("\n=== DATA INTEGRITY CHECKS ===")

# Verify record counts
assert len(wiki_modular) == len(wiki_local), "Original corpus count mismatch!"
assert len(chunks_modular) == len(chunks_local), "Chunk corpus count mismatch!"

# Verify source content
assert wiki_modular[0]["text"] == wiki_local[0]["text"], "Original article content mismatch!"
assert chunks_modular[0]["text"] == chunks_local[0]["text"], "Chunk content mismatch!"

# Verify chunk metadata
assert chunks_modular[0]["chunk_id"] == chunks_local[0]["chunk_id"], "Chunk identifier mismatch!"

print("Article count verification: Passed")
print("Chunk count verification: Passed")
print("Article content verification: Passed")
print("Chunk content verification: Passed")
print("Chunk metadata verification: Passed")

print("\nAll data loader module verification checks passed.")

=== DATA LOADER MODULE VERIFICATION ===
Original articles loaded: 1000
Processed chunks loaded: 8191

=== DATA INTEGRITY CHECKS ===
Article count verification: Passed
Chunk count verification: Passed
Article content verification: Passed
Chunk content verification: Passed
Chunk metadata verification: Passed

All data loader module verification checks passed.


### 3.3 Modularization and Software Engineering

**Objective**

To separate reusable RAG application logic from experimental
notebook code and improve maintainability and reproducibility.

**Methodology**

A Python package named `rag` was introduced under the `src`
directory.

Two reusable modules were implemented:

- `chunking.py`: Word-based passage generation.
- `data_loader.py`: Loading locally persisted datasets.

The modular implementations were validated against the previously
verified notebook outputs.

**Experimental Results**

The modular chunking function generated the same three passages
as the original prototype.

The modular data loader successfully loaded:

- 1,000 original Wikipedia articles.
- 8,191 processed Wikipedia passages.

All article-count, chunk-count, content, and metadata integrity
checks passed.

**Conclusion**

Reusable document-processing functionality has been separated
from the notebook.

This modular structure supports subsequent indexing, retrieval,
generation, evaluation, and backend integration.


In [15]:
# ============================================================
# Stage 3.4.1 — TF-IDF Index Construction (Jupyter Version)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer #this is a scikit-learn class that converts text into numerical features.

# Step 1 : extract text from our locally saved dataset
passage_texts = chunks_local["text"]

print("=== TF-IDF INDEX CONSTRUCTION ===")
print("Number of input passages:", len(passage_texts))

# Step 2: Configure the vectorizer - L2 
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=30000, #Limit the vocabulary to at most 30,000 terms
    norm="l2" #Normalize each passage vector to unit Euclidean length when nonzero, in short scale to common value the each chunk size
)

# Step 3 
tfidf_matrix = tfidf_vectorizer.fit_transform(passage_texts)

# Step 4: Examine the result
print("\n=== TF-IDF MATRIX INFORMATION ===")

print("Matrix shape:", tfidf_matrix.shape)
print("Matrix type:", type(tfidf_matrix).__name__)

print("Number of passages:", tfidf_matrix.shape[0])
print("Vocabulary size:", tfidf_matrix.shape[1])

# Step 5 : Examine the MAtrix sparsity
nonzero_values = tfidf_matrix.nnz
total_values = (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])

density = nonzero_values / total_values
sparsity = 1 - density

print("\n=== SPARSITY ANALYSIS ===")

print("Non-zero values:", nonzero_values)
print("Density (%):", round(density * 100, 4))
print("Sparsity (%):", round(sparsity * 100, 4))

# Step 6 : Inspect learned vocabulary
vocabulary = tfidf_vectorizer.get_feature_names_out()

print("\n=== VOCABULARY ===")
print("First 20 vocabulary terms:")
print(vocabulary[:20])

# Step 7: Validate expected matrix dimensions
assert tfidf_matrix.shape[0] == len(chunks_local), "Passage count mismatch!"
assert tfidf_matrix.shape[1] <= 30000, "Vocabulary exceeded maximum features!"

print("\nTF-IDF indexing verification passed.")

=== TF-IDF INDEX CONSTRUCTION ===
Number of input passages: 8191

=== TF-IDF MATRIX INFORMATION ===
Matrix shape: (8191, 30000)
Matrix type: csr_matrix
Number of passages: 8191
Vocabulary size: 30000

=== SPARSITY ANALYSIS ===
Non-zero values: 364728
Density (%): 0.1484
Sparsity (%): 99.8516

=== VOCABULARY ===
First 20 vocabulary terms:
['00' '000' '000000' '000080' '000th' '001' '0023' '003366' '004' '006400'
 '008' '00pm' '01' '010' '011' '0138' '0186828' '02' '0215' '024']

TF-IDF indexing verification passed.


### 3.4.1 TF-IDF Index Construction

#### Objective

To convert the 8,191 Wikipedia text passages into numerical
representations suitable for similarity-based information retrieval.

#### Methodology

The `TfidfVectorizer` implementation from scikit-learn was used
to construct a sparse document-term matrix.

The following configuration was selected:

| Parameter                   | Value   |
| --------------------------- | ------- |
| Lowercase conversion        | True    |
| English stop-word removal   | Enabled |
| Maximum vocabulary features | 30,000  |
| Vector normalization        | L2      |
| Number of indexed passages  | 8,191   |

The vectorizer learned vocabulary terms and inverse document
frequency statistics from the complete passage collection.

Each passage was subsequently represented as a TF-IDF vector
in the learned feature space.

#### Mathematical Formulation

TF-IDF combines term frequency and inverse document frequency:

$$
\operatorname{TFIDF}(t,d)=TF(t,d)\times IDF(t)
$$

where:

- $t$ represents a vocabulary term.
- $d$ represents a document or passage.
- $TF(t,d)$ measures the frequency of term $t$ in passage $d$.
- $IDF(t)$ measures the relative rarity of term $t$ across passages.

The smoothed inverse document frequency used by scikit-learn is:

$$
IDF(t)=\ln\left(\frac{1+N}{1+df(t)}\right)+1
$$

where:

- $N$ is the total number of indexed passages.
- $df(t)$ is the number of passages containing term $t$.

The resulting TF-IDF vectors are normalized using the L2 norm:

$$
\hat{\mathbf{x}}_d=
\frac{\mathbf{x}_d}{\|\mathbf{x}_d\|_2}
$$

for nonzero document vectors.

#### Experimental Results

| Metric                            |       Actual Result |
| --------------------------------- | ------------------: |
| Indexed passages                  |               8,191 |
| Vocabulary size                   |              30,000 |
| Matrix shape                      |      8,191 × 30,000 |
| Matrix representation             |   CSR sparse matrix |
| Total possible elements           |         245,730,000 |
| Nonzero elements                  |             364,728 |
| Matrix density                    |             0.1484% |
| Matrix sparsity                   |            99.8516% |
| Mean nonzero features per passage | Approximately 44.53 |

#### Interpretation

The TF-IDF vectorizer successfully converted the Wikipedia
passages into a numerical document-term matrix.

The matrix contains one row per passage and one column per
selected vocabulary feature.

The resulting representation is highly sparse, with approximately
99.85% of possible entries equal to zero.

This sparsity is expected because individual passages contain
only a small subset of the complete vocabulary.

The CSR representation stores nonzero weights efficiently,
avoiding the memory cost of a dense matrix containing
approximately 245.7 million entries.

The vocabulary inspection revealed numerical and digit-prefixed
terms. Their usefulness for retrieval will be evaluated before
considering additional preprocessing.

#### Limitations

TF-IDF primarily captures lexical relationships and does not
directly model semantic meaning.

Consequently, questions and passages expressing similar concepts
using different vocabulary may receive relatively low similarity
scores.

The vocabulary is restricted to 30,000 features, potentially
excluding less frequent terms.

The index has not yet been evaluated for retrieval relevance,
answer quality, or computational performance.

#### Conclusion

A TF-IDF index containing 8,191 Wikipedia passages and 30,000
vocabulary features was successfully constructed.

The sparse representation provides the numerical foundation
for the subsequent cosine-similarity retrieval system.

Retrieval effectiveness will be assessed in the next project stage.


In [16]:
# ============================================================
# Stage 3.4.3 — Verify TF-IDF Python Module
# ============================================================

import numpy as np

from rag.indexer import build_tfidf_index

print("=== MODULAR TF-IDF VERIFICATION ===")

# Build the index using our reusable Python module
modular_vectorizer, modular_matrix = build_tfidf_index(
    passages=passage_texts,
    max_features=30000
)

print("\n=== MATRIX COMPARISON ===")

print("Jupyter matrix shape:", tfidf_matrix.shape)
print("Modular matrix shape:", modular_matrix.shape)

print("Jupyter non-zero values:", tfidf_matrix.nnz)
print("Modular non-zero values:", modular_matrix.nnz)

# Verify matrix dimensions
assert modular_matrix.shape == tfidf_matrix.shape, \
    "TF-IDF matrix shapes do not match!"

print("\nMatrix shape verification: Passed")

# Verify vocabulary terms and their column order
jupyter_vocab = tfidf_vectorizer.get_feature_names_out()
modular_vocab = modular_vectorizer.get_feature_names_out()

assert np.array_equal(jupyter_vocab, modular_vocab), \
    "Vocabulary mismatch!"

print("Vocabulary verification: Passed")

# Compare numerical TF-IDF weights
difference = (tfidf_matrix - modular_matrix).tocsr()
difference.eliminate_zeros()

if difference.nnz == 0:
    max_difference = 0.0
else:
    max_difference = float(np.max(np.abs(difference.data)))

print("\nMaximum TF-IDF weight difference:", max_difference)

assert max_difference < 1e-10, \
    "TF-IDF numerical values differ!"

print("TF-IDF weight verification: Passed")

# Verify input validation
try:
    build_tfidf_index([], max_features=30000)
except ValueError:
    print("Empty input validation: Passed")
else:
    raise AssertionError("Empty passages were not rejected!")

print("\nAll modular TF-IDF verification checks passed.")

=== MODULAR TF-IDF VERIFICATION ===

=== MATRIX COMPARISON ===
Jupyter matrix shape: (8191, 30000)
Modular matrix shape: (8191, 30000)
Jupyter non-zero values: 364728
Modular non-zero values: 364728

Matrix shape verification: Passed
Vocabulary verification: Passed

Maximum TF-IDF weight difference: 0.0
TF-IDF weight verification: Passed
Empty input validation: Passed

All modular TF-IDF verification checks passed.


In [17]:
# Verify that the reusable module matches the notebook implementation

import numpy as np
from rag.indexer import build_tfidf_index

modular_vectorizer, modular_matrix = build_tfidf_index(
    passage_texts,
    max_features=30000
)

assert modular_matrix.shape == tfidf_matrix.shape

assert np.array_equal(
    modular_vectorizer.get_feature_names_out(),
    tfidf_vectorizer.get_feature_names_out()
)

assert np.allclose(
    (modular_matrix - tfidf_matrix).data,
    0,
    atol=1e-10
)

print("TF-IDF notebook and Python module verification: PASSED")

TF-IDF notebook and Python module verification: PASSED


### TF-IDF Implementation Verification

The TF-IDF indexing algorithm was implemented both directly in
Jupyter and as a reusable Python module (`rag.indexer`).

Both implementations were configured identically.

The resulting matrix dimensions, vocabulary ordering, and numerical
TF-IDF weights were compared.

This verification was designed to ensure consistency between the
experimental notebook and reusable application implementation.


In [18]:
# ============================================================
# Stage 3.5.1 — Save TF-IDF Index Locally
# ============================================================

from pathlib import Path
from scipy.sparse import save_npz
import joblib
import json

# Index storage directory
INDEX_DIR = PROJECT_ROOT / "data" / "tfidf_index"

# Define artifact paths
VECTORIZER_PATH = INDEX_DIR / "vectorizer.joblib"
MATRIX_PATH = INDEX_DIR / "tfidf_matrix.npz"
METADATA_PATH = INDEX_DIR / "index_metadata.json"

print("=== TF-IDF INDEX PERSISTENCE ===")

print("Index directory:", INDEX_DIR)

# Protect previously saved artifacts
artifact_paths = [
    VECTORIZER_PATH,
    MATRIX_PATH,
    METADATA_PATH
]

existing_files = [path for path in artifact_paths if path.exists()]

if existing_files:
    raise FileExistsError(
        f"Index artifacts already exist: {existing_files}"
    )

# Create the index directory
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# Save the fitted vectorizer
joblib.dump(tfidf_vectorizer, VECTORIZER_PATH)

# Save the sparse TF-IDF matrix
save_npz(MATRIX_PATH, tfidf_matrix)

# Save basic metadata for reproducibility
index_metadata = {
    "dataset": "wikimedia/wikipedia",
    "dataset_config": "20231101.en",
    "passage_dataset": "wiki_chunks_100w_20overlap",
    "num_passages": int(tfidf_matrix.shape[0]),
    "vocabulary_size": int(tfidf_matrix.shape[1]),
    "chunk_size_words": 100,
    "chunk_overlap_words": 20,
    "max_features": 30000,
    "vectorizer": "TfidfVectorizer",
    "normalization": "l2"
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(index_metadata, f, indent=4)

print("\n=== SAVED ARTIFACTS ===")

for path in artifact_paths:
    print(path.name, ":", path.exists())

print("\nIndex persistence completed.")

=== TF-IDF INDEX PERSISTENCE ===
Index directory: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\data\tfidf_index


In [19]:
# Retrieve TF-IDF vector for the first passage
first_vector = tfidf_matrix[0]

# Number of nonzero term weights
print("Nonzero features:", first_vector.nnz)

# Convert the nonzero feature positions into vocabulary terms
feature_names = tfidf_vectorizer.get_feature_names_out()

for index, weight in zip(
    first_vector.indices[:10],
    first_vector.data[:10]
):
    print(feature_names[index], round(weight, 4))

Nonzero features: 49
diana 0.1146
lynn 0.1198
richburg 0.4155
born 0.1203
july 0.0657
1963 0.0875
american 0.0544
middle 0.0807
distance 0.0917
runner 0.1012


### 3.5 TF-IDF Index Persistence

**Objective**

To persist the fitted TF-IDF vectorizer, sparse document-term
matrix, and index configuration for subsequent offline retrieval.

**Methodology**

The fitted TF-IDF vectorizer was serialized using Joblib.

The sparse document-term matrix was saved using SciPy's
compressed sparse matrix format (`.npz`).

Index configuration and dataset provenance were recorded
in a JSON metadata file.

**Saved Artifacts**

| Artifact            | Purpose                                                        |
| ------------------- | -------------------------------------------------------------- |
| vectorizer.joblib   | Fitted vocabulary, IDF weights and preprocessing configuration |
| tfidf_matrix.npz    | Sparse numerical passage representations                       |
| index_metadata.json | Index configuration and dataset information                    |

**Experimental Observations**

All three artifacts were successfully written to the local
`data/tfidf_index` directory.

The saved index corresponds to the previously constructed
TF-IDF representation containing 8,191 passages and
30,000 vocabulary features.

**Interpretation**

Persisting the fitted vectorizer allows future user questions
to be transformed using the same feature space as the indexed
Wikipedia passages.

Persisting the sparse matrix eliminates the need to recalculate
TF-IDF weights whenever the application starts.

The passage dataset must retain its original ordering because
matrix rows correspond positionally to passage records.

**Limitations**

Successful file creation does not establish that the saved
artifacts can be reloaded correctly.

Joblib serialization should only be loaded from trusted sources.

**Conclusion**

The TF-IDF indexing artifacts were successfully saved locally.
Their integrity and alignment with passage metadata will be
verified before implementing retrieval.


In [20]:
# Stage 3.5.2 — Verify locally saved TF-IDF index

import joblib
import json
import numpy as np
from scipy.sparse import load_npz

print("=== TF-IDF INDEX RELOAD VERIFICATION ===")

# Load the locally saved artifacts
loaded_vectorizer = joblib.load(VECTORIZER_PATH)
loaded_matrix = load_npz(MATRIX_PATH)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print("Loaded matrix shape:", loaded_matrix.shape)
print("Loaded matrix type:", type(loaded_matrix).__name__)
print("Loaded vocabulary size:", len(loaded_vectorizer.vocabulary_))

# Verify dimensions
assert loaded_matrix.shape == tfidf_matrix.shape

# Verify vocabulary
assert np.array_equal(
    loaded_vectorizer.get_feature_names_out(),
    tfidf_vectorizer.get_feature_names_out()
)

# Verify numerical values
difference = (loaded_matrix - tfidf_matrix).tocsr()
difference.eliminate_zeros()

assert difference.nnz == 0, \
    "Loaded TF-IDF matrix differs from original!"

# Verify passage alignment
assert loaded_matrix.shape[0] == len(chunks_local)

assert loaded_metadata["num_passages"] == len(chunks_local)

assert loaded_metadata["vocabulary_size"] == loaded_matrix.shape[1]

print("\nMatrix dimensions: Passed")
print("Vocabulary consistency: Passed")
print("Matrix numerical integrity: Passed")
print("Passage count alignment: Passed")
print("Metadata consistency: Passed")

print("\nAll TF-IDF index reload checks passed.")

=== TF-IDF INDEX RELOAD VERIFICATION ===
Loaded matrix shape: (8191, 30000)
Loaded matrix type: csr_matrix
Loaded vocabulary size: 30000

Matrix dimensions: Passed
Vocabulary consistency: Passed
Matrix numerical integrity: Passed
Passage count alignment: Passed
Metadata consistency: Passed

All TF-IDF index reload checks passed.


### 3.5.2 TF-IDF Index Reload and Integrity Verification

**Objective**

To verify that the locally persisted TF-IDF indexing artifacts
can be reloaded without recomputing the document representations.

**Methodology**

The fitted TF-IDF vectorizer was reloaded from its Joblib file,
and the sparse document-term matrix was reloaded using SciPy.

The reloaded artifacts were compared with the original
in-memory representations.

Verification included matrix dimensions, vocabulary ordering,
numerical values, passage counts, and stored metadata.

**Experimental Results**

| Verification               | Result            |
| -------------------------- | ----------------- |
| Reloaded matrix shape      | 8,191 × 30,000    |
| Matrix representation      | CSR sparse matrix |
| Vocabulary size            | 30,000            |
| Matrix dimensions          | Passed            |
| Vocabulary consistency     | Passed            |
| Numerical matrix integrity | Passed            |
| Passage-count alignment    | Passed            |
| Metadata consistency       | Passed            |

**Interpretation**

The saved TF-IDF index was successfully reconstructed from
local storage.

All tested properties matched the original in-memory index,
demonstrating that the vectorizer and passage representations
were persisted correctly.

The index can therefore be reused without performing the
TF-IDF fitting process again.

**Limitations**

The verification confirms matrix and vocabulary integrity,
but it does not independently prove the semantic correctness
of retrieved passages.

Passage-to-matrix row ordering must remain unchanged.
The saved metadata records the source dataset but does not
currently include a cryptographic fingerprint of passage order.

Retrieval effectiveness will be evaluated separately.

**Conclusion**

The TF-IDF index was successfully saved, reloaded, and validated.

The index is ready for integration with a passage retrieval
algorithm.


In [21]:
# ============================================================
# Stage 3.5.4 — Verify Reusable TF-IDF Index Loader
# ============================================================

import importlib
import numpy as np
import rag.indexer as indexer

# Reload because indexer.py was modified after its earlier import
importlib.reload(indexer)

print("=== REUSABLE INDEX LOADER VERIFICATION ===")

# Load using our Python module
module_vectorizer, module_matrix, module_metadata = (
    indexer.load_tfidf_index(INDEX_DIR)
)

print("Loaded matrix shape:", module_matrix.shape)
print("Loaded matrix type:", type(module_matrix).__name__)

print(
    "Loaded vocabulary size:",
    len(module_vectorizer.vocabulary_)
)

print("\n=== INTEGRITY CHECKS ===")

# Verify matrix dimensions
assert module_matrix.shape == tfidf_matrix.shape

print("Matrix dimensions: Passed")

# Verify vocabulary
assert np.array_equal(
    module_vectorizer.get_feature_names_out(),
    tfidf_vectorizer.get_feature_names_out()
)

print("Vocabulary consistency: Passed")

# Verify numerical values
difference = (module_matrix - tfidf_matrix).tocsr()
difference.eliminate_zeros()

assert difference.nnz == 0

print("Numerical matrix integrity: Passed")

# Verify passage count
assert module_matrix.shape[0] == len(chunks_local)

print("Passage count alignment: Passed")

# Verify metadata
assert module_metadata["num_passages"] == 8191
assert module_metadata["vocabulary_size"] == 30000

print("Metadata consistency: Passed")

print("\nAll reusable index loader verification checks passed.")

=== REUSABLE INDEX LOADER VERIFICATION ===
Loaded matrix shape: (8191, 30000)
Loaded matrix type: csr_matrix
Loaded vocabulary size: 30000

=== INTEGRITY CHECKS ===
Matrix dimensions: Passed
Vocabulary consistency: Passed
Numerical matrix integrity: Passed
Passage count alignment: Passed
Metadata consistency: Passed

All reusable index loader verification checks passed.


### 3.6 Stage 3 — Summary and Conclusions

**Objective**

To transform the locally stored Wikipedia corpus into searchable
numerical representations suitable for information retrieval.

**Methodology**

The 1,000-article corpus was divided into overlapping passages
using a fixed-size word-based sliding window.

The resulting passage collection was indexed using TF-IDF
with L2 normalization.

The passage dataset, fitted vectorizer, sparse document-term
matrix, and index metadata were persisted locally.

**Final Experimental Results**

| Metric                 |         Result |
| ---------------------- | -------------: |
| Source articles        |          1,000 |
| Generated passages     |          8,191 |
| Chunk size             |      100 words |
| Chunk overlap          |       20 words |
| TF-IDF vocabulary size |         30,000 |
| TF-IDF matrix shape    | 8,191 × 30,000 |
| Matrix sparsity        |       99.8516% |

**Verification**

The locally saved TF-IDF index was reloaded successfully.

Matrix dimensions, vocabulary consistency, numerical values,
passage counts, and index metadata passed all implemented
integrity checks.

**Interpretation**

The Wikipedia corpus has been transformed into a compact,
sparse numerical representation.

Each matrix row corresponds to a passage, while each column
represents a selected vocabulary term.

The fitted vectorizer enables future user questions to be
transformed into the same feature space as the indexed passages.

**Limitations**

The baseline TF-IDF representation primarily captures lexical
similarity rather than learned semantic relationships.

Word-based chunking may split sentences or paragraphs.

Longer source articles contribute more candidate passages,
potentially influencing retrieval behavior.

The index has not yet been evaluated for retrieval relevance,
ranking quality, or downstream answer generation.

**Conclusion**

A locally persisted TF-IDF index containing 8,191 Wikipedia
passages and 30,000 vocabulary features was successfully built
and validated.

The project is ready for implementation of passage retrieval
using cosine similarity.


In [25]:
# ============================================================
# Stage 4.1 — TF-IDF Passage Retrieval (Jupyter Version)
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

# Our first test question
question = "Who is Diana Richburg?"

TOP_K = 3

print("=== TF-IDF PASSAGE RETRIEVAL ===")
print("Question:", question)
print("Top-K:", TOP_K)

# Step 1: Transform question into TF-IDF feature space
question_vector = module_vectorizer.transform([question])
print("\nQuestion vector shape:", question_vector.shape)

# Step 2: calculate similarity against all passages 
similarity_scores = cosine_similarity(
    question_vector,
    modular_matrix
).flatten()
print("Number of similarity scores:", len(similarity_scores))

# Step 3 : Rank passages by similarity 
top_indices = np.argsort(similarity_scores)[::-1][:TOP_K]
print("\n=== TOP RETRIEVED PASSAGES ===")

for rank, passage_index in enumerate(top_indices, start=1):
    passage = chunks_local[int(passage_index)]
    score = float(similarity_scores[passage_index])
    print(f"\n{'=' * 65}")
    print(f"RANK {rank}")
    print(f"{'=' * 65}")

    print("Similarity score:", round(score, 4))
    print("Article:", passage["title"])
    print("Chunk ID:", passage["chunk_id"])
    print("Wikipedia URL:", passage["url"])

    print("\nPassage text:")
    print(passage["text"][:700])

# Step 4 : Validate ranking order
top_scores = similarity_scores[top_indices]

assert np.all(top_scores[:-1] >= top_scores[1:]) , " Retrieved passages are not correctly ranked"

print("\nRetrieval ranking verification passed.")

=== TF-IDF PASSAGE RETRIEVAL ===
Question: Who is Diana Richburg?
Top-K: 3

Question vector shape: (1, 30000)
Number of similarity scores: 8191

=== TOP RETRIEVED PASSAGES ===

RANK 1
Similarity score: 0.3932
Article: Diana Richburg
Chunk ID: 55617566::0
Wikipedia URL: https://en.wikipedia.org/wiki/Diana%20Richburg

Passage text:
Diana Lynn Richburg (born July 2, 1963) is an American former middle-distance runner. She competed at the 1984 Los Angeles Olympics, and won the 1985 US national title at 1500 metres. Career Born in Troy, New York, Richburg finished fourth in the 800m at the 1984 US Olympic Trials, before earning selection for the Los Angeles Olympics by finishing third in the 1500m in 4:04.07. At the Olympics, she failed to reach the final. She won the 1500m title at the 1985 USA Outdoor Track and Field Championships in 4:04.73. Richburg achieved her lifetime best for 1500m when finishing seventh

RANK 2
Similarity score: 0.2261
Article: Diana Richburg
Chunk ID: 55617566::1
W

### Stage 4.1 — TF-IDF Passage Retrieval

**Objective:** Retrieve the most relevant Wikipedia passages for a natural-language question.

**Methodology:** The input question was transformed using the previously fitted TF-IDF vectorizer. Cosine similarity was computed between the question vector and all 8,191 indexed passages. The three highest-scoring passages were retrieved.

The cosine similarity between a question vector $\mathbf{q}$ and passage vector $\mathbf{p}$ is:

$$
\operatorname{sim}(\mathbf{q},\mathbf{p})
=
\frac{\mathbf{q}\cdot\mathbf{p}}
{\|\mathbf{q}\|\|\mathbf{p}\|}
$$

where $\mathbf{q}$ represents the TF-IDF question vector, $\mathbf{p}$ represents a passage vector, and $\|\cdot\|$ denotes vector magnitude.

**Experimental Results:**

- Query: "Who is Diana Richburg?"
- Indexed passages: 8,191
- Vocabulary size: 30,000
- Top-K: 3

| Rank | Article        | Cosine Similarity |
| ---- | -------------- | ----------------: |
| 1    | Diana Richburg |            0.3932 |
| 2    | Diana Richburg |            0.2261 |
| 3    | Diana Baciu    |            0.1121 |

**Interpretation:**

The highest-ranked passage correctly identifies Diana Richburg as an American former middle-distance runner. The second passage provides additional information about the same person. The third passage concerns a different individual, demonstrating a limitation of lexical retrieval when documents share similar words or names.

**Conclusion:**

TF-IDF retrieval successfully identified relevant evidence for the sample question. However, similarity scores measure lexical relevance rather than factual correctness or confidence. The retrieved passages will subsequently be used as evidence for grounded answer generation.


In [26]:
# ============================================================
# Stage 4.2.3 — Verify Reusable TF-IDF Retriever
# ============================================================

import numpy as np

from rag.retriever import TfidfRetriever

print("=== REUSABLE TF-IDF RETRIEVER VERIFICATION ===")

# Step 1: Initialize the retriever using our existing index
retriever = TfidfRetriever(
    vectorizer=module_vectorizer,
    tfidf_matrix=module_matrix,
    passages=chunks_local
)

# Step 2: Retrieve passages for our original test question
question = "Who is Diana Richburg?"
TOP_K = 3

retrieved_results = retriever.retrieve(
    question=question,
    top_k=TOP_K
)

print("\nQuestion:", question)
print("Top-K:", TOP_K)

print("\n=== RETRIEVED RESULTS ===")

for result in retrieved_results:

    print(f"\nRank: {result['rank']}")
    print(f"Article: {result['title']}")
    print(f"Similarity: {result['score']:.4f}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"URL: {result['url']}")

# Step 3: Verify number of retrieved passages
assert len(retrieved_results) == TOP_K

print("\nNumber of retrieved passages: Passed")

# Step 4: Compare retrieved scores with our Stage 4.1 results
original_scores = similarity_scores[top_indices]

modular_scores = np.array([
    result["score"]
    for result in retrieved_results
])

assert np.allclose(
    original_scores,
    modular_scores,
    atol=1e-10
), "Similarity scores differ from Stage 4.1!"

print("Similarity score consistency: Passed")

# Step 5: Verify ranking order
assert all(
    retrieved_results[i]["score"] >= retrieved_results[i + 1]["score"]
    for i in range(len(retrieved_results) - 1)
)

print("Ranking order verification: Passed")

# Step 6: Verify top-ranked passage identity
assert retrieved_results[0]["chunk_id"] == chunks_local[
    int(top_indices[0])
]["chunk_id"]

print("Top-ranked passage identity: Passed")

# Step 7: Verify invalid input handling
try:
    retriever.retrieve("", top_k=3)
except ValueError:
    print("Empty question validation: Passed")
else:
    raise AssertionError("Empty question was not rejected!")

print("\nAll reusable retriever verification checks passed.")

=== REUSABLE TF-IDF RETRIEVER VERIFICATION ===

Question: Who is Diana Richburg?
Top-K: 3

=== RETRIEVED RESULTS ===

Rank: 1
Article: Diana Richburg
Similarity: 0.3932
Chunk ID: 55617566::0
URL: https://en.wikipedia.org/wiki/Diana%20Richburg

Rank: 2
Article: Diana Richburg
Similarity: 0.2261
Chunk ID: 55617566::1
URL: https://en.wikipedia.org/wiki/Diana%20Richburg

Rank: 3
Article: Diana Baciu
Similarity: 0.1121
Chunk ID: 59670679::0
URL: https://en.wikipedia.org/wiki/Diana%20Baciu

Number of retrieved passages: Passed
Similarity score consistency: Passed
Ranking order verification: Passed
Top-ranked passage identity: Passed
Empty question validation: Passed

All reusable retriever verification checks passed.


In [27]:
# ============================================================
# Stage 4.3.1 — Retrieval Testing with Multiple Questions
# ============================================================

test_questions = [
    "Who is Diana Richburg?",
    "Which Olympic Games did Diana Richburg compete in?",
    "What is the capital of France?"
]

TOP_K = 3

print("=== MULTIPLE QUESTION RETRIEVAL TEST ===")



for question_number , question in enumerate ( test_questions, start=1):
    print(f"\n{'=' * 65}")
    print(f"QUESTION {question_number}: {question}")
    print(f"{'=' * 65}")

    results = retriever.retrieve(
        question=question,
        top_k=TOP_K
    )

    for result in results:

        print(f"\nRank: {result['rank']}")
        print(f"Article: {result['title']}")
        print(f"Similarity: {result['score']:.4f}")
        print(f"Chunk ID: {result['chunk_id']}")
        print(f"Passage: {result['text'][:350]}")

print("\nMultiple-question retrieval test completed.")




=== MULTIPLE QUESTION RETRIEVAL TEST ===

QUESTION 1: Who is Diana Richburg?

Rank: 1
Article: Diana Richburg
Similarity: 0.3932
Chunk ID: 55617566::0
Passage: Diana Lynn Richburg (born July 2, 1963) is an American former middle-distance runner. She competed at the 1984 Los Angeles Olympics, and won the 1985 US national title at 1500 metres. Career Born in Troy, New York, Richburg finished fourth in the 800m at the 1984 US Olympic Trials, before earning selection for the Los Angeles Olympics by finishing 

Rank: 2
Article: Diana Richburg
Similarity: 0.2261
Chunk ID: 55617566::1
Passage: the 1985 USA Outdoor Track and Field Championships in 4:04.73. Richburg achieved her lifetime best for 1500m when finishing seventh in the 1987 World Championship final, running 4:01.79. The following year, she missed out on Olympic selection when she finished sixth in the 1500m at the 1988 US Olympic Trials in 4:09.68. At the 1992 US Olympic Trial

Rank: 3
Article: Diana Baciu
Similarity: 0.1121
Chunk 

### Stage 4.3 — Evaluation of TF-IDF Retrieval Across Multiple Questions

#### Objective

To qualitatively evaluate the effectiveness of the TF-IDF retrieval system across multiple natural-language questions and identify potential limitations.

#### Methodology

Three questions were submitted to the TF-IDF retriever using a fixed retrieval size of $K=3$.

For each question, cosine similarity was calculated between the query vector and all 8,191 indexed Wikipedia passages.

The three highest-scoring passages were retrieved and examined for relevance and evidence sufficiency.

#### Experimental Results

| Question                                           | Highest Cosine Similarity | Observation                                             |
| -------------------------------------------------- | ------------------------: | ------------------------------------------------------- |
| Who is Diana Richburg?                             |                    0.3932 | Relevant biographical information retrieved             |
| Which Olympic Games did Diana Richburg compete in? |                    0.3197 | Passage identifies the 1984 Los Angeles Olympics        |
| What is the capital of France?                     |                    0.4398 | Retrieved passage does not directly answer the question |

#### Interpretation

The retriever successfully identified relevant information for the first two questions.

However, the third question demonstrated an important limitation of TF-IDF retrieval: a relatively high cosine similarity score does not necessarily indicate that a passage contains the information required to answer the question.

Additionally, irrelevant passages were retrieved for some questions because of shared vocabulary.

These observations indicate that lexical relevance and factual answerability are distinct properties.

#### Limitations

1. TF-IDF relies on lexical similarity rather than semantic understanding.
2. The system may retrieve irrelevant passages containing overlapping vocabulary.
3. High similarity scores cannot be directly interpreted as confidence or factual correctness.
4. The restricted Wikipedia corpus may not contain sufficient information for every question.
5. The evaluation is qualitative and limited to three questions; it does not establish overall retrieval accuracy.

#### Conclusion

The TF-IDF retrieval system demonstrated successful evidence retrieval for two of the three evaluated questions.

However, the experiment highlights the need to evaluate evidence sufficiency and answer grounding independently of cosine similarity.

These findings will inform the subsequent evaluation of retrieval size and grounded answer generation.


In [28]:
# ============================================================
# Stage 4.4.1 — Retrieval Size Experiment
# Compare K = 1, 3, 5
# ============================================================

import pandas as pd

QUESTION = "Who is Diana Richburg?"

K_VALUES = [1, 3, 5]

experiment_results = []

print("=== RETRIEVAL SIZE EXPERIMENT ===")
print("Question:", QUESTION)

for k in K_VALUES:

    print(f"\n{'=' * 60}")
    print(f"TOP-K = {k}")
    print(f"{'=' * 60}")

    results = retriever.retrieve(
        question=QUESTION,
        top_k=k
    )

    for result in results:

        print(
            f"Rank {result['rank']} | "
            f"Article: {result['title']} | "
            f"Similarity: {result['score']:.4f}"
        )

        experiment_results.append({
            "k": k,
            "rank": result["rank"],
            "article": result["title"],
            "chunk_id": result["chunk_id"],
            "similarity": result["score"]
        })

# Convert experimental observations into a DataFrame
retrieval_k_df = pd.DataFrame(experiment_results)

print("\n=== RETRIEVAL SIZE COMPARISON ===")

display(retrieval_k_df)

print("\nRetrieval size experiment completed.")

=== RETRIEVAL SIZE EXPERIMENT ===
Question: Who is Diana Richburg?

TOP-K = 1
Rank 1 | Article: Diana Richburg | Similarity: 0.3932

TOP-K = 3
Rank 1 | Article: Diana Richburg | Similarity: 0.3932
Rank 2 | Article: Diana Richburg | Similarity: 0.2261
Rank 3 | Article: Diana Baciu | Similarity: 0.1121

TOP-K = 5
Rank 1 | Article: Diana Richburg | Similarity: 0.3932
Rank 2 | Article: Diana Richburg | Similarity: 0.2261
Rank 3 | Article: Diana Baciu | Similarity: 0.1121
Rank 4 | Article: Natasha Lako | Similarity: 0.0979
Rank 5 | Article: Maurice Hurley | Similarity: 0.0963

=== RETRIEVAL SIZE COMPARISON ===


,k,rank,article,chunk_id,similarity
0,1,1,Diana Richburg,55617566::0,0.393219
1,3,1,Diana Richburg,55617566::0,0.393219
2,3,2,Diana Richburg,55617566::1,0.226085
3,3,3,Diana Baciu,59670679::0,0.112125
4,5,1,Diana Richburg,55617566::0,0.393219
5,5,2,Diana Richburg,55617566::1,0.226085
6,5,3,Diana Baciu,59670679::0,0.112125
7,5,4,Natasha Lako,45522263::0,0.097909
8,5,5,Maurice Hurley,45520700::3,0.096326



Retrieval size experiment completed.


### Stage 4.4 — Retrieval Size Ablation

#### Objective

To investigate how changing the number of retrieved passages (Top-K) affects retrieval relevance and the amount of potentially irrelevant evidence.

#### Methodology

The TF-IDF retriever was evaluated using three retrieval sizes:

$$
K \in \{1,3,5\}
$$

The question "Who is Diana Richburg?" was used for the initial controlled experiment.

The same fitted TF-IDF vectorizer, passage matrix and cosine similarity function were maintained across all configurations.

#### Experimental Results

| K   | Relevant Passages | Irrelevant Passages | Estimated Precision@K |
| --- | ----------------: | ------------------: | --------------------: |
| 1   |                 1 |                   0 |                 1.000 |
| 3   |                 2 |                   1 |                 0.667 |
| 5   |                 2 |                   3 |                 0.400 |

Relevance was judged provisionally using the retrieved article identities and previously inspected passage content.

Precision@K is defined as:

$$
P@K = \frac{R_K}{K}
$$

where:

- $P@K$ is precision at retrieval size $K$.
- $R_K$ is the number of relevant passages among the top $K$ results.
- $K$ is the total number of retrieved passages.

#### Observations

1. The highest-ranked passage remained unchanged for all three retrieval sizes.
2. Increasing K from 1 to 3 introduced additional information about Diana Richburg, but also retrieved an unrelated article.
3. Increasing K from 3 to 5 introduced two further apparently unrelated articles.
4. Estimated retrieval precision decreased as K increased for this specific question.
5. The highest cosine similarity remained 0.393219 across all configurations.

#### Interpretation

The experiment demonstrates a trade-off between the quantity of retrieved context and its relevance.

Retrieving additional passages can increase evidence coverage, but may also introduce noise.

For the evaluated question, the highest-ranked passage alone was sufficient to identify Diana Richburg.

However, this observation cannot establish the optimal retrieval size for other questions.

#### Limitations

- Only one question was evaluated.
- Relevance judgments were manual and provisional.
- Precision@K was calculated for retrieval, not generated-answer accuracy.
- The effect of retrieval size on Flan-T5-small generation remains to be evaluated.

#### Conclusion

The retrieval-size experiment demonstrated that increasing Top-K does not necessarily improve evidence relevance.

The results motivate further evaluation of K values across multiple questions and subsequent comparison of grounded answers generated using different retrieval configurations.


In [29]:
# ============================================================
# Stage 5.1.1 — Flan-T5 Tokenizer Inspection
# ============================================================


from transformers import AutoTokenizer, AutoConfig

MODEL_NAME = "google/flan-t5-small"

print("=== FLAN-T5-SMALL TOKENIZER INSPECTION ===")

# LOAD  TOKENIZER & MODEL CONFIGURATION
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_config = AutoConfig.from_pretrained(MODEL_NAME)


print("Model:", MODEL_NAME)
print("Model architecture:", model_config.model_type)
print("Encoder-decoder:", model_config.is_encoder_decoder)

print("\n=== TOKENIZER INFORMATION ===")

print("Tokenizer class:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("Model maximum length:", tokenizer.model_max_length)

# Use your original RAG question
question = "Who is Diana Richburg?"

# Tokenize without truncation
encoded = tokenizer(
    question,
    return_tensors="pt",
    truncation = False
)

token_ids = encoded["input_ids"][0]
print("\n=== QUESTION TOKENIZATION ===")

print("Original question:", question)
print("Token IDs:", token_ids.tolist())
print("Number of tokens:", len(token_ids))

print("\nIndividual tokens:")
print(tokenizer.convert_ids_to_tokens(token_ids.tolist()))

# Inspect First Retrieved passage
passage = retrieved_results[0]["text"]
passage_tokens = tokenizer(
    passage,
    truncation =False
)["input_ids"]

print("\n=== PASSAGE TOKENIZATION ===")

print("Passage word count:", len(passage.split()))
print("Passage token count:", len(passage_tokens))

print("\nTokenizer inspection completed.")


=== FLAN-T5-SMALL TOKENIZER INSPECTION ===


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

c:\Users\Ashutosh Chittora\anaconda3\envs\.cvenv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ashutosh Chittora\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

c:\Users\Ashutosh Chittora\anaconda3\envs\.cvenv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ashutosh Chittora\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Model: google/flan-t5-small
Model architecture: t5
Encoder-decoder: True

=== TOKENIZER INFORMATION ===
Tokenizer class: T5Tokenizer
Vocabulary size: 32100
Model maximum length: 512

=== QUESTION TOKENIZATION ===
Original question: Who is Diana Richburg?
Token IDs: [2645, 19, 18821, 10825, 4824, 58, 1]
Number of tokens: 7

Individual tokens:
['▁Who', '▁is', '▁Diana', '▁Rich', 'burg', '?', '</s>']

=== PASSAGE TOKENIZATION ===
Passage word count: 100
Passage token count: 133

Tokenizer inspection completed.


### Stage 5.1 — Flan-T5-small Architecture and Tokenization

#### Objective

To investigate the architecture and tokenization behaviour of Flan-T5-small before integrating it into the offline Retrieval-Augmented Generation pipeline.

#### Methodology

The pretrained tokenizer and model configuration associated with `google/flan-t5-small` were loaded using the Hugging Face Transformers library.

The tokenizer was examined using a sample question and a previously retrieved Wikipedia passage.

Tokenization was performed without truncation to observe the actual input token counts.

#### Experimental Results

| Property                        | Observed Value       |
| ------------------------------- | -------------------- |
| Model                           | google/flan-t5-small |
| Architecture                    | T5                   |
| Encoder–decoder                 | Yes                  |
| Tokenizer                       | T5Tokenizer          |
| Vocabulary size                 | 32,100               |
| Configured maximum input length | 512 tokens           |
| Question token count            | 7                    |
| Passage word count              | 100                  |
| Passage token count             | 133                  |

#### Observations

The tokenizer divided the word "Richburg" into two subword tokens: `Rich` and `burg`.

The 100-word Wikipedia passage required 133 tokens.

The observed token-to-word ratio was:

$$
r = \frac{N_{\text{tokens}}}{N_{\text{words}}}
$$

where:

- $r$ is the token-to-word ratio.
- $N_{\text{tokens}}$ is the number of generated tokens.
- $N_{\text{words}}$ is the number of whitespace-separated words.

For the evaluated passage:

$$
r = \frac{133}{100} = 1.33
$$

#### Interpretation

Word-based chunk sizes do not directly correspond to transformer token counts.

Consequently, the amount of retrieved evidence that can be supplied to Flan-T5-small depends on the tokenizer and the available input-token budget.

Increasing retrieval size may exceed the configured input-length limit, particularly when the question and task instructions are included.

#### Conclusion

Flan-T5-small uses an encoder–decoder transformer architecture and subword tokenization.

The experiment demonstrates the importance of measuring token lengths before constructing prompts from retrieved evidence.

A token-budgeting strategy will therefore be necessary during grounded answer generation.


In [ ]:
# ============================================================
# Stage 5.1.2 — RAG Prompt Token Budget Analysis
# ============================================================

QUESTION = "Who is Diana Richburg?"
K_VALUES = [1, 3, 5]

MAX_INPUT_TOKENS = 512

print("=== RAG TOKEN BUDGET ANALYSIS ===")

token_budget_results = []

for k in K_VALUES:

    # Retrieve the top-k Wikipedia passages
    results = retriever.retrieve(
        question=QUESTION,
        top_k=k
    )

    # Combine the retrieved passages into evidence
    evidence = "\n\n".join(
        result["text"]
        for result in results
    )

    # Construct the input prompt
    prompt = (
        "Answer the question using only the evidence below. "
        "If the evidence does not contain the answer, say "
        "'Insufficient evidence'.\n\n"
        f"Evidence:\n{evidence}\n\n"
        f"Question: {QUESTION}\n"
        "Answer:"
    )

    # Tokenize the complete prompt without truncation
    tokenized = tokenizer(
        prompt,
        truncation=False
    )

    token_count = len(tokenized["input_ids"])
    remaining_tokens = MAX_INPUT_TOKENS - token_count
    fits = token_count <= MAX_INPUT_TOKENS
    token_budget_results.append({
        "K": k,
        "Input Tokens": token_count,
        "Remaining Tokens": remaining_tokens,
        "Within Budget": fits
    })

    print(f"\nK = {k}")
    print("Input tokens:", token_count)
    print("Remaining input tokens:", remaining_tokens)
    print("Within 512-token budget:", fits)

print("\n=== TOKEN BUDGET SUMMARY ===")

import pandas as pd

token_budget_df = pd.DataFrame(token_budget_results)

display(token_budget_df)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (725 > 512). Running this sequence through the model will result in indexing errors


=== RAG TOKEN BUDGET ANALYSIS ===

K = 1
Input tokens: 171
Remaining input tokens: 341
Within 512-token budget: True

K = 3
Input tokens: 454
Remaining input tokens: 58
Within 512-token budget: True

K = 5
Input tokens: 725
Remaining input tokens: -213
Within 512-token budget: False

=== TOKEN BUDGET SUMMARY ===


,K,Input Tokens,Remaining Tokens,Within Budget
0,1,171,341,True
1,3,454,58,True
2,5,725,-213,False


### Stage 5.1.2 — RAG Input Token Budget Analysis

#### Objective

To evaluate how retrieval size affects the input-token requirements of Flan-T5-small.

#### Methodology

A grounded question-answering prompt was constructed using retrieved Wikipedia passages for three retrieval sizes:

$$
K \in \{1,3,5\}
$$

The complete prompt, including instructions, retrieved evidence and the question, was tokenized using the Flan-T5-small tokenizer.

A maximum input budget of 512 tokens was used.

#### Experimental Results

| Retrieval Size | Input Tokens | Remaining Budget | Within Budget |
| -------------- | -----------: | ---------------: | ------------- |
| K=1            |          171 |              341 | Yes           |
| K=3            |          454 |               58 | Yes           |
| K=5            |          725 |             -213 | No            |

#### Mathematical Formulation

The remaining token budget is:

$$
B_{\text{remaining}} = B_{\max} - N_{\text{input}}
$$

where:

- $B_{\max}$ represents the maximum permitted input tokens.
- $N_{\text{input}}$ represents the number of tokens in the complete prompt.
- $B_{\text{remaining}}$ represents the remaining input-token capacity.

For K=5:

$$
B_{\text{remaining}} = 512 - 725 = -213
$$

The negative value indicates that the prompt exceeds the selected input budget by 213 tokens.

#### Interpretation

Increasing retrieval size increases the amount of evidence included in the generation prompt.

For the evaluated question, K=1 and K=3 fit within the selected input budget, while K=5 exceeded it.

This demonstrates that retrieval size must be considered together with model context limitations.

#### Conclusion

The experiment identified a practical limitation affecting retrieval-size ablation.

A consistent token-budget management strategy will be required to compare generation quality across retrieval configurations without introducing uncontrolled truncation.


In [ ]:
# ============================================================
# Stage 5.2.1 — Load Flan-T5-small onto GPU
# ============================================================

import torch 
from transformers import AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"
print("=== FLAN-T5-SMALL MODEL LOADING ===")

# ------------------------------------------------------------
# Step 1 : Select execution device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

if device.type =="cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Step 2 : Load pre-trained model weights
# ------------------------------------------------------------
print("\nLoading Flan-T5-small...")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# ------------------------------------------------------------
# Step 3 : Move model to GPU or CPU
# ------------------------------------------------------------

model = model.to(device)

# ------------------------------------------------------------
# Step 4 : Enable Evaluation mode
# ------------------------------------------------------------

model.eval()

# ------------------------------------------------------------
# Step 5 : Inspect Model - We are using pretrained weights, so no training is required for our current RAG assignment.
# ------------------------------------------------------------

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("\n=== MODEL INFORMATION ===")
print("Model:", MODEL_NAME)
print("Architecture:", type(model).__name__)
print("Model device:", next(model.parameters()).device)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Training mode:", model.training)

# ------------------------------------------------------------
# Step 6 : Verification
# ------------------------------------------------------------

assert next(model.parameters()).device.type == device.type, "Model is not on the selected device!"
assert model.training is False, "Model is not in evaluation mode!"
print("\nModel loading verification passed.")

=== FLAN-T5-SMALL MODEL LOADING ===
PyTorch version: 2.10.0+cu128
CUDA available: True
Selected device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU

Loading Flan-T5-small...


W1031 19:45:05.249000 24456 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


=== MODEL INFORMATION ===
Model: google/flan-t5-small
Architecture: T5ForConditionalGeneration
Model device: cuda:0
Total parameters: 76,961,152
Trainable parameters: 76,961,152
Training mode: False

Model loading verification passed.


In [ ]:
# How to release GPU memory manually . When you're completely finished with the model:

# import gc
# import torch

# # Remove reference to the model
# del model

# # Run python garbage collection
# gc.collect()

# # Release unused GPU memory cached by PyTorch
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# print("GPU cleanup requested.")


In [34]:
# ============================================================
# GPU Memory and Utilization Monitoring
# ============================================================

import torch
import subprocess

print("=== NVIDIA GPU MONITORING ===")

if torch.cuda.is_available():

    device_id = torch.cuda.current_device()

    gpu_name = torch.cuda.get_device_name(device_id)

    # Total GPU memory
    total_memory = torch.cuda.get_device_properties(
        device_id
    ).total_memory

    # PyTorch memory currently allocated
    allocated_memory = torch.cuda.memory_allocated(device_id)

    # PyTorch memory reserved by caching allocator
    reserved_memory = torch.cuda.memory_reserved(device_id)

    # Available memory based on PyTorch allocations
    available_memory = total_memory - allocated_memory

    MB = 1024 ** 2
    GB = 1024 ** 3

    # Calculate percentages
    allocated_pct = (allocated_memory / total_memory) * 100
    reserved_pct = (reserved_memory / total_memory) * 100
    available_pct = (available_memory / total_memory) * 100

    print("\nGPU:", gpu_name)

    print("\n=== PYTORCH GPU MEMORY ===")

    print(f"Total GPU Memory     : {total_memory / GB:.2f} GB")

    print(
        f"Allocated Memory     : "
        f"{allocated_memory / MB:.2f} MB "
        f"({allocated_pct:.2f}%)"
    )

    print(
        f"Reserved Memory      : "
        f"{reserved_memory / MB:.2f} MB "
        f"({reserved_pct:.2f}%)"
    )

    print(
        f"Not PyTorch-Allocated: "
        f"{available_memory / MB:.2f} MB "
        f"({available_pct:.2f}%)"
    )

    # Overall GPU memory and compute utilization
    print("\n=== OVERALL NVIDIA GPU USAGE ===")

    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=memory.total,memory.used,memory.free,utilization.gpu",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True,
        text=True,
        check=True
    )

    for line in result.stdout.strip().splitlines():

        total_mb, used_mb, free_mb, gpu_util = [
            float(value.strip())
            for value in line.split(",")
        ]

        used_pct = (used_mb / total_mb) * 100
        free_pct = (free_mb / total_mb) * 100

        print(f"Overall VRAM Used    : {used_mb:.2f} MB ({used_pct:.2f}%)")
        print(f"Overall VRAM Free    : {free_mb:.2f} MB ({free_pct:.2f}%)")
        print(f"GPU Compute Usage    : {gpu_util:.2f}%")

else:
    print("CUDA GPU is unavailable.")

=== NVIDIA GPU MONITORING ===

GPU: NVIDIA GeForce RTX 5060 Laptop GPU

=== PYTORCH GPU MEMORY ===
Total GPU Memory     : 7.96 GB
Allocated Memory     : 293.58 MB (3.60%)
Reserved Memory      : 324.00 MB (3.98%)
Not PyTorch-Allocated: 7856.98 MB (96.40%)

=== OVERALL NVIDIA GPU USAGE ===
Overall VRAM Used    : 0.00 MB (0.00%)
Overall VRAM Free    : 7899.00 MB (96.91%)
GPU Compute Usage    : 0.00%


# 🚀 Stage 5.2.2 — Our First Grounded Answer Generation

We are about to complete an important milestone in our RAG pipeline:

> **Wikipedia → Retrieval → Flan-T5 → Generated Answer**

---

## 🧠 A. Layman's Explanation

Our retriever has already found the following evidence from Wikipedia:

> **"Diana Lynn Richburg is an American former middle-distance runner."**

Now, we will ask **Flan-T5-small** to read this evidence and answer the question:

### ❓ Question

**Who is Diana Richburg?**

The goal is for the model to generate an answer **using the retrieved evidence**, rather than relying on unsupported or hallucinated information.

In other words:

```text
Retrieved Evidence
        ↓
    Flan-T5
        ↓
Generated Answer
```

This is our first example of **Grounded Generation**, where the answer is tied directly to the retrieved source.

---

## ⚙️ B. Technical Explanation

To generate a grounded answer, we perform the following five steps:

### 1️⃣ Retrieve the Most Relevant Passage

Fetch the highest-ranked Wikipedia passage returned by the retriever.

### 2️⃣ Construct the Prompt

Build a prompt that contains:

- The retrieved evidence
- The user question

Example:

```text
Context:
Diana Lynn Richburg is an American former middle-distance runner.

Question:
Who is Diana Richburg?

Answer:
```

### 3️⃣ Tokenize the Prompt

Convert the prompt into **token IDs** that can be processed by Flan-T5.

```text
Prompt
   ↓
Tokenizer
   ↓
Token IDs
```

### 4️⃣ Generate the Answer

Use **Flan-T5-small** to generate an answer from the provided context.

For this experiment, we use:

```python
do_sample=False
```

This enables **deterministic greedy decoding**, ensuring reproducible results across runs.

### 5️⃣ Decode the Output Tokens

Convert the generated token IDs back into readable text.

```text
Generated Token IDs
           ↓
       Decoder
           ↓
    Human-Readable Answer
```

---

## 📊 Why We Use K = 1

For this first experiment:

```text
K = 1
```

Meaning:

- Only the single most relevant passage is retrieved.
- The prompt requires approximately **171 input tokens**.
- This comfortably fits within our selected **512-token budget**.

### Token Budget Check

| Setting                | Value                |
| ---------------------- | -------------------- |
| Retrieved Passages (K) | 1                    |
| Input Tokens           | ~171                 |
| Model Context Budget   | 512                  |
| Status                 | ✅ Fits Within Limit |

---

## 🎯 Expected Pipeline Flow

```text
Wikipedia Article
        ↓
 Passage Retrieval (K=1)
        ↓
 Retrieved Evidence
        ↓
 Prompt Construction
        ↓
 Tokenization
        ↓
 Flan-T5-small
        ↓
 Generated Tokens
        ↓
 Decoding
        ↓
 Final Grounded Answer
```

### Example Output

**Evidence**

> Diana Lynn Richburg is an American former middle-distance runner.

**Question**

> Who is Diana Richburg?

**Generated Answer**

> Diana Richburg is an American former middle-distance runner.

✅ The answer is grounded in the retrieved evidence rather than generated from unsupported knowledge.

---

## 🏁 Milestone Achieved

With this step, our complete end-to-end pipeline becomes:

```text
Wikipedia
    ↓
Chunking
    ↓
Indexing
    ↓
Retrieval
    ↓
Prompt Construction
    ↓
Flan-T5 Generation
    ↓
Grounded Answer
```

This is the first fully functional **Retrieval-Augmented Generation (RAG)** workflow.


In [ ]:
# ============================================================
# Stage 5.2.2 — First Grounded Answer Generation
# ============================================================

import time

QUESTION = "Who is Diana Richburg?"
TOP_K = 1

MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 64

print("=== FIRST GROUNDED RAG GENERATION ===")


# ------------------------------------------------------------
# Step 1: Retrieve relevant wikipedia evidence
# ------------------------------------------------------------
results = retriever.retrieve(
    question=QUESTION,
    top_k=TOP_K
)

evidence = results[0]["text"]

print("\nQuestion:", QUESTION)
print("Evidence source:", results[0]["title"])
print("Retrieval similarity:", round(results[0]["score"], 4))


# ------------------------------------------------------------
# Step 2: Construct grounded prompt
# ------------------------------------------------------------

prompt = (
    "Answer the question using only the evidence below. "
    "If the evidence does not contain the answer, say "
    "'Insufficient evidence'.\n\n"
    f"Evidence:\n{evidence}\n\n"
    f"Question: {QUESTION}\n"
    "Answer:"
)


# ------------------------------------------------------------
# Step 3: Tokenize prompt
# ------------------------------------------------------------

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=False
)

input_token_count = inputs["input_ids"].shape[1]

print("\nInput token count:", input_token_count)

# Prevent accidental input truncation
assert input_token_count <= MAX_INPUT_TOKENS, "Prompt exceeds the selected token budget!"

# Move input tensors onto the same device as the model
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


# ------------------------------------------------------------
# Step 4: Generate Answer
# ------------------------------------------------------------

model.eval()

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.inference_mode():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False, ### DISABLE RANDOM SAMPLING
        num_beams=1 ###  USE GREEDY DECODING
    )

if device.type == "cuda": torch.cuda.synchronize()

generation_time = time.perf_counter() - start_time

# ------------------------------------------------------------
# Step 5: Decode generated tokens
# ------------------------------------------------------------

generated_answer = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("\n=== GENERATED ANSWER ===")
print("Question:", QUESTION)
print("Answer:", generated_answer)
print("\n=== GENERATION METRICS ===")
print("Input tokens:", input_token_count)
print("Generated tokens:", output_ids.shape[1])
print(f"Generation time: {generation_time:.4f} seconds")
print("Generation device:", device)
print("\nFirst RAG generation experiment completed.")

=== FIRST GROUNDED RAG GENERATION ===

Question: Who is Diana Richburg?
Evidence source: Diana Richburg
Retrieval similarity: 0.3932

Input token count: 171

=== GENERATED ANSWER ===
Question: Who is Diana Richburg?
Answer: middle-distance runner

=== GENERATION METRICS ===
Input tokens: 171
Generated tokens: 9
Generation time: 1.3301 seconds
Generation device: cuda

First RAG generation experiment completed.


### Stage 5.2.2 — First Grounded Answer Generation

#### Objective

To verify that the pretrained Flan-T5-small model can generate an answer using evidence retrieved from the local Wikipedia corpus.

#### Methodology

A question-answering pipeline was constructed using:

1. TF-IDF-based passage retrieval.
2. Cosine similarity ranking.
3. Top-1 evidence selection.
4. Evidence-grounded prompting.
5. Flan-T5-small answer generation.

The model was executed on an NVIDIA GPU using PyTorch inference mode.

Deterministic greedy decoding was used with `do_sample=False` and `num_beams=1`.

#### Experimental Results

| Parameter             | Observed Value         |
| --------------------- | ---------------------- |
| Question              | Who is Diana Richburg? |
| Evidence source       | Diana Richburg         |
| Retrieval similarity  | 0.3932                 |
| Input token count     | 171                    |
| Generated answer      | middle-distance runner |
| Generated token count | 9                      |
| Generation latency    | 1.3301 seconds         |
| Execution device      | CUDA                   |

#### Answer Quality Analysis

The retrieved Wikipedia passage identifies Diana Richburg as an American former middle-distance runner.

The generated answer, "middle-distance runner", is factually consistent with this evidence.

However, the answer is abbreviated and does not provide the complete biographical description.

This demonstrates that an answer can be factually supported while still lacking completeness.

#### Interpretation

The experiment confirms that the retrieval and generation components can operate together successfully.

The model generated a short, evidence-supported response using the retrieved Wikipedia passage.

The measured latency reflects a single GPU inference call and should not be interpreted as representative production performance.

#### Limitations

1. Only one question was evaluated.
2. The response was factually supported but incomplete.
3. Groundedness was assessed manually.
4. No automated answer-quality metric was calculated.
5. Generation latency was measured for only one inference call.

#### Conclusion

The first end-to-end RAG experiment successfully generated an evidence-supported answer using Flan-T5-small.

Further experiments are required to evaluate answer completeness, retrieval-size sensitivity, hallucination behaviour and the effect of different evidence-fusion strategies.


In [36]:
# ============================================================
# Stage 5.3.4 — Verify Reusable Flan-T5 Generator
# ============================================================

from rag.generator import FlanT5Generator

print("=== REUSABLE FLAN-T5 GENERATOR VERIFICATION ===")

# ------------------------------------------------------------
# Step 1: Initialize reusable generator
# ------------------------------------------------------------

generator = FlanT5Generator(
    model_name="google/flan-t5-small",
    max_input_tokens=512,
    max_new_tokens=64,
    device=device
)

print("\nGenerator initialized successfully.")
print("Model device:", generator.device)

# ------------------------------------------------------------
# Step 2: Retrieve Wikipedia evidence
# ------------------------------------------------------------

question = "Who is Diana Richburg?"

evidence_passages = retriever.retrieve(
    question=question,
    top_k=1
)

# ------------------------------------------------------------
# Step 3: Generate answer
# ------------------------------------------------------------

result = generator.generate(
    question=question,
    passages=evidence_passages
)

print("\n=== GENERATED ANSWER ===")

print("Question:", result["question"])
print("Answer:", result["answer"])
print("Evidence count:", result["evidence_count"])

print("\n=== GENERATION METRICS ===")

print("Input tokens:", result["input_tokens"])
print("Generated tokens:", result["generated_tokens"])

print(
    "Generation time:",
    f"{result['generation_time_seconds']:.4f}",
    "seconds"
)

print("Device:", result["device"])

# ------------------------------------------------------------
# Step 4: Compare with original Jupyter experiment
# ------------------------------------------------------------

print("\n=== CONSISTENCY CHECKS ===")

assert result["answer"] == generated_answer, \
    "Generated answer differs from original Jupyter result!"

print("Generated answer consistency: Passed")

assert result["input_tokens"] == input_token_count, \
    "Input token count differs!"

print("Input token consistency: Passed")

assert result["evidence_count"] == 1

print("Evidence count verification: Passed")

# ------------------------------------------------------------
# Step 5: Verify input validation
# ------------------------------------------------------------

try:
    generator.generate(
        question="",
        passages=evidence_passages
    )
except ValueError:
    print("Empty question validation: Passed")
else:
    raise AssertionError("Empty question was not rejected!")

# ------------------------------------------------------------
# Step 6: Verify token-budget protection
# ------------------------------------------------------------

five_passages = retriever.retrieve(
    question=question,
    top_k=5
)

try:
    generator.generate(
        question=question,
        passages=five_passages
    )
except ValueError as error:
    print("Token-budget protection: Passed")
    print("Validation message:", str(error))
else:
    raise AssertionError("Oversized prompt was not rejected!")

print("\nAll reusable generator verification checks passed.")

=== REUSABLE FLAN-T5 GENERATOR VERIFICATION ===


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (725 > 512). Running this sequence through the model will result in indexing errors



Generator initialized successfully.
Model device: cuda

=== GENERATED ANSWER ===
Question: Who is Diana Richburg?
Answer: middle-distance runner
Evidence count: 1

=== GENERATION METRICS ===
Input tokens: 171
Generated tokens: 9
Generation time: 0.1130 seconds
Device: cuda

=== CONSISTENCY CHECKS ===
Generated answer consistency: Passed
Input token consistency: Passed
Evidence count verification: Passed
Empty question validation: Passed
Token-budget protection: Passed
Validation message: Prompt requires 725 tokens, but maximum is 512. Reduce retrieval size or evidence length.

All reusable generator verification checks passed.


In [ ]:
# ============================================================
# Stage 5.4.1 — End-to-End RAG Pipeline (Jupyter Prototype)
# ============================================================

import time

print("=== END-TO-END RAG PIPELINE ===")


def run_rag_pipeline(question, top_k=1):
    """
    Run retrieval and grounded generation.
    Uses the existing retriever and generator objects.
    """

    # --------------------------------------------------------
    # Step 1: Retrieve Wikipedia evidence
    # --------------------------------------------------------

    start_time = time.perf_counter()

    retrieved_passages = retriever.retrieve(
        question=question,
        top_k=top_k
    )

    retrieval_time = time.perf_counter() - start_time

    # --------------------------------------------------------
    # Step 2: Generate grounded answer
    # --------------------------------------------------------

    generation_result = generator.generate(
        question=question,
        passages=retrieved_passages
    )

    # --------------------------------------------------------
    # Step 3: Return structured result
    # --------------------------------------------------------

    return {
        "question": question,
        "answer": generation_result["answer"],

        "top_k": top_k,
        "sources": retrieved_passages,

        "retrieval_time_seconds": retrieval_time,
        "generation_time_seconds":generation_result["generation_time_seconds"],
        "input_tokens": generation_result["input_tokens"],
        "generated_tokens": generation_result["generated_tokens"],
        "device": generation_result["device"]
    }


# ------------------------------------------------------------
# Step 4: Test the complete pipeline
# ------------------------------------------------------------

QUESTION = "Who is Diana Richburg?"

rag_result = run_rag_pipeline(
    question=QUESTION,
    top_k=1
)

print("\n=== GENERATED ANSWER ===")
print("Question:", rag_result["question"])
print("Answer:", rag_result["answer"])

print("\n=== RETRIEVED SOURCES ===")

for source in rag_result["sources"]:

    print("Article:", source["title"])
    print("Similarity:", round(source["score"], 4))
    print("URL:", source["url"])

print("\n=== PERFORMANCE METRICS ===")
print("Retrieval time:",f"{rag_result['retrieval_time_seconds']:.4f} seconds")
print("Generation time:",f"{rag_result['generation_time_seconds']:.4f} seconds")
print("Input tokens:", rag_result["input_tokens"])
print("Generated tokens:", rag_result["generated_tokens"])
print("Device:", rag_result["device"])

# ------------------------------------------------------------
# Step 5: Basic verification
# ------------------------------------------------------------

assert rag_result["answer"].strip(), "Generated answer is empty!"
assert len(rag_result["sources"]) == 1, "Incorrect number of retrieved sources!"
assert rag_result["input_tokens"] <= 512, "Input exceeds token budget!"
print("\nEnd-to-end RAG pipeline verification passed.")

=== END-TO-END RAG PIPELINE ===

=== GENERATED ANSWER ===
Question: Who is Diana Richburg?
Answer: middle-distance runner

=== RETRIEVED SOURCES ===
Article: Diana Richburg
Similarity: 0.3932
URL: https://en.wikipedia.org/wiki/Diana%20Richburg

=== PERFORMANCE METRICS ===
Retrieval time: 0.0077 seconds
Generation time: 0.0744 seconds
Input tokens: 171
Generated tokens: 9
Device: cuda

End-to-end RAG pipeline verification passed.


### Stage 5.4 — End-to-End RAG Pipeline Integration

#### Objective

To integrate TF-IDF passage retrieval with Flan-T5-small answer generation and verify the operation of the complete Retrieval-Augmented Generation pipeline.

#### Methodology

The pipeline consists of two sequential operations:

1. Retrieval of relevant Wikipedia passages using TF-IDF and cosine similarity.
2. Generation of an evidence-grounded answer using Flan-T5-small.

The initial experiment used a retrieval size of K=1.

The generator received the retrieved passage and question through an evidence-grounded prompt.

Inference was performed on an NVIDIA GPU using deterministic greedy decoding.

#### Mathematical Formulation

The retriever selects the highest-ranked passages:

$$
R_K(q)=\operatorname{TopK}_{p_i}
\left(\operatorname{sim}(q,p_i)\right)
$$

The generator produces an answer conditioned on the question and retrieved evidence:

$$
\hat{a}=G(q,R_K(q))
$$

where:

- $q$ is the user question.
- $p_i$ is a Wikipedia passage.
- $K$ is the retrieval size.
- $R_K(q)$ is the retrieved evidence.
- $G$ represents Flan-T5-small.
- $\hat{a}$ is the generated answer.

#### Experimental Results

| Metric               | Observed Value         |
| -------------------- | ---------------------- |
| Question             | Who is Diana Richburg? |
| Generated answer     | middle-distance runner |
| Evidence source      | Diana Richburg         |
| Retrieval similarity | 0.3932                 |
| Retrieval latency    | 0.0077 seconds         |
| Generation latency   | 0.0744 seconds         |
| Input tokens         | 171                    |
| Generated tokens     | 9                      |
| Execution device     | CUDA                   |

#### Interpretation

The pipeline successfully retrieved the relevant Wikipedia passage and generated an answer supported by the retrieved evidence.

The generated response correctly identified Diana Richburg's occupation but omitted additional descriptive information.

Generation required more execution time than retrieval in this experiment.

The measured retrieval and generation times were approximately 7.7 milliseconds and 74.4 milliseconds, respectively.

#### Limitations

1. Only one question was evaluated.
2. Retrieval size was fixed at K=1.
3. The generated answer was factually supported but incomplete.
4. No automated grounding or hallucination metric was calculated.
5. The timing results represent a single execution and do not establish average performance.

#### Conclusion

The end-to-end RAG pipeline was successfully integrated and verified.

The experiment confirms that the system can retrieve Wikipedia evidence and generate an answer using Flan-T5-small.

Further experiments will investigate retrieval-size sensitivity, evidence-fusion strategies, answer quality and computational performance.


In [42]:
# ============================================================
# Stage 5.4.3 — Verify Reusable RAG Pipeline
# ============================================================

from rag.pipeline import RAGPipeline

print("=== REUSABLE RAG PIPELINE VERIFICATION ===")

# ------------------------------------------------------------
# Step 1: Initialize pipeline using existing components
# ------------------------------------------------------------

rag_pipeline = RAGPipeline(
    retriever=retriever,
    generator=generator
)

print("\nPipeline initialized successfully.")

# Verify existing objects are reused
assert rag_pipeline.retriever is retriever
assert rag_pipeline.generator is generator

print("Existing retriever reused: Passed")
print("Existing generator reused: Passed")

# ------------------------------------------------------------
# Step 2: Execute end-to-end RAG
# ------------------------------------------------------------

QUESTION = "Who is Diana Richburg?"

pipeline_result = rag_pipeline.answer(
    question=QUESTION,
    top_k=1
)

print("\n=== GENERATED ANSWER ===")

print("Question:", pipeline_result["question"])
print("Answer:", pipeline_result["answer"])

# ------------------------------------------------------------
# Step 3: Display retrieved evidence
# ------------------------------------------------------------

print("\n=== RETRIEVED EVIDENCE ===")

for source in pipeline_result["sources"]:

    print("Article:", source["title"])
    print("Similarity:", round(source["score"], 4))
    print("Chunk ID:", source["chunk_id"])
    print("URL:", source["url"])

# ------------------------------------------------------------
# Step 4: Display execution metrics
# ------------------------------------------------------------

print("\n=== PERFORMANCE METRICS ===")

print(
    "Retrieval time:",
    f"{pipeline_result['retrieval_time_seconds']:.4f} seconds"
)

print(
    "Generation time:",
    f"{pipeline_result['generation_time_seconds']:.4f} seconds"
)

print(
    "Total pipeline time:",
    f"{pipeline_result['total_time_seconds']:.4f} seconds"
)

print("Input tokens:", pipeline_result["input_tokens"])
print("Generated tokens:", pipeline_result["generated_tokens"])
print("Device:", pipeline_result["device"])

# ------------------------------------------------------------
# Step 5: Verify consistency with Jupyter prototype
# ------------------------------------------------------------

print("\n=== CONSISTENCY CHECKS ===")

assert pipeline_result["answer"] == rag_result["answer"], \
    "Generated answer differs from Jupyter prototype!"

print("Generated answer consistency: Passed")

assert pipeline_result["input_tokens"] == rag_result["input_tokens"], \
    "Input token count differs!"

print("Input token consistency: Passed")

assert (
    pipeline_result["sources"][0]["chunk_id"]
    == rag_result["sources"][0]["chunk_id"]
), "Retrieved passage differs!"

print("Retrieved passage consistency: Passed")

assert pipeline_result["total_time_seconds"] > 0

print("Pipeline timing verification: Passed")

print("\nAll reusable RAG pipeline verification checks passed.")

=== REUSABLE RAG PIPELINE VERIFICATION ===

Pipeline initialized successfully.
Existing retriever reused: Passed
Existing generator reused: Passed

=== GENERATED ANSWER ===
Question: Who is Diana Richburg?
Answer: middle-distance runner

=== RETRIEVED EVIDENCE ===
Article: Diana Richburg
Similarity: 0.3932
Chunk ID: 55617566::0
URL: https://en.wikipedia.org/wiki/Diana%20Richburg

=== PERFORMANCE METRICS ===
Retrieval time: 0.0067 seconds
Generation time: 0.1968 seconds
Total pipeline time: 0.2062 seconds
Input tokens: 171
Generated tokens: 9
Device: cuda

=== CONSISTENCY CHECKS ===
Generated answer consistency: Passed
Input token consistency: Passed
Retrieved passage consistency: Passed
Pipeline timing verification: Passed

All reusable RAG pipeline verification checks passed.


In [43]:
# ============================================================
# Stage 6.1.1 — Inspect Candidate Evaluation Questions
# ============================================================

import pandas as pd

print("=== RAG EVALUATION DATASET PREPARATION ===")

# Initial candidate questions
candidate_questions = [
    {
        "question_id": "Q001",
        "question": "Who is Diana Richburg?",
        "candidate_reference": "An American former middle-distance runner"
    },
    {
        "question_id": "Q002",
        "question": "Which Olympic Games did Diana Richburg compete in?",
        "candidate_reference": "1984 Los Angeles Olympics"
    },
    {
        "question_id": "Q003",
        "question": "Which US national title did Diana Richburg win in 1985?",
        "candidate_reference": "1500 metres"
    }
]

# Convert candidates into a DataFrame
candidate_df = pd.DataFrame(candidate_questions)

print("\n=== CANDIDATE QUESTIONS ===")
display(candidate_df)

# Retrieve supporting evidence for each question
for candidate in candidate_questions:

    print(f"\n{'=' * 70}")
    print("Question ID:", candidate["question_id"])
    print("Question:", candidate["question"])
    print("Candidate reference:", candidate["candidate_reference"])

    results = retriever.retrieve(
        question=candidate["question"],
        top_k=1
    )

    top_result = results[0]

    print("\nTop retrieved article:", top_result["title"])
    print("Similarity:", round(top_result["score"], 4))
    print("Chunk ID:", top_result["chunk_id"])

    print("\nRetrieved evidence:")
    print(top_result["text"])

print("\nCandidate evidence inspection completed.")

=== RAG EVALUATION DATASET PREPARATION ===

=== CANDIDATE QUESTIONS ===


,question_id,question,candidate_reference
0,Q001,Who is Diana Richburg?,An American former middle-distance runner
1,Q002,Which Olympic Games did Diana Richburg compete...,1984 Los Angeles Olympics
2,Q003,Which US national title did Diana Richburg win...,1500 metres



Question ID: Q001
Question: Who is Diana Richburg?
Candidate reference: An American former middle-distance runner

Top retrieved article: Diana Richburg
Similarity: 0.3932
Chunk ID: 55617566::0

Retrieved evidence:
Diana Lynn Richburg (born July 2, 1963) is an American former middle-distance runner. She competed at the 1984 Los Angeles Olympics, and won the 1985 US national title at 1500 metres. Career Born in Troy, New York, Richburg finished fourth in the 800m at the 1984 US Olympic Trials, before earning selection for the Los Angeles Olympics by finishing third in the 1500m in 4:04.07. At the Olympics, she failed to reach the final. She won the 1500m title at the 1985 USA Outdoor Track and Field Championships in 4:04.73. Richburg achieved her lifetime best for 1500m when finishing seventh

Question ID: Q002
Question: Which Olympic Games did Diana Richburg compete in?
Candidate reference: 1984 Los Angeles Olympics

Top retrieved article: Diana Richburg
Similarity: 0.3197
Chunk ID: 5

In [ ]:
# ============================================================
# Stage 6.1.2 — Inspect Diverse Wikipedia Articles
# ============================================================

import random

print("=== DIVERSE WIKIPEDIA ARTICLE SELECTION ===")

RANDOM_SEED = 42
NUM_ARTICLES = 8

# Create a local random generator for reproducibility
rng = random.Random(RANDOM_SEED)

# Keep one candidate passage per Wikipedia article.
# The first chunk often contains useful introductory facts.
first_chunks = {}

for passage in chunks_local:

    article_id = passage["article_id"]

    if (article_id not in first_chunks and passage["chunk_number"] == 0):
        first_chunks[article_id] = passage

# Exclude Diana Richburg because we already evaluated it
candidate_passages = [
    passage
    for passage in first_chunks.values()
    if passage["title"] != "Diana Richburg"
]

# Select distinct articles reproducibly
selected_articles = rng.sample(
    candidate_passages,
    k=min(NUM_ARTICLES, len(candidate_passages))
)

print("Unique available articles:", len(candidate_passages))
print("Selected articles:", len(selected_articles))

for i, article in enumerate(selected_articles, start=1):

    print(f"\n{'=' * 70}")
    print(f"ARTICLE {i}: {article['title']}")
    print(f"{'=' * 70}")
    print("Article ID:", article["article_id"])
    print("Chunk ID:", article["chunk_id"])
    print("Wikipedia URL:", article["url"])
    print("\nEvidence:")
    print(article["text"])

print("\nDiverse article inspection completed.")

=== DIVERSE WIKIPEDIA ARTICLE SELECTION ===
Unique available articles: 999
Selected articles: 8

ARTICLE 1: Mir Basri
Article ID: 59669912
Chunk ID: 59669912::0
Wikipedia URL: https://en.wikipedia.org/wiki/Mir%20Basri

Evidence:
Mir (also transliterated as Me'īr and Meer) S. Baṣrī (; 1911-2006) was an Iraqi Jewish writer, economist, journalist, and poet. Among many public positions he held, Basri served as the head and central leader of Baghdad's Jewish community. Life Basri was born on 19 September 1911 in Baghdad, Iraq to Shaool Basri and Farha Dangoor (the daughter of the Chief Rabbi of Baghdad, Ezra Reuben Dangoor). Basri was educated Baghdad at al-Ta'awun and the Alliance school where he studied Hebrew, English, and French. Following his secondary education, he trained as an economist and studied Arabic literature. In 1928, Basri joined

ARTICLE 2: 1993–94 Michigan Wolverines men's basketball team
Article ID: 28824182
Chunk ID: 28824182::0
Wikipedia URL: https://en.wikipedia.org/w

In [45]:
# ============================================================
# Stage 6.1.3 — Diverse Evaluation Question Verification
# ============================================================

import pandas as pd

diverse_questions = [
    {
        "question_id": "Q004",
        "question": "In which city was Mir Basri born?",
        "reference_answer": "Baghdad",
        "source_article": "Mir Basri"
    },
    {
        "question_id": "Q005",
        "question": "Who coached the 1993-94 Michigan Wolverines basketball team?",
        "reference_answer": "Steve Fisher",
        "source_article": "1993–94 Michigan Wolverines men's basketball team"
    },
    {
        "question_id": "Q006",
        "question": "Who was the captain of the 1954 Harvard Crimson football team?",
        "reference_answer": "J. Timothy Anderson",
        "source_article": "1954 Harvard Crimson football team"
    },
    {
        "question_id": "Q007",
        "question": "What numbered Chinese Super League season was 2018 for Shanghai Greenland Shenhua?",
        "reference_answer": "15th",
        "source_article": "2018 Shanghai Greenland Shenhua F.C. season"
    },
    {
        "question_id": "Q008",
        "question": "In which month and year was Kavbiuro established?",
        "reference_answer": "April 1920",
        "source_article": "Kavbiuro"
    },
    {
        "question_id": "Q009",
        "question": "What was B. Reith's first album called?",
        "reference_answer": "Urgent Message",
        "source_article": "B. Reith"
    },
    {
        "question_id": "Q010",
        "question": "In which Indian state is Gir National Park located?",
        "reference_answer": "Gujarat",
        "source_article": "Gir National Park"
    },
    {
        "question_id": "Q011",
        "question": "What architectural style is Puerto Princesa Cathedral associated with?",
        "reference_answer": "Neo-Gothic",
        "source_article": "Puerto Princesa Cathedral"
    }
]

print("=== DIVERSE EVALUATION QUESTION VERIFICATION ===")

verification_results = []

for item in diverse_questions:

    results = retriever.retrieve(
        question=item["question"],
        top_k=1
    )

    top_result = results[0]

    correct_article = (
        top_result["title"] == item["source_article"]
    )

    verification_results.append({
        "question_id": item["question_id"],
        "question": item["question"],
        "expected_article": item["source_article"],
        "retrieved_article": top_result["title"],
        "similarity": top_result["score"],
        "expected_article_retrieved": correct_article
    })

    print(f"\n{'=' * 70}")
    print("Question:", item["question"])
    print("Expected article:", item["source_article"])
    print("Retrieved article:", top_result["title"])
    print("Similarity:", round(top_result["score"], 4))
    print("Expected article retrieved:", correct_article)

    print("\nRetrieved passage:")
    print(top_result["text"][:350])

verification_df = pd.DataFrame(verification_results)

print("\n=== RETRIEVAL VERIFICATION SUMMARY ===")

display(verification_df)

print(
    "\nExpected article retrieved:",
    int(verification_df["expected_article_retrieved"].sum()),
    "out of",
    len(verification_df)
)

print("\nDiverse evaluation verification completed.")

=== DIVERSE EVALUATION QUESTION VERIFICATION ===

Question: In which city was Mir Basri born?
Expected article: Mir Basri
Retrieved article: Mir Basri
Similarity: 0.4604
Expected article retrieved: True

Retrieved passage:
Mir (also transliterated as Me'īr and Meer) S. Baṣrī (; 1911-2006) was an Iraqi Jewish writer, economist, journalist, and poet. Among many public positions he held, Basri served as the head and central leader of Baghdad's Jewish community. Life Basri was born on 19 September 1911 in Baghdad, Iraq to Shaool Basri and Farha Dangoor (the daughter of t

Question: Who coached the 1993-94 Michigan Wolverines basketball team?
Expected article: 1993–94 Michigan Wolverines men's basketball team
Retrieved article: 1993–94 Michigan Wolverines men's basketball team
Similarity: 0.5279
Expected article retrieved: True

Retrieved passage:
The 1993–94 Michigan Wolverines men's basketball team represented the University of Michigan in intercollegiate college basketball during the 199

,question_id,question,expected_article,retrieved_article,similarity,expected_article_retrieved
0,Q004,In which city was Mir Basri born?,Mir Basri,Mir Basri,0.460358,True
1,Q005,Who coached the 1993-94 Michigan Wolverines ba...,1993–94 Michigan Wolverines men's basketball team,1993–94 Michigan Wolverines men's basketball team,0.527854,True
2,Q006,Who was the captain of the 1954 Harvard Crimso...,1954 Harvard Crimson football team,1954 Harvard Crimson football team,0.779658,True
3,Q007,What numbered Chinese Super League season was ...,2018 Shanghai Greenland Shenhua F.C. season,2018 Shanghai Greenland Shenhua F.C. season,0.576531,True
4,Q008,In which month and year was Kavbiuro established?,Kavbiuro,Kavbiuro,0.408148,True
5,Q009,What was B. Reith's first album called?,B. Reith,B. Reith,0.566713,True
6,Q010,In which Indian state is Gir National Park loc...,Gir National Park,Gir National Park,0.427565,True
7,Q011,What architectural style is Puerto Princesa Ca...,Puerto Princesa Cathedral,Puerto Princesa Cathedral,0.686083,True



Expected article retrieved: 8 out of 8

Diverse evaluation verification completed.


In [ ]:
# ============================================================
# Stage 6.1.4 — Correct Ambiguous Evaluation Question
# ============================================================

print("=== EVALUATION QUESTION CORRECTION ===")

for item in diverse_questions:

    if item["question_id"] == "Q009":

        print("\nOriginal question:", item["question"])
        print("Original reference:", item["reference_answer"])

        # Correct the question to match the verified evidence
        item["question"] = ("What was B. Reith's first full-length album?")
        item["reference_answer"] = "Now Is Not Forever"

        print("\nRevised question:", item["question"])
        print("Revised reference:", item["reference_answer"])

        # Retrieve evidence for the revised question
        results = retriever.retrieve(
            question=item["question"],
            top_k=1
        )

        top_result = results[0]

        print("\n=== RETRIEVED EVIDENCE ===")
        print("Article:", top_result["title"])
        print("Similarity:", round(top_result["score"], 4))
        print("Chunk ID:", top_result["chunk_id"])
        print("\nPassage:")
        print(top_result["text"])

        break

print("\nEvaluation question correction completed.")

=== EVALUATION QUESTION CORRECTION ===

Original question: What was B. Reith's first album called?
Original reference: Urgent Message

Revised question: What was B. Reith's first full-length album?
Revised reference: Now Is Not Forever

=== RETRIEVED EVIDENCE ===
Article: B. Reith
Similarity: 0.5695
Chunk ID: 21921907::4

Passage:
music videos. Reith's first full-length album, Now Is Not Forever, was released on September 22, 2009. In late 2009, Reith toured again with TobyMac on the Winter Wonder Slam tour. In early 2011, Reith released a buzz single "Knockin On My Door". His third studio album, How the Story Ends, was released on November 1, 2011. On September 2, 2016, Reith released his crowdfunded EP Heart on My Sleeve, through Pledge Music. The EP was released on regular digital markets on October 14, 2016. On October 21, 2017, Reith released the EP Heart on My Sleeve (Acoustic). Style Reith's style

Evaluation question correction completed.


In [47]:
# ============================================================
# Stage 6.1.5 — Select Additional Evaluation Articles
# ============================================================

import random

print("=== ADDITIONAL EVALUATION ARTICLE SELECTION ===")

RANDOM_SEED = 101
NUM_NEW_ARTICLES = 5

rng = random.Random(RANDOM_SEED)

# Identify articles already used for evaluation
used_article_titles = {
    "Diana Richburg"
}

used_article_titles.update(
    item["source_article"]
    for item in diverse_questions
)

# Select first chunks from previously unused articles
available_articles = [
    passage
    for passage in first_chunks.values()
    if passage["title"] not in used_article_titles
]

additional_articles = rng.sample(
    available_articles,
    k=min(NUM_NEW_ARTICLES, len(available_articles))
)

print("Previously used articles:", len(used_article_titles))
print("Additional articles selected:", len(additional_articles))

for i, article in enumerate(additional_articles, start=1):

    print(f"\n{'=' * 70}")
    print(f"ARTICLE {i}: {article['title']}")
    print(f"{'=' * 70}")

    print("Article ID:", article["article_id"])
    print("Chunk ID:", article["chunk_id"])
    print("Wikipedia URL:", article["url"])

    print("\nEvidence:")
    print(article["text"])

print("\nAdditional article inspection completed.")

=== ADDITIONAL EVALUATION ARTICLE SELECTION ===
Previously used articles: 9
Additional articles selected: 5

ARTICLE 1: Agonothetes
Article ID: 1301575
Chunk ID: 1301575::0
Wikipedia URL: https://en.wikipedia.org/wiki/Agonothetes

Evidence:
In ancient Greece, an agonothetes (plural ; ) was the president or superintendent of one of the sacred Panhellenic Games. Alternative names for the same role included athlothetes, particularly in Athens. At first the person who instituted the games and defrayed the expenses was the agonothetes; but in the great public games, such as the Olympic Games and Pythian Games, these presidents were the representatives of different states, or were chosen from the people in whose country the games were celebrated; thus at the Panathenaic Games at Athens ten were elected for four years to superintend the various contests. In

ARTICLE 2: FK Tauras Tauragė
Article ID: 21922459
Chunk ID: 21922459::0
Wikipedia URL: https://en.wikipedia.org/wiki/FK%20Tauras%20Taura

In [48]:
# ============================================================
# Stage 6.1.6 — Verify Remaining Evaluation Questions
# ============================================================

import pandas as pd

additional_questions = [
    {
        "question_id": "Q012",
        "question": "What was the role of an agonothetes in ancient Greece?",
        "reference_answer": "President or superintendent of the sacred Panhellenic Games",
        "source_article": "Agonothetes"
    },
    {
        "question_id": "Q013",
        "question": "In what year was FK Tauras founded?",
        "reference_answer": "1922",
        "source_article": "FK Tauras Tauragė"
    },
    {
        "question_id": "Q014",
        "question": "Who portrays Kate Beckett in the television series Castle?",
        "reference_answer": "Stana Katic",
        "source_article": "Kate Beckett"
    },
    {
        "question_id": "Q015",
        "question": "Who was John Hubley's second wife?",
        "reference_answer": "Faith Hubley",
        "source_article": "John Hubley"
    },
    {
        "question_id": "Q016",
        "question": "From which university did Anthony J. DeMaria receive his Ph.D.?",
        "reference_answer": "University of Connecticut",
        "source_article": "Anthony J. DeMaria"
    },
    {
        "question_id": "Q017",
        "question": "In what year was Mir Basri born?",
        "reference_answer": "1911",
        "source_article": "Mir Basri"
    },
    {
        "question_id": "Q018",
        "question": "Where did the 1993-94 Michigan Wolverines basketball team play its home games?",
        "reference_answer": "Crisler Arena",
        "source_article": "1993–94 Michigan Wolverines men's basketball team"
    },
    {
        "question_id": "Q019",
        "question": "Who coached the 1954 Harvard Crimson football team?",
        "reference_answer": "Lloyd Jordan",
        "source_article": "1954 Harvard Crimson football team"
    },
    {
        "question_id": "Q020",
        "question": "In which year was Gir National Park established?",
        "reference_answer": "1965",
        "source_article": "Gir National Park"
    }
]

print("=== FINAL EVALUATION CANDIDATE VERIFICATION ===")

verification_records = []

for item in additional_questions:

    results = retriever.retrieve(
        question=item["question"],
        top_k=1
    )

    top_result = results[0]

    article_match = (
        top_result["title"] == item["source_article"]
    )

    verification_records.append({
        "question_id": item["question_id"],
        "expected_article": item["source_article"],
        "retrieved_article": top_result["title"],
        "similarity": top_result["score"],
        "article_match": article_match
    })

    print(f"\n{'=' * 65}")
    print("Question ID:", item["question_id"])
    print("Question:", item["question"])
    print("Reference:", item["reference_answer"])
    print("Retrieved article:", top_result["title"])
    print("Similarity:", round(top_result["score"], 4))
    print("Expected article retrieved:", article_match)

    print("\nRetrieved evidence:")
    print(top_result["text"][:500])

verification_remaining_df = pd.DataFrame(
    verification_records
)

print("\n=== VERIFICATION SUMMARY ===")

display(verification_remaining_df)

print(
    "\nExpected article retrieved:",
    int(verification_remaining_df["article_match"].sum()),
    "out of",
    len(verification_remaining_df)
)

print("\nRemaining evaluation question verification completed.")

=== FINAL EVALUATION CANDIDATE VERIFICATION ===

Question ID: Q012
Question: What was the role of an agonothetes in ancient Greece?
Reference: President or superintendent of the sacred Panhellenic Games
Retrieved article: Agonothetes
Similarity: 0.3629
Expected article retrieved: True

Retrieved evidence:
In ancient Greece, an agonothetes (plural ; ) was the president or superintendent of one of the sacred Panhellenic Games. Alternative names for the same role included athlothetes, particularly in Athens. At first the person who instituted the games and defrayed the expenses was the agonothetes; but in the great public games, such as the Olympic Games and Pythian Games, these presidents were the representatives of different states, or were chosen from the people in whose country the games were ce

Question ID: Q013
Question: In what year was FK Tauras founded?
Reference: 1922
Retrieved article: FK Tauras Tauragė
Similarity: 0.5754
Expected article retrieved: True

Retrieved evidence:
T

,question_id,expected_article,retrieved_article,similarity,article_match
0,Q012,Agonothetes,Agonothetes,0.362875,True
1,Q013,FK Tauras Tauragė,FK Tauras Tauragė,0.575384,True
2,Q014,Kate Beckett,Kate Beckett,0.349863,True
3,Q015,John Hubley,John Hubley,0.370328,True
4,Q016,Anthony J. DeMaria,Anthony J. DeMaria,0.313757,True
5,Q017,Mir Basri,Mir Basri,0.463979,True
6,Q018,1993–94 Michigan Wolverines men's basketball team,1993–94 Michigan Wolverines men's basketball team,0.552827,True
7,Q019,1954 Harvard Crimson football team,1954 Harvard Crimson football team,0.696390,True
8,Q020,Gir National Park,Gir National Park,0.478607,True



Expected article retrieved: 9 out of 9

Remaining evaluation question verification completed.


In [49]:
# ============================================================
# Stage 6.1.7 — Passage-Level Evidence Verification
# ============================================================

print("=== PASSAGE-LEVEL EVIDENCE VERIFICATION ===")

questions_to_inspect = [
    item
    for item in additional_questions
    if item["question_id"] in ["Q014", "Q015"]
]

for item in questions_to_inspect:

    print(f"\n{'=' * 70}")
    print("Question ID:", item["question_id"])
    print("Question:", item["question"])
    print("Reference answer:", item["reference_answer"])

    results = retriever.retrieve(
        question=item["question"],
        top_k=5
    )

    for result in results:

        # Diagnostic substring check.
        # This is not a complete semantic relevance metric.
        answer_present = (
            item["reference_answer"].casefold()
            in result["text"].casefold()
        )

        print(f"\n{'-' * 60}")
        print("Rank:", result["rank"])
        print("Article:", result["title"])
        print("Chunk ID:", result["chunk_id"])
        print("Similarity:", round(result["score"], 4))
        print("Reference text present:", answer_present)

        print("\nComplete retrieved passage:")
        print(result["text"])

print("\nPassage-level evidence inspection completed.")

=== PASSAGE-LEVEL EVIDENCE VERIFICATION ===

Question ID: Q014
Question: Who portrays Kate Beckett in the television series Castle?
Reference answer: Stana Katic

------------------------------------------------------------
Rank: 1
Article: Kate Beckett
Chunk ID: 21922319::84
Similarity: 0.3499
Reference text present: False

Complete retrieved passage:
when Castle wakes up in his bed the next morning to find a semi-clothed Beckett walking into his bedroom with coffee. They speak about their night, with Beckett asking Castle if he liked it, to which Castle replies with a yes. They both agree that it was not a one time thing, but their conversation is interrupted when Castle's mother comes home, causing Castle to send Beckett to hide in his closet. Beckett then has to do a secretive "Walk of Shame" out of Castle's loft, so that Martha and Alexis don't see her. Castle then comes to visit her

------------------------------------------------------------
Rank: 2
Article: Kate Beckett
Chunk 

In [50]:
# ============================================================
# Stage 6.1.8 — Build and Validate Evaluation Dataset
# ============================================================


print("=== FINAL EVALUATION DATASET PREPARATION ===")

# Convert the first three questions to the common structure
initial_questions = [
    {
        "question_id": item["question_id"],
        "question": item["question"],
        "reference_answer": item["candidate_reference"],
        "source_article": "Diana Richburg"
    }
    for item in candidate_questions
]

# Combine all 20 questions
evaluation_questions = (
    initial_questions
    + diverse_questions
    + additional_questions
)

evaluation_df = pd.DataFrame(evaluation_questions)

print("\n=== DATASET INFORMATION ===")
print("Total questions:", len(evaluation_df))
print("Unique question IDs:",evaluation_df["question_id"].nunique())
print("Unique source articles:",evaluation_df["source_article"].nunique())

# Verify expected dataset structure
assert len(evaluation_df) == 20
assert evaluation_df["question_id"].nunique() == 20
assert not evaluation_df[["question", "reference_answer", "source_article"]].isna().any().any()
assert (evaluation_df["question"].str.strip() != "").all()
assert (evaluation_df["reference_answer"].str.strip() != "").all()

# Verify that every expected source article is in our local corpus
available_titles = set(chunks_local["title"])

missing_sources = (
    set(evaluation_df["source_article"])
    - available_titles
)

assert not missing_sources, (f"Missing source articles: {missing_sources}")
print("\n=== QUESTION PREVIEW ===")

display(evaluation_df)

print("\nDataset structure verification: Passed")
print("Question ID uniqueness: Passed")
print("Missing-value verification: Passed")
print("Source availability verification: Passed")
print("\nEvaluation dataset preparation completed.")

=== FINAL EVALUATION DATASET PREPARATION ===

=== DATASET INFORMATION ===
Total questions: 20
Unique question IDs: 20
Unique source articles: 14

=== QUESTION PREVIEW ===


,question_id,question,reference_answer,source_article
0,Q001,Who is Diana Richburg?,An American former middle-distance runner,Diana Richburg
1,Q002,Which Olympic Games did Diana Richburg compete...,1984 Los Angeles Olympics,Diana Richburg
2,Q003,Which US national title did Diana Richburg win...,1500 metres,Diana Richburg
3,Q004,In which city was Mir Basri born?,Baghdad,Mir Basri
4,Q005,Who coached the 1993-94 Michigan Wolverines ba...,Steve Fisher,1993–94 Michigan Wolverines men's basketball team
5,Q006,Who was the captain of the 1954 Harvard Crimso...,J. Timothy Anderson,1954 Harvard Crimson football team
6,Q007,What numbered Chinese Super League season was ...,15th,2018 Shanghai Greenland Shenhua F.C. season
7,Q008,In which month and year was Kavbiuro established?,April 1920,Kavbiuro
8,Q009,What was B. Reith's first full-length album?,Now Is Not Forever,B. Reith
9,Q010,In which Indian state is Gir National Park loc...,Gujarat,Gir National Park



Dataset structure verification: Passed
Question ID uniqueness: Passed
Missing-value verification: Passed
Source availability verification: Passed

Evaluation dataset preparation completed.


In [51]:
# ============================================================
# Stage 6.1.9 — Associate Reference Evidence with Questions
# ============================================================

print("=== REFERENCE EVIDENCE VERIFICATION ===")

# Record known supporting evidence for the exceptional cases.
supporting_chunk_overrides = {
    "Q009": "21921907::4",
    "Q014": "21922319::0"
}

# Build a lookup of all local chunks
chunk_lookup = {
    passage["chunk_id"]: passage
    for passage in chunks_local
}

# First chunk of each article
first_chunk_lookup = {
    passage["title"]: passage
    for passage in chunks_local
    if passage["chunk_number"] == 0
}

supporting_chunk_ids = []
verification_records = []

for _, row in evaluation_df.iterrows():

    question_id = row["question_id"]
    reference = row["reference_answer"]
    source_article = row["source_article"]

    # Use an explicitly selected supporting chunk if required.
    if question_id in supporting_chunk_overrides:

        supporting_chunk = chunk_lookup[
            supporting_chunk_overrides[question_id]
        ]

    else:

        supporting_chunk = first_chunk_lookup[
            source_article
        ]

    # Basic textual check
    reference_present = (
        reference.casefold()
        in supporting_chunk["text"].casefold()
    )

    # Q012 is supported by a paraphrase of the source sentence.
    if question_id == "Q012":
        text_lower = supporting_chunk["text"].casefold()

        reference_present = all(
            phrase in text_lower
            for phrase in [
                "president or superintendent",
                "sacred panhellenic games"
            ]
        )

    supporting_chunk_ids.append(
        supporting_chunk["chunk_id"]
    )

    verification_records.append({
        "question_id": question_id,
        "source_article": source_article,
        "supporting_chunk_id": supporting_chunk["chunk_id"],
        "reference_supported": reference_present
    })

# Add supporting chunk IDs to the evaluation dataset
evaluation_df["supporting_chunk_id"] = supporting_chunk_ids

evidence_verification_df = pd.DataFrame(
    verification_records
)

print("\n=== EVIDENCE VERIFICATION SUMMARY ===")

display(evidence_verification_df)

print(
    "\nReferences supported:",
    int(evidence_verification_df["reference_supported"].sum()),
    "out of",
    len(evidence_verification_df)
)

assert evidence_verification_df["reference_supported"].all(), \
    "At least one reference answer requires manual review!"

print("\nAll reference evidence verification checks passed.")

=== REFERENCE EVIDENCE VERIFICATION ===

=== EVIDENCE VERIFICATION SUMMARY ===


,question_id,source_article,supporting_chunk_id,reference_supported
0,Q001,Diana Richburg,55617566::0,True
1,Q002,Diana Richburg,55617566::0,True
2,Q003,Diana Richburg,55617566::0,True
3,Q004,Mir Basri,59669912::0,True
4,Q005,1993–94 Michigan Wolverines men's basketball team,28824182::0,True
5,Q006,1954 Harvard Crimson football team,64509002::0,True
6,Q007,2018 Shanghai Greenland Shenhua F.C. season,55620591::0,True
7,Q008,Kavbiuro,41016179::0,True
8,Q009,B. Reith,21921907::4,True
9,Q010,Gir National Park,1302663::0,True



References supported: 20 out of 20

All reference evidence verification checks passed.


In [52]:
# ============================================================
# Stage 6.1.10 — Save Verified RAG Evaluation Dataset
# ============================================================

from pathlib import Path
import json

print("=== RAG EVALUATION DATASET PERSISTENCE ===")

# Use the existing project root
EVALUATION_DIR = PROJECT_ROOT / "data" / "evaluation"

CSV_PATH = EVALUATION_DIR / "rag_evaluation_20.csv"
JSON_PATH = EVALUATION_DIR / "rag_evaluation_20.json"

# Avoid accidentally overwriting existing evaluation files
existing_files = [
    path
    for path in [CSV_PATH, JSON_PATH]
    if path.exists()
]

if existing_files:
    raise FileExistsError(
        f"Evaluation files already exist: {existing_files}"
    )

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Keep a fixed column order
evaluation_columns = [
    "question_id",
    "question",
    "reference_answer",
    "source_article",
    "supporting_chunk_id"
]

final_evaluation_df = evaluation_df[
    evaluation_columns
].copy()

# Verify dataset integrity before saving
assert len(final_evaluation_df) == 20

assert final_evaluation_df[
    "question_id"
].is_unique

assert final_evaluation_df.notna().all().all()

# Save as CSV
final_evaluation_df.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8"
)

# Save as JSON
evaluation_records = final_evaluation_df.to_dict(
    orient="records"
)

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_records,
        f,
        indent=4,
        ensure_ascii=False
    )

print("\n=== SAVED EVALUATION FILES ===")

print("CSV:", CSV_PATH)
print("JSON:", JSON_PATH)

print("\nNumber of saved questions:", len(final_evaluation_df))

# Verify that both files exist
assert CSV_PATH.is_file()
assert JSON_PATH.is_file()

# Reload both files to verify persistence
reloaded_csv = __import__("pandas").read_csv(
    CSV_PATH,
    encoding="utf-8"
)

with open(JSON_PATH, "r", encoding="utf-8") as f:
    reloaded_json = json.load(f)

assert len(reloaded_csv) == 20
assert len(reloaded_json) == 20

assert (
    reloaded_csv["question_id"].tolist()
    == final_evaluation_df["question_id"].tolist()
)

print("\nCSV reload verification: Passed")
print("JSON reload verification: Passed")

print("\nEvaluation dataset persistence completed.")

=== RAG EVALUATION DATASET PERSISTENCE ===

=== SAVED EVALUATION FILES ===
CSV: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\data\evaluation\rag_evaluation_20.csv
JSON: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\data\evaluation\rag_evaluation_20.json

Number of saved questions: 20

CSV reload verification: Passed
JSON reload verification: Passed

Evaluation dataset persistence completed.


## Stage 6.1 — Construction of the RAG Evaluation Dataset

### Objective

To construct a fixed, reproducible question-answering evaluation dataset for comparing retrieval sizes and evidence-fusion strategies in the offline Wikipedia RAG system.

### Methodology

A corpus-specific evaluation dataset was constructed using factual information from the locally indexed Wikipedia articles.

Twenty questions were prepared across fourteen distinct Wikipedia articles, covering topics such as history, sports, geography, music, television, wildlife and scientific research.

Each evaluation record contains:

1. A unique question identifier.
2. A natural-language question.
3. A reference answer.
4. The source Wikipedia article.
5. The identifier of a supporting evidence passage.

Reference answers were verified against the actual text stored in the local Wikipedia corpus.

Questions with ambiguous reference answers were corrected before finalizing the evaluation dataset.

The evaluation set was persisted in CSV and JSON formats for reproducibility.

### Evaluation Dataset Characteristics

| Property                                       |     Value |
| ---------------------------------------------- | --------: |
| Total questions                                |        20 |
| Distinct source articles                       |        14 |
| Unique question identifiers                    |        20 |
| References with identified supporting evidence |        20 |
| Missing evaluation fields                      |         0 |
| Persistence formats                            | CSV, JSON |

### Reference Evidence Verification

For every question, a supporting Wikipedia passage was identified.

This establishes that the reference answer is supported by the selected corpus.

However, the existence of a supporting passage does not guarantee that the retrieval system will return it among the highest-ranked passages.

For example, the Kate Beckett question demonstrated a passage-ranking failure.

The correct answer, "Stana Katic", was present in the indexed Wikipedia article, but none of the five highest-ranked retrieved passages contained that reference answer.

This distinction motivates separate evaluation of retrieval quality and generated-answer quality.

### Experimental Design

The finalized evaluation dataset will remain fixed across experimental configurations.

Retrieval size and evidence-fusion strategy will be varied while maintaining consistent questions, reference answers, corpus and evaluation criteria.

The primary experimental variables will include:

$$
K \in \{1,3,5\}
$$

and two evidence-fusion approaches:

1. Concatenation-based generation.
2. Generate-from-evidence fusion.

### Limitations

The evaluation dataset is relatively small and was constructed from a selected subset of Wikipedia.

Multiple questions originate from some of the same articles, creating potential dependence between evaluation examples.

Because the questions were constructed using the source corpus, the resulting evaluation does not establish performance on unseen Wikipedia articles or general open-domain question answering.

Furthermore, the current dataset contains factual questions intended to be answerable from the corpus. Unsupported-question behaviour will be assessed separately.

### Conclusion

A fixed evaluation dataset comprising twenty questions from fourteen Wikipedia articles was successfully constructed and verified.

Each question has a reference answer supported by an identified passage in the local corpus.

The evaluation dataset provides a consistent foundation for subsequent retrieval-size ablation, fusion-method comparison and grounded-answer evaluation.


In [53]:
# ============================================================
# Stage 6.2.1 — Token Budget Analysis Across Evaluation Dataset
# ============================================================

import pandas as pd

print("=== EVALUATION TOKEN BUDGET ANALYSIS ===")

K_VALUES = [1, 3, 5]
MAX_INPUT_TOKENS = 512

token_budget_records = []

for _, item in final_evaluation_df.iterrows():

    question = item["question"]

    for k in K_VALUES:

        # Retrieve evidence
        passages = retriever.retrieve(
            question=question,
            top_k=k
        )

        # Use the generator's existing prompt construction
        prompt = generator.build_prompt(
            question=question,
            passages=passages
        )

        # Measure complete prompt length
        token_ids = tokenizer(
            prompt,
            truncation=False
        )["input_ids"]

        token_count = len(token_ids)

        token_budget_records.append({
            "question_id": item["question_id"],
            "k": k,
            "input_tokens": token_count,
            "within_budget": token_count <= MAX_INPUT_TOKENS,
            "excess_tokens": max(
                0,
                token_count - MAX_INPUT_TOKENS
            )
        })

token_budget_all_df = pd.DataFrame(
    token_budget_records
)

print("\n=== TOKEN BUDGET SUMMARY BY K ===")

summary = token_budget_all_df.groupby("k").agg(
    questions=("question_id", "count"),
    min_tokens=("input_tokens", "min"),
    mean_tokens=("input_tokens", "mean"),
    max_tokens=("input_tokens", "max"),
    within_budget=("within_budget", "sum")
)

summary["exceeding_budget"] = (
    summary["questions"]
    - summary["within_budget"]
)

display(summary)

print("\n=== QUESTIONS EXCEEDING TOKEN BUDGET ===")

over_budget = token_budget_all_df[
    ~token_budget_all_df["within_budget"]
]

display(over_budget)

print("\nToken budget analysis completed.")

=== EVALUATION TOKEN BUDGET ANALYSIS ===

=== TOKEN BUDGET SUMMARY BY K ===


,questions,min_tokens,mean_tokens,max_tokens,within_budget,exceeding_budget
k,,,,,,
1,20,150,191.80,232,20,0
3,20,357,471.85,583,17,3
5,20,582,748.90,932,0,20



=== QUESTIONS EXCEEDING TOKEN BUDGET ===


,question_id,k,input_tokens,within_budget,excess_tokens
2,Q001,5,725,False,213
5,Q002,5,633,False,121
8,Q003,5,768,False,256
11,Q004,5,932,False,420
14,Q005,5,625,False,113
17,Q006,5,604,False,92
20,Q007,5,742,False,230
23,Q008,5,782,False,270
26,Q009,5,793,False,281
28,Q010,3,524,False,12



Token budget analysis completed.


In [54]:
# ============================================================
# Stage 6.2.2 — Fixed Per-Passage Token Budget Experiment
# ============================================================

import pandas as pd

print("=== FIXED EVIDENCE TOKEN BUDGET EXPERIMENT ===")

K_VALUES = [1, 3, 5]

MAX_INPUT_TOKENS = 512
MAX_PASSAGE_TOKENS = 80


def shorten_passage(text, tokenizer, max_tokens=80):
    """
    Keep at most max_tokens tokenizer tokens from a passage.

    This explicitly shortens evidence before prompt construction.
    """

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    shortened_ids = token_ids[:max_tokens]

    shortened_text = tokenizer.decode(
        shortened_ids,
        skip_special_tokens=True
    )

    return shortened_text


budget_results = []

for _, item in final_evaluation_df.iterrows():

    question = item["question"]

    for k in K_VALUES:

        # Retrieve original passages
        original_passages = retriever.retrieve(
            question=question,
            top_k=k
        )

        shortened_passages = []

        for passage in original_passages:

            shortened = passage.copy()

            shortened["text"] = shorten_passage(
                passage["text"],
                tokenizer,
                max_tokens=MAX_PASSAGE_TOKENS
            )

            shortened_passages.append(shortened)

        # Build prompt using shortened evidence
        prompt = generator.build_prompt(
            question=question,
            passages=shortened_passages
        )

        prompt_tokens = len(
            tokenizer(
                prompt,
                truncation=False
            )["input_ids"]
        )

        budget_results.append({
            "question_id": item["question_id"],
            "k": k,
            "input_tokens": prompt_tokens,
            "within_budget": prompt_tokens <= MAX_INPUT_TOKENS
        })


shortened_budget_df = pd.DataFrame(
    budget_results
)

print("\n=== TOKEN BUDGET SUMMARY ===")

summary = shortened_budget_df.groupby("k").agg(
    questions=("question_id", "count"),
    min_tokens=("input_tokens", "min"),
    mean_tokens=("input_tokens", "mean"),
    max_tokens=("input_tokens", "max"),
    within_budget=("within_budget", "sum")
)

summary["exceeding_budget"] = (
    summary["questions"] -
    summary["within_budget"]
)

display(summary)

print("\n=== REMAINING OVER-BUDGET QUESTIONS ===")

display(
    shortened_budget_df[
        ~shortened_budget_df["within_budget"]
    ]
)

print("\nFixed evidence token budget experiment completed.")

=== FIXED EVIDENCE TOKEN BUDGET EXPERIMENT ===

=== TOKEN BUDGET SUMMARY ===


,questions,min_tokens,mean_tokens,max_tokens,within_budget,exceeding_budget
k,,,,,,
1,20,119,125.10,133,20,0
3,20,246,282.55,292,20,0
5,20,395,436.90,452,20,0



=== REMAINING OVER-BUDGET QUESTIONS ===


,question_id,k,input_tokens,within_budget



Fixed evidence token budget experiment completed.


In [55]:
# ============================================================
# Stage 6.2.3 — Evidence Retention After Shortening
# ============================================================

import pandas as pd

print("=== EVIDENCE RETENTION ANALYSIS ===")

K_VALUES = [1, 3, 5]
MAX_PASSAGE_TOKENS = 80


def contains_reference(question_id, reference, passages):
    """
    Check whether retrieved evidence contains the reference.

    This is a diagnostic textual check, not a semantic metric.
    """

    combined_text = " ".join(
        passage["text"]
        for passage in passages
    ).casefold()

    # Q012 has a paraphrased reference answer.
    if question_id == "Q012":
        return (
            "president or superintendent" in combined_text
            and "panhellenic games" in combined_text
        )

    return reference.casefold() in combined_text


retention_records = []

for _, item in final_evaluation_df.iterrows():

    question_id = item["question_id"]
    question = item["question"]
    reference = item["reference_answer"]

    for k in K_VALUES:

        original_passages = retriever.retrieve(
            question=question,
            top_k=k
        )

        shortened_passages = []

        for passage in original_passages:

            shortened = passage.copy()

            shortened["text"] = shorten_passage(
                passage["text"],
                tokenizer,
                max_tokens=MAX_PASSAGE_TOKENS
            )

            shortened_passages.append(shortened)

        original_has_answer = contains_reference(
            question_id,
            reference,
            original_passages
        )

        shortened_has_answer = contains_reference(
            question_id,
            reference,
            shortened_passages
        )

        retention_records.append({
            "question_id": question_id,
            "k": k,
            "original_has_answer": original_has_answer,
            "shortened_has_answer": shortened_has_answer,
            "answer_lost": (
                original_has_answer
                and not shortened_has_answer
            )
        })


retention_df = pd.DataFrame(
    retention_records
)

print("\n=== EVIDENCE RETENTION SUMMARY ===")

retention_summary = retention_df.groupby("k").agg(
    questions=("question_id", "count"),
    original_answer_present=("original_has_answer", "sum"),
    shortened_answer_present=("shortened_has_answer", "sum"),
    answers_lost=("answer_lost", "sum")
)

display(retention_summary)

print("\n=== CASES WHERE SHORTENING REMOVED AN ANSWER ===")

lost_answers_df = retention_df[
    retention_df["answer_lost"]
]

display(lost_answers_df)

print("\nEvidence retention analysis completed.")

=== EVIDENCE RETENTION ANALYSIS ===

=== EVIDENCE RETENTION SUMMARY ===


,questions,original_answer_present,shortened_answer_present,answers_lost
k,,,,
1,20,19,18,1
3,20,19,18,1
5,20,19,18,1



=== CASES WHERE SHORTENING REMOVED AN ANSWER ===


,question_id,k,original_has_answer,shortened_has_answer,answer_lost
42,Q015,1,True,False,True
43,Q015,3,True,False,True
44,Q015,5,True,False,True



Evidence retention analysis completed.


In [ ]:
# ============================================================
# Stage 6.2.4 — Investigate Lost Reference Evidence
# ============================================================

print("=== LOST EVIDENCE INVESTIGATION ===")

QUESTION_ID = "Q015"
REFERENCE = "Faith Hubley"

# Find the evaluation question
evaluation_row = final_evaluation_df[
    final_evaluation_df["question_id"] == QUESTION_ID
].iloc[0]

question = evaluation_row["question"]

# Retrieve the original top-ranked passage
original_passage = retriever.retrieve(
    question=question,
    top_k=1
)[0]

original_text = original_passage["text"]

# Find where the reference begins
reference_position = original_text.casefold().find(
    REFERENCE.casefold()
)

assert reference_position >= 0, "Reference was not found in the original passage!"

# Count tokens before the reference
prefix_text = original_text[:reference_position]

tokens_before_reference = len(
    tokenizer(
        prefix_text,
        add_special_tokens=False
    )["input_ids"]
)

# Create shortened version
shortened_text = shorten_passage(
    original_text,
    tokenizer,
    max_tokens=80
)

print("Question:", question)
print("Reference:", REFERENCE)
print("\n=== REFERENCE LOCATION ===")
print("Original passage words:", len(original_text.split()))
print("Original passage tokens:",len(
        tokenizer(
            original_text,
            add_special_tokens=False
        )["input_ids"]
    )
)

print("Tokens before reference:", tokens_before_reference)
print("\nReference present in original:",REFERENCE.casefold() in original_text.casefold())
print("Reference present after shortening:",REFERENCE.casefold() in shortened_text.casefold())
print("\n=== ORIGINAL PASSAGE ===")
print(original_text)
print("\n=== SHORTENED PASSAGE ===")
print(shortened_text)
print("\nLost evidence investigation completed.")

=== LOST EVIDENCE INVESTIGATION ===
Question: Who was John Hubley's second wife?
Reference: Faith Hubley

=== REFERENCE LOCATION ===
Original passage words: 100
Original passage tokens: 143
Tokens before reference: 112

Reference present in original: True
Reference present after shortening: False

=== ORIGINAL PASSAGE ===
John Kirkham Hubley (May 21, 1914 – February 21, 1977) was an American animated film director, art director, producer, and writer known for his work with the United Productions of America (UPA) and his own independent studio, Storyboard, Inc. (later renamed Hubley Studio). A pioneer and innovator in the American animation industry, Hubley pushed for more visually and emotionally complex films than those being produced by contemporaries like the Walt Disney Company and Warner Brothers Animation. He and his second wife, Faith Hubley (née Chestman), who he worked alongside from 1953 onward, were nominated for seven Academy Awards, winning three.

=== SHORTENED PASSAGE ==

In [58]:
# ============================================================
# Stage 6.2.5 — Query-Aware Sentence Selection Prototype
# ============================================================

import re

print("=== QUERY-AWARE EVIDENCE SELECTION ===")

def select_relevant_sentences(
        question,
        passage_text,
        vectorizer,
        tokenizer,
        max_tokens=80
):
    """
    Select relevant sentences from a passage using TF-IDF.

    Selection is based only on the question and passage text.
    Reference answers are never used.
    """
    
    # Step 1 : Split passage into sentence
    sentences = [
        sentence.strip()
        for sentence in re.split(
            r"(?<=[.!?])\s+",
            passage_text
        )
        if sentence.strip()
    ]
    
    if not sentences:
        return ""

   
    # Step 2 : Convert question and sentence into TF-IDF vectors
    question_vector = vectorizer.transform([question])
    sentence_vector = vectorizer.transform(sentences)


    # Step 3 : Calculate sentence relevance
    scores = cosine_similarity(
        question_vector,
        sentence_vector
    ).ravel()


    # Step 4 : Rank sentence by relevance
    ranked_indices = np.argsort(
        -scores,
        kind="stable"
    )
    
    selected_indices = []
    used_tokens = 0


    # Step 5 : Select sentence within the token budget

    for index in ranked_indices:
        sentence = sentences[int(index)]    
        sentence_tokens = tokenizer(
            sentence,
            add_special_tokens=False
        )["input_ids"]
        
        # Reserve one token for seperatino when needed
        separator_tokens = 1 if selected_indices else 0

        required_tokens = (
            len(sentence_tokens)
            + separator_tokens
        )

        if used_tokens + required_tokens <= max_tokens:
            selected_indices.append(int(index))
            used_tokens += required_tokens
    
    
    # Fallback if no complete sentence fits
    if not selected_indices:

        best_sentence = sentences[
            int(ranked_indices[0])
        ]

        best_tokens = tokenizer(
            best_sentence,
            add_special_tokens=False
        )["input_ids"]

        return tokenizer.decode(
            best_tokens[:max_tokens],
            skip_special_tokens=True
        )

    
    
    # Preserve original sentence order for readability
    selected_indices.sort()

    selected_text = " ".join(
        sentences[index]
        for index in selected_indices
    )
    
    
    # Ensure the decoded text satisfies the actual token budget
    selected_token_ids = tokenizer(
        selected_text,
        add_special_tokens=False
    )["input_ids"]

    if len(selected_token_ids) > max_tokens:

        selected_text = tokenizer.decode(
            selected_token_ids[:max_tokens],
            skip_special_tokens=True
        )

    return selected_text

# ------------------------------------------------------------
# Test on John Hubley
# ------------------------------------------------------------

QUESTION = "Who was John Hubley's second wife?"

original_passage = retriever.retrieve(
    question=QUESTION,
    top_k=1
)[0]

selected_evidence = select_relevant_sentences(
    question=QUESTION,
    passage_text=original_passage["text"],
    vectorizer=module_vectorizer,
    tokenizer=tokenizer,
    max_tokens=80
)

print("\nQuestion:", QUESTION)

print("\n=== SELECTED EVIDENCE ===")
print(selected_evidence)

selected_token_count = len(
    tokenizer(
        selected_evidence,
        add_special_tokens=False
    )["input_ids"]
)

print("\nSelected evidence tokens:", selected_token_count)

print(
    "Within 80-token budget:",
    selected_token_count <= 80
)

# Diagnostic verification only.
# The reference answer is NOT used by the selection function.
print(
    "Reference retained:",
    "faith hubley" in selected_evidence.casefold()
)

assert selected_token_count <= 80

print("\nQuery-aware sentence selection test completed.")


=== QUERY-AWARE EVIDENCE SELECTION ===

Question: Who was John Hubley's second wife?

=== SELECTED EVIDENCE ===
(later renamed Hubley Studio). He and his second wife, Faith Hubley (née Chestman), who he worked alongside from 1953 onward, were nominated for seven Academy Awards, winning three.

Selected evidence tokens: 48
Within 80-token budget: True
Reference retained: True

Query-aware sentence selection test completed.


In [60]:
# ============================================================
# Stage 6.2.6 — Compare Evidence Selection Strategies
# ============================================================

print("=== EVIDENCE SELECTION STRATEGY COMPARISON ===")

K_VALUES = [1, 3, 5]

MAX_PASSAGE_TOKENS = 80
MAX_INPUT_TOKENS = 512

comparison_records = []

for _, item in final_evaluation_df.iterrows():

    question_id = item["question_id"]
    question = item["question"]
    reference = item["reference_answer"]

    for k in K_VALUES:

        # ----------------------------------------------------
        # Step 1: Retrieve original passages
        # ----------------------------------------------------

        original_passages = retriever.retrieve(
            question=question,
            top_k=k
        )

        # ----------------------------------------------------
        # Step 2: Apply both evidence-selection strategies
        # ----------------------------------------------------

        prefix_passages = []
        query_aware_passages = []

        for passage in original_passages:

            # Prefix-based selection
            prefix_passage = passage.copy()

            prefix_passage["text"] = shorten_passage(
                passage["text"],
                tokenizer,
                max_tokens=MAX_PASSAGE_TOKENS
            )

            prefix_passages.append(prefix_passage)

            # Query-aware sentence selection
            query_passage = passage.copy()

            query_passage["text"] = select_relevant_sentences(
                question=question,
                passage_text=passage["text"],
                vectorizer=module_vectorizer,
                tokenizer=tokenizer,
                max_tokens=MAX_PASSAGE_TOKENS
            )

            query_aware_passages.append(query_passage)

        # ----------------------------------------------------
        # Step 3: Check reference evidence availability
        # ----------------------------------------------------

        original_has_answer = contains_reference(
            question_id,
            reference,
            original_passages
        )

        prefix_has_answer = contains_reference(
            question_id,
            reference,
            prefix_passages
        )

        query_aware_has_answer = contains_reference(
            question_id,
            reference,
            query_aware_passages
        )

        # ----------------------------------------------------
        # Step 4: Check complete query-aware prompt length
        # ----------------------------------------------------

        query_aware_prompt = generator.build_prompt(
            question=question,
            passages=query_aware_passages
        )

        input_tokens = len(
            tokenizer(
                query_aware_prompt,
                truncation=False
            )["input_ids"]
        )

        comparison_records.append({
            "question_id": question_id,
            "k": k,
            "original_has_answer": original_has_answer,
            "prefix_has_answer": prefix_has_answer,
            "query_aware_has_answer": query_aware_has_answer,
            "query_aware_input_tokens": input_tokens,
            "within_budget": input_tokens <= MAX_INPUT_TOKENS
        })


selection_comparison_df = pd.DataFrame(
    comparison_records
)

# ------------------------------------------------------------
# Step 5: Summarize results
# ------------------------------------------------------------

print("\n=== EVIDENCE RETENTION BY METHOD ===")

selection_summary = selection_comparison_df.groupby("k").agg(
    questions=("question_id", "count"),
    original_answer_present=("original_has_answer", "sum"),
    prefix_answer_present=("prefix_has_answer", "sum"),
    query_aware_answer_present=("query_aware_has_answer", "sum"),
    within_budget=("within_budget", "sum"),
    max_input_tokens=("query_aware_input_tokens", "max")
)

display(selection_summary)

# ------------------------------------------------------------
# Step 6: Identify differences between strategies
# ------------------------------------------------------------

print("\n=== QUESTIONS WITH DIFFERENT RETENTION RESULTS ===")

different_results = selection_comparison_df[
    selection_comparison_df["prefix_has_answer"]
    != selection_comparison_df["query_aware_has_answer"]
]

display(different_results)

# ------------------------------------------------------------
# Step 7: Check token budget
# ------------------------------------------------------------

print("\n=== TOKEN BUDGET VIOLATIONS ===")

over_budget = selection_comparison_df[
    ~selection_comparison_df["within_budget"]
]

display(over_budget)

print("\nEvidence selection comparison completed.")

=== EVIDENCE SELECTION STRATEGY COMPARISON ===

=== EVIDENCE RETENTION BY METHOD ===


,questions,original_answer_present,prefix_answer_present,query_aware_answer_present,within_budget,max_input_tokens
k,,,,,,
1,20,19,18,13,20,127
3,20,19,18,13,20,276
5,20,19,18,13,20,433



=== QUESTIONS WITH DIFFERENT RETENTION RESULTS ===


,question_id,k,original_has_answer,prefix_has_answer,query_aware_has_answer,query_aware_input_tokens,within_budget
3,Q002,1,True,True,False,121,True
4,Q002,3,True,True,False,228,True
5,Q002,5,True,True,False,350,True
12,Q005,1,True,True,False,108,True
13,Q005,3,True,True,False,256,True
14,Q005,5,True,True,False,360,True
30,Q011,1,True,True,False,123,True
31,Q011,3,True,True,False,263,True
32,Q011,5,True,True,False,371,True
42,Q015,1,True,False,True,91,True



=== TOKEN BUDGET VIOLATIONS ===


,question_id,k,original_has_answer,prefix_has_answer,query_aware_has_answer,query_aware_input_tokens,within_budget



Evidence selection comparison completed.


In [62]:
# ============================================================
# Stage 6.3.1 — Concatenation Fusion Experiment
# ============================================================


print("=== CONCATENATION FUSION EXPERIMENT ===")

QUESTION = "Who is Diana Richburg?"

K_VALUES = [1, 3, 5]
MAX_PASSAGE_TOKENS = 80

concatenation_results = []

for k in K_VALUES:

    print(f"\n{'=' * 65}")
    print(f"CONCATENATION — K={k}")
    print(f"{'=' * 65}")

    # Step 1: Retrieve K relevant passages
    retrieved_passages = retriever.retrieve(
        question=QUESTION,
        top_k=k
    )

    # Step 2: Apply our fixed Prefix-80 evidence policy
    prepared_passages = []

    for passage in retrieved_passages:

        prepared_passage = passage.copy()

        prepared_passage["text"] = shorten_passage(
            passage["text"],
            tokenizer,
            max_tokens=MAX_PASSAGE_TOKENS
        )

        prepared_passages.append(prepared_passage)

    # Step 3: Generate one answer from concatenated evidence
    result = generator.generate(
        question=QUESTION,
        passages=prepared_passages
    )

    # Step 4: Record actual results
    concatenation_results.append({
        "method": "concatenation",
        "k": k,
        "question": QUESTION,
        "answer": result["answer"],
        "input_tokens": result["input_tokens"],
        "generated_tokens": result["generated_tokens"],
        "generation_time_seconds":
            result["generation_time_seconds"]
    })

    print("\nGenerated answer:", result["answer"])
    print("Input tokens:", result["input_tokens"])
    print("Generated tokens:", result["generated_tokens"])
    print(
        "Generation time:",
        f"{result['generation_time_seconds']:.4f} seconds"
    )

    print("\nRetrieved sources:")

    for passage in retrieved_passages:
        print(
            f"Rank {passage['rank']}: "
            f"{passage['title']} "
            f"(similarity={passage['score']:.4f})"
        )

# Step 5: Display comparison table
concatenation_df = pd.DataFrame(
    concatenation_results
)

print("\n=== CONCATENATION RESULTS SUMMARY ===")

display(concatenation_df)

print("\nConcatenation fusion experiment completed.")

=== CONCATENATION FUSION EXPERIMENT ===

CONCATENATION — K=1

Generated answer: middle-distance runner
Input tokens: 119
Generated tokens: 9
Generation time: 0.1762 seconds

Retrieved sources:
Rank 1: Diana Richburg (similarity=0.3932)

CONCATENATION — K=3

Generated answer: Moldovan chess player
Input tokens: 279
Generated tokens: 8
Generation time: 0.0762 seconds

Retrieved sources:
Rank 1: Diana Richburg (similarity=0.3932)
Rank 2: Diana Richburg (similarity=0.2261)
Rank 3: Diana Baciu (similarity=0.1121)

CONCATENATION — K=5

Generated answer: a Moldovan chess player
Input tokens: 438
Generated tokens: 10
Generation time: 0.0832 seconds

Retrieved sources:
Rank 1: Diana Richburg (similarity=0.3932)
Rank 2: Diana Richburg (similarity=0.2261)
Rank 3: Diana Baciu (similarity=0.1121)
Rank 4: Natasha Lako (similarity=0.0979)
Rank 5: Maurice Hurley (similarity=0.0963)

=== CONCATENATION RESULTS SUMMARY ===


,method,k,question,answer,input_tokens,generated_tokens,generation_time_seconds
0,concatenation,1,Who is Diana Richburg?,middle-distance runner,119,9,0.176246
1,concatenation,3,Who is Diana Richburg?,Moldovan chess player,279,8,0.076204
2,concatenation,5,Who is Diana Richburg?,a Moldovan chess player,438,10,0.083200



Concatenation fusion experiment completed.


### Stage 6.3.1 — Concatenation-Based Evidence Fusion

#### Objective

To investigate the effect of retrieval size on answer generation when multiple Wikipedia passages are concatenated into a single evidence block.

#### Methodology

Three retrieval configurations were evaluated:

$$
K \in \{1,3,5\}
$$

For each configuration, the highest-ranked Wikipedia passages were retrieved using TF-IDF and cosine similarity.

Each passage was shortened to a maximum of 80 Flan-T5 tokens.

The selected passages were concatenated and provided to Flan-T5-small through a grounded question-answering prompt.

Deterministic greedy decoding was used for generation.

#### Experimental Results

| K   | Input Tokens | Generated Answer        | Generation Time (s) |
| --- | -----------: | ----------------------- | ------------------: |
| 1   |          119 | middle-distance runner  |              0.1762 |
| 3   |          279 | Moldovan chess player   |              0.0762 |
| 5   |          438 | a Moldovan chess player |              0.0832 |

#### Interpretation

At K=1, the model generated a factually supported answer identifying Diana Richburg as a middle-distance runner.

At K=3 and K=5, the model incorrectly identified her as a Moldovan chess player.

The additional retrieved evidence included a passage about Diana Baciu, who is a different person.

The results are consistent with answer contamination caused by irrelevant retrieved context.

Although the relevant Diana Richburg passage remained ranked first, the concatenation approach did not explicitly communicate retrieval relevance scores to the generator.

#### Conclusion

For the evaluated question, increasing retrieval size from K=1 to K=3 or K=5 reduced answer correctness.

This experiment demonstrates that adding retrieved passages can introduce misleading context and negatively affect grounded generation.

The observation motivates comparison with alternative evidence-fusion strategies.

#### Limitations

The experiment used only one question and deterministic generation.

The findings cannot establish that K=1 is universally optimal.

A larger evaluation across the fixed 20-question dataset is required before drawing broader conclusions.


In [63]:
# ============================================================
# Stage 6.3.2 — Generate-from-Evidence Fusion Prototype
# ============================================================

print("=== GENERATE-FROM-EVIDENCE FUSION ===")

QUESTION = "Who is Diana Richburg?"
TOP_K = 3
MAX_PASSAGE_TOKENS = 80


# ------------------------------------------------------------
# Step 1: Retrieve Wikipedia passages
# ------------------------------------------------------------
retrieved_passages = retriever.retrieve(
    question=QUESTION,
    top_k=TOP_K
)

# ------------------------------------------------------------
# Step 2: Prepare evidence using Prefix-80
# ------------------------------------------------------------

prepared_passages = []

for passage in retrieved_passages:

    prepared = passage.copy()

    prepared["text"] = shorten_passage(
        passage["text"],
        tokenizer,
        max_tokens=MAX_PASSAGE_TOKENS
    )

    prepared_passages.append(prepared)


# ------------------------------------------------------------
# Step 3: Generate one candidate answer per passage
# ------------------------------------------------------------

candidate_answers = []

total_generation_time = 0.0
total_input_tokens = 0
total_output_tokens = 0

for passage in prepared_passages:

    result = generator.generate(
        question=QUESTION,
        passages=[passage]
    )

    candidate = {
        "rank": passage["rank"],
        "title": passage["title"],
        "chunk_id": passage["chunk_id"],
        "similarity": passage["score"],
        "answer": result["answer"]
    }

    candidate_answers.append(candidate)

    total_generation_time += result[
        "generation_time_seconds"
    ]

    total_input_tokens += result["input_tokens"]
    total_output_tokens += result["generated_tokens"]

print("\n=== INDIVIDUAL EVIDENCE ANSWERS ===")

for candidate in candidate_answers:

    print(f"\nRank {candidate['rank']}")
    print("Source:", candidate["title"])
    print("Similarity:", round(candidate["similarity"], 4))
    print("Candidate answer:", candidate["answer"])


# ------------------------------------------------------------
# Step 4: Prepare candidate answers for final synthesis
# ------------------------------------------------------------

candidate_text = "\n".join(
    f"Source rank {candidate['rank']} "
    f"({candidate['title']}): {candidate['answer']}"
    for candidate in candidate_answers
)

# The existing generator accepts passages containing a text field.
fusion_evidence = [
    {"text": candidate_text}
]

# ------------------------------------------------------------
# Step 5: Generate final synthesized answer
# -----------------------------------------------------------

fusion_result = generator.generate(
    question=QUESTION,
    passages=fusion_evidence
)

total_generation_time += fusion_result[
    "generation_time_seconds"
]

total_input_tokens += fusion_result["input_tokens"]
total_output_tokens += fusion_result["generated_tokens"]

print("\n=== FINAL FUSED ANSWER ===")

print("Question:", QUESTION)
print("Final answer:", fusion_result["answer"])

print("\n=== FUSION METRICS ===")

print("Retrieved passages:", TOP_K)
print("Individual generation calls:", TOP_K)
print("Final synthesis calls:", 1)
print("Total model calls:", TOP_K + 1)

print("Total input tokens processed:", total_input_tokens)
print("Total generated tokens:", total_output_tokens)

print(
    "Combined generation time:",
    f"{total_generation_time:.4f} seconds"
)

print(
    "Final synthesis input tokens:",
    fusion_result["input_tokens"]
)

print("\nGenerate-from-evidence prototype completed.")

=== GENERATE-FROM-EVIDENCE FUSION ===

=== INDIVIDUAL EVIDENCE ANSWERS ===

Rank 1
Source: Diana Richburg
Similarity: 0.3932
Candidate answer: middle-distance runner

Rank 2
Source: Diana Richburg
Similarity: 0.2261
Candidate answer: Insufficient evidence

Rank 3
Source: Diana Baciu
Similarity: 0.1121
Candidate answer: Moldovan chess player

=== FINAL FUSED ANSWER ===
Question: Who is Diana Richburg?
Final answer: Insufficient evidence

=== FUSION METRICS ===
Retrieved passages: 3
Individual generation calls: 3
Final synthesis calls: 1
Total model calls: 4
Total input tokens processed: 442
Total generated tokens: 27
Combined generation time: 0.3137 seconds
Final synthesis input tokens: 85

Generate-from-evidence prototype completed.


### Stage 6.3.2 — Generate-from-Evidence Fusion

#### Objective

To investigate whether generating candidate answers independently from retrieved passages can improve answer quality compared with concatenation-based evidence fusion.

#### Methodology

Three Wikipedia passages were retrieved using TF-IDF and cosine similarity.

Each passage was shortened to a maximum of 80 tokens.

Flan-T5-small independently generated one candidate answer from each passage.

The resulting candidate answers were then combined into a single input for a final synthesis step.

The experiment used deterministic greedy decoding.

#### Mathematical Formulation

For retrieved passages $p_1,\ldots,p_K$:

$$
a_i = G(q,p_i)
$$

where:

- $q$ is the input question.
- $p_i$ is retrieved passage $i$.
- $G$ is Flan-T5-small.
- $a_i$ is the candidate answer generated from passage $i$.

The final answer is generated as:

$$
\hat{a}=G(q,a_1\oplus a_2\oplus\cdots\oplus a_K)
$$

where $\oplus$ denotes concatenation.

#### Experimental Results

| Rank | Retrieved Article | Similarity | Candidate Answer       |
| ---- | ----------------- | ---------: | ---------------------- |
| 1    | Diana Richburg    |     0.3932 | middle-distance runner |
| 2    | Diana Richburg    |     0.2261 | Insufficient evidence  |
| 3    | Diana Baciu       |     0.1121 | Moldovan chess player  |

Final generated answer:

**Insufficient evidence**

#### Computational Results

| Metric                       | Observed Value |
| ---------------------------- | -------------: |
| Retrieved passages           |              3 |
| Individual generation calls  |              3 |
| Final synthesis calls        |              1 |
| Total model calls            |              4 |
| Total input tokens processed |            442 |
| Total generated tokens       |             27 |
| Combined generation time     | 0.3137 seconds |
| Final synthesis input tokens |             85 |

#### Interpretation

The first retrieved passage generated a factually supported answer.

However, the final synthesis step failed to preserve that correct candidate and instead returned "Insufficient evidence".

This indicates that independently generating candidate answers does not guarantee successful answer fusion.

The experiment also demonstrated additional computational requirements compared with single-pass concatenation.

#### Comparison with Concatenation

For K=3, concatenation produced an incorrect answer identifying Diana Richburg as a Moldovan chess player.

Generate-from-evidence produced an abstention response despite the availability of a correct intermediate candidate.

Neither method produced the correct final answer in this initial experiment.

#### Limitations

1. Only one question was evaluated.
2. The final synthesis prompt was not specifically designed to resolve conflicting candidate answers.
3. Candidate source ranks were supplied, but their correct interpretation was not guaranteed.
4. The final synthesis did not receive the original full evidence passages.
5. Timing results were obtained from individual runs.

#### Conclusion

Generate-from-evidence fusion successfully produced candidate answers from individual passages but failed during final synthesis.

The results demonstrate that evidence fusion is a separate source of potential errors in RAG systems.

Further evaluation and refinement of the synthesis strategy are required before comparing the methods across the complete evaluation dataset.


In [65]:
# ============================================================
# Stage 6.3.3 — Generate-from-Evidence: K = 1, 3, 5
# ============================================================

import pandas as pd

print("=== GENERATE-FROM-EVIDENCE RETRIEVAL-SIZE EXPERIMENT ===")

QUESTION = "Who is Diana Richburg?"

K_VALUES = [1, 3, 5]
MAX_PASSAGE_TOKENS = 80

generate_from_evidence_results = []

for k in K_VALUES:

    print(f"\n{'=' * 65}")
    print(f"GENERATE-FROM-EVIDENCE — K={k}")
    print(f"{'=' * 65}")

    # --------------------------------------------------------
    # Step 1: Retrieve K passages
    # --------------------------------------------------------

    retrieved_passages = retriever.retrieve(
        question=QUESTION,
        top_k=k
    )

    # --------------------------------------------------------
    # Step 2: Apply Prefix-80 evidence preparation
    # --------------------------------------------------------

    prepared_passages = []

    for passage in retrieved_passages:

        prepared = passage.copy()

        prepared["text"] = shorten_passage(
            passage["text"],
            tokenizer,
            max_tokens=MAX_PASSAGE_TOKENS
        )

        prepared_passages.append(prepared)

    # --------------------------------------------------------
    # Step 3: Generate candidate answers independently
    # --------------------------------------------------------

    candidates = []

    total_generation_time = 0.0
    total_input_tokens = 0
    total_generated_tokens = 0

    for passage in prepared_passages:

        candidate_result = generator.generate(
            question=QUESTION,
            passages=[passage]
        )

        candidates.append({
            "rank": passage["rank"],
            "title": passage["title"],
            "answer": candidate_result["answer"]
        })

        total_generation_time += candidate_result[
            "generation_time_seconds"
        ]

        total_input_tokens += candidate_result[
            "input_tokens"
        ]

        total_generated_tokens += candidate_result[
            "generated_tokens"
        ]

    # --------------------------------------------------------
    # Step 4: Combine candidate answers
    # --------------------------------------------------------

    candidate_text = "\n".join(
        f"Source rank {candidate['rank']} "
        f"({candidate['title']}): {candidate['answer']}"
        for candidate in candidates
    )

    # --------------------------------------------------------
    # Step 5: Final synthesis
    # --------------------------------------------------------

    fusion_result = generator.generate(
        question=QUESTION,
        passages=[{"text": candidate_text}]
    )

    total_generation_time += fusion_result[
        "generation_time_seconds"
    ]

    total_input_tokens += fusion_result[
        "input_tokens"
    ]

    total_generated_tokens += fusion_result[
        "generated_tokens"
    ]

    # --------------------------------------------------------
    # Step 6: Record results
    # --------------------------------------------------------

    generate_from_evidence_results.append({
        "method": "generate_from_evidence",
        "k": k,
        "question": QUESTION,
        "answer": fusion_result["answer"],
        "model_calls": k + 1,
        "total_input_tokens": total_input_tokens,
        "total_generated_tokens": total_generated_tokens,
        "generation_time_seconds": total_generation_time
    })

    print("\nIndividual candidate answers:")

    for candidate in candidates:
        print(
            f"Rank {candidate['rank']} "
            f"({candidate['title']}): "
            f"{candidate['answer']}"
        )

    print("\nFinal answer:", fusion_result["answer"])
    print("Total model calls:", k + 1)
    print("Total input tokens:", total_input_tokens)
    print("Total generated tokens:", total_generated_tokens)

    print(
        "Combined generation time:",
        f"{total_generation_time:.4f} seconds"
    )

# ------------------------------------------------------------
# Step 7: Display comparison
# ------------------------------------------------------------

generate_from_evidence_df = pd.DataFrame(
    generate_from_evidence_results
)

print("\n=== GENERATE-FROM-EVIDENCE RESULTS SUMMARY ===")

display(generate_from_evidence_df)

print("\nGenerate-from-evidence ablation completed.")

=== GENERATE-FROM-EVIDENCE RETRIEVAL-SIZE EXPERIMENT ===

GENERATE-FROM-EVIDENCE — K=1

Individual candidate answers:
Rank 1 (Diana Richburg): middle-distance runner

Final answer: middle-distance runner
Total model calls: 2
Total input tokens: 175
Total generated tokens: 18
Combined generation time: 0.2585 seconds

GENERATE-FROM-EVIDENCE — K=3

Individual candidate answers:
Rank 1 (Diana Richburg): middle-distance runner
Rank 2 (Diana Richburg): Insufficient evidence
Rank 3 (Diana Baciu): Moldovan chess player

Final answer: Insufficient evidence
Total model calls: 4
Total input tokens: 442
Total generated tokens: 27
Combined generation time: 0.2480 seconds

GENERATE-FROM-EVIDENCE — K=5

Individual candidate answers:
Rank 1 (Diana Richburg): middle-distance runner
Rank 2 (Diana Richburg): Insufficient evidence
Rank 3 (Diana Baciu): Moldovan chess player
Rank 4 (Natasha Lako): Insufficient evidence
Rank 5 (Maurice Hurley): Insufficient evidence

Final answer: middle-distance runner
Tot

,method,k,question,answer,model_calls,total_input_tokens,total_generated_tokens,generation_time_seconds
0,generate_from_evidence,1,Who is Diana Richburg?,middle-distance runner,2,175,18,0.258495
1,generate_from_evidence,3,Who is Diana Richburg?,Insufficient evidence,4,442,27,0.247997
2,generate_from_evidence,5,Who is Diana Richburg?,middle-distance runner,6,708,41,0.390366



Generate-from-evidence ablation completed.


## Stage 6.3.3 — Retrieval-Size Ablation and Evidence-Fusion Comparison

### 1. Objective

To investigate how retrieval size and evidence-fusion strategy affect the factual correctness, groundedness and computational requirements of the offline Wikipedia RAG system.

Two fusion methods were evaluated:

1. **Concatenation:** Combine retrieved evidence passages before generating an answer.
2. **Generate-from-Evidence:** Generate candidate answers independently from retrieved passages and subsequently synthesize a final answer.

The experiment considered three retrieval sizes:

$$
K \in \{1,3,5\}
$$

### 2. Experimental Methodology

The following experimental settings were maintained:

| Parameter                           | Configuration                          |
| ----------------------------------- | -------------------------------------- |
| Dataset                             | Local Wikipedia slice                  |
| Indexed passages                    | 8,191                                  |
| Retriever                           | TF-IDF with cosine similarity          |
| Generator                           | google/flan-t5-small                   |
| Execution device                    | NVIDIA GPU                             |
| Retrieval sizes                     | 1, 3, 5                                |
| Maximum evidence tokens per passage | 80                                     |
| Maximum complete input tokens       | 512                                    |
| Maximum generated tokens            | 64                                     |
| Decoding                            | Greedy, deterministic                  |
| Evaluation question                 | Who is Diana Richburg?                 |
| Reference answer                    | American former middle-distance runner |

Both fusion strategies used the same retrieved passages and Prefix-80 evidence-preparation policy.

### 3. Fusion Methods

#### 3.1 Concatenation-Based Fusion

The retrieved passages are combined into a single evidence block:

$$
E_K = p_1 \oplus p_2 \oplus \cdots \oplus p_K
$$

The generator produces:

$$
\hat{a}_{concat}=G(q,E_K)
$$

where:

- $q$ represents the user question.
- $p_i$ represents retrieved passage $i$.
- $K$ represents the number of retrieved passages.
- $\oplus$ denotes concatenation.
- $G$ represents Flan-T5-small.
- $\hat{a}_{concat}$ represents the final generated answer.

This approach requires one generation call per question.

#### 3.2 Generate-from-Evidence Fusion

A candidate answer is generated independently from each retrieved passage:

$$
a_i=G(q,p_i)
$$

The candidate answers are then combined for final synthesis:

$$
\hat{a}_{fusion}
=
G(q,a_1\oplus a_2\oplus\cdots\oplus a_K)
$$

The final synthesis input also includes the source titles and retrieval ranks associated with the candidate answers.

This approach requires:

$$
N_{\text{calls}}=K+1
$$

generation calls per question.

### 4. Experimental Results

#### 4.1 Generated Answers

| K   | Concatenation           | Generate-from-Evidence |
| --- | ----------------------- | ---------------------- |
| 1   | middle-distance runner  | middle-distance runner |
| 3   | Moldovan chess player   | Insufficient evidence  |
| 5   | a Moldovan chess player | middle-distance runner |

For this particular question:

- At K=1, both methods generated factually correct answers.
- At K=3, both methods failed.
- At K=5, generate-from-evidence generated the correct answer, while concatenation generated an incorrect answer.

#### 4.2 Generate-from-Evidence Candidate Answers

For K=3, the independently generated candidate answers were:

| Rank | Wikipedia Article | Candidate Answer       |
| ---- | ----------------- | ---------------------- |
| 1    | Diana Richburg    | middle-distance runner |
| 2    | Diana Richburg    | Insufficient evidence  |
| 3    | Diana Baciu       | Moldovan chess player  |

Despite the presence of a correct candidate answer, the final synthesis returned:

**Insufficient evidence**

For K=5, two additional candidates were produced:

| Rank | Wikipedia Article | Candidate Answer      |
| ---- | ----------------- | --------------------- |
| 4    | Natasha Lako      | Insufficient evidence |
| 5    | Maurice Hurley    | Insufficient evidence |

The final synthesis at K=5 produced:

**middle-distance runner**

These results demonstrate that the synthesis stage may produce different outcomes depending on the number and content of candidate answers.

### 5. Computational Comparison

#### 5.1 Generation Latency

| K   | Concatenation (seconds) | Generate-from-Evidence (seconds) |
| --- | ----------------------: | -------------------------------: |
| 1   |                  0.1762 |                           0.2585 |
| 3   |                  0.0762 |                           0.2480 |
| 5   |                  0.0832 |                           0.3904 |

The measurements represent individual experimental runs and are not statistically reliable latency benchmarks.

#### 5.2 Number of Model Calls

| K   | Concatenation | Generate-from-Evidence |
| --- | ------------: | ---------------------: |
| 1   |             1 |                      2 |
| 3   |             1 |                      4 |
| 5   |             1 |                      6 |

Generate-from-evidence incurs higher computational cost because it requires multiple candidate-generation calls followed by a final synthesis call.

#### 5.3 Generate-from-Evidence Token Usage

| K   | Total Input Tokens Processed | Total Generated Tokens |
| --- | ---------------------------: | ---------------------: |
| 1   |                          175 |                     18 |
| 3   |                          442 |                     27 |
| 5   |                          708 |                     41 |

The total input-token count includes tokens processed across all generation calls rather than the length of a single input prompt.

### 6. Interpretation

#### 6.1 Effect of Retrieval Size

Increasing retrieval size did not consistently improve answer correctness.

At K=3, additional evidence introduced information about Diana Baciu, a different individual.

The concatenation method generated an incorrect answer associated with this irrelevant passage.

At K=5, the generate-from-evidence method recovered the correct answer, but the concatenation method continued to generate an incorrect response.

These observations indicate that retrieval size can affect answer quality in non-monotonic ways.

#### 6.2 Evidence Contamination

The incorrect concatenation outputs are consistent with answer contamination caused by irrelevant retrieved passages.

Although the correct Diana Richburg passage remained ranked first, the generator was exposed to evidence about another person.

The current concatenation prompt did not explicitly communicate numerical retrieval similarity scores to the generator.

#### 6.3 Fusion-Stage Errors

Generate-from-evidence successfully produced a correct candidate answer at K=3.

However, the final synthesis failed to preserve it.

This demonstrates that errors can occur during answer fusion even when individual evidence-conditioned generations are correct.

#### 6.4 Computational Trade-off

Concatenation requires fewer model calls and was faster in the observed runs.

Generate-from-evidence requires additional computation but produced a correct answer at K=5 in this experiment.

Consequently, fusion strategies should be compared using both answer-quality and computational metrics.

### 7. Limitations

1. The experiment used only one evaluation question.
2. Correctness and groundedness were assessed manually.
3. The final synthesis prompt was not separately optimized for conflicting candidate answers.
4. The experiment used a fixed 80-token prefix from each passage, potentially removing relevant information.
5. Latency measurements were obtained from single runs.
6. The results do not establish that either fusion strategy is generally superior.
7. The effects of fusion strategy and retrieval size require systematic evaluation across the fixed 20-question dataset.

### 8. Conclusion

The retrieval-size and fusion-method experiments demonstrated that increasing the number of retrieved passages does not necessarily improve generated-answer correctness.

Concatenation produced a correct answer at K=1 but incorrect answers at K=3 and K=5.

Generate-from-evidence produced correct answers at K=1 and K=5 but returned an incorrect abstention at K=3.

The experiments also revealed a computational trade-off: generate-from-evidence requires additional model calls compared with concatenation.

Overall, these preliminary results demonstrate the importance of evidence quality, fusion behaviour and computational efficiency in RAG systems.

A systematic comparison across the 20-question evaluation dataset is necessary before selecting a preferred fusion strategy.


In [71]:
# ============================================================
# Stage 6.4.2 — Verify Evidence Module Import
# ============================================================

import importlib
import rag.evidence as evidence_module

importlib.invalidate_caches()
evidence_module = importlib.reload(evidence_module)

from rag.evidence import shorten_passage, prepare_evidence

print("=== EVIDENCE MODULE VERIFICATION ===")

print("shorten_passage available:", callable(shorten_passage))
print("prepare_evidence available:", callable(prepare_evidence))

sample_text = (
    "Diana Richburg was an American middle-distance runner "
    "who competed in the 1984 Los Angeles Olympics."
)

shortened = shorten_passage(
    sample_text,
    tokenizer,
    max_tokens=10
)

print("\nOriginal text:", sample_text)
print("Shortened text:", shortened)

print(
    "Shortened token count:",
    len(
        tokenizer(
            shortened,
            add_special_tokens=False
        )["input_ids"]
    )
)

assert callable(shorten_passage)
assert callable(prepare_evidence)

assert len(
    tokenizer(
        shortened,
        add_special_tokens=False
    )["input_ids"]
) <= 10

print("\nEvidence module verification: PASSED")

=== EVIDENCE MODULE VERIFICATION ===
shorten_passage available: True
prepare_evidence available: True

Original text: Diana Richburg was an American middle-distance runner who competed in the 1984 Los Angeles Olympics.
Shortened text: Diana Richburg was an American middle-di
Shortened token count: 10

Evidence module verification: PASSED


In [72]:
# ============================================================
# Stage 6.4.3 — Verify Evidence Preparation Consistency
# ============================================================

from rag.evidence import prepare_evidence

print("=== EVIDENCE PREPARATION CONSISTENCY ===")

QUESTION = "Who is Diana Richburg?"
TOP_K = 3
MAX_EVIDENCE_TOKENS = 80

# Retrieve original passages
retrieved_passages = retriever.retrieve(
    QUESTION,
    top_k=TOP_K
)

# Prepare passages using the reusable module
prepared_passages = prepare_evidence(
    passages=retrieved_passages,
    tokenizer=tokenizer,
    max_tokens=MAX_EVIDENCE_TOKENS
)

print("Question:", QUESTION)
print("Retrieved passages:", len(retrieved_passages))
print("Prepared passages:", len(prepared_passages))

assert len(prepared_passages) == TOP_K

print("\n=== PREPARED EVIDENCE ===")

for rank, (original, prepared) in enumerate(
    zip(retrieved_passages, prepared_passages),
    start=1
):
    original_tokens = len(
        tokenizer(
            original["text"],
            add_special_tokens=False
        )["input_ids"]
    )

    prepared_tokens = len(
        tokenizer(
            prepared["text"],
            add_special_tokens=False
        )["input_ids"]
    )

    print(f"\nRank {rank}")
    print("Article:", prepared["title"])
    print("Original tokens:", original_tokens)
    print("Prepared tokens:", prepared_tokens)
    print("Prepared text:", prepared["text"][:180])

    assert prepared_tokens <= MAX_EVIDENCE_TOKENS

    # Verify retrieval metadata is preserved
    assert prepared["chunk_id"] == original["chunk_id"]
    assert prepared["title"] == original["title"]
    assert prepared["score"] == original["score"]

    # Verify the original passage has not been modified
    assert prepared is not original

print("\nEvidence token limits: Passed")
print("Retrieval metadata preservation: Passed")
print("Original passage protection: Passed")

print("\nAll evidence preparation consistency checks passed.")

=== EVIDENCE PREPARATION CONSISTENCY ===
Question: Who is Diana Richburg?
Retrieved passages: 3
Prepared passages: 3

=== PREPARED EVIDENCE ===

Rank 1
Article: Diana Richburg
Original tokens: 132
Prepared tokens: 80
Prepared text: Diana Lynn Richburg (born July 2, 1963) is an American former middle-distance runner. She competed at the 1984 Los Angeles Olympics, and won the 1985 US national title at 1500 metr

Rank 2
Article: Diana Richburg
Original tokens: 139
Prepared tokens: 80
Prepared text: the 1985 USA Outdoor Track and Field Championships in 4:04.73. Richburg achieved her lifetime best for 1500m when finishing seventh in the 1987 World Championship final, running 4:

Rank 3
Article: Diana Baciu
Original tokens: 144
Prepared tokens: 80
Prepared text: Diana Baciu (born 26 March 1994) is a Moldovan chess player. She received the FIDE title of Woman International Master (WIM) in 2013. She won the Moldovan Women's Chess Championshi

Evidence token limits: Passed
Retrieval metadata pr

In [73]:
# ============================================================
# Stage 6.4.5 — Fusion Module Compatibility Check
# ============================================================

import importlib
import inspect

import rag.fusion as fusion_module

importlib.invalidate_caches()
fusion_module = importlib.reload(fusion_module)

from rag.fusion import (
    concatenate_evidence,
    generate_from_evidence
)

print("=== FUSION MODULE COMPATIBILITY CHECK ===")

print("\n=== FUNCTION AVAILABILITY ===")
print(
    "concatenate_evidence:",
    callable(concatenate_evidence)
)
print(
    "generate_from_evidence:",
    callable(generate_from_evidence)
)

print("\n=== GENERATOR INFORMATION ===")
print("Generator class:", type(generator).__name__)
print("Generator device:", next(generator.model.parameters()).device)

print("\n=== GENERATE METHOD SIGNATURE ===")
print(inspect.signature(generator.generate))

print("\n=== GENERATE METHOD SOURCE ===")
print(inspect.getsource(type(generator).generate))

print("\n=== CONCATENATION CHECK ===")

question = "Who is Diana Richburg?"

retrieved = retriever.retrieve(
    question,
    top_k=3
)

prepared, combined = concatenate_evidence(
    passages=retrieved,
    tokenizer=tokenizer,
    max_tokens=80
)

print("Retrieved passages:", len(retrieved))
print("Prepared passages:", len(prepared))
print("Combined evidence entries:", len(combined))

assert len(prepared) == 3
assert len(combined) == 1

print("\nConcatenation preparation: Passed")
print("\nFusion module compatibility inspection completed.")

=== FUSION MODULE COMPATIBILITY CHECK ===

=== FUNCTION AVAILABILITY ===
concatenate_evidence: True
generate_from_evidence: True

=== GENERATOR INFORMATION ===
Generator class: FlanT5Generator
Generator device: cuda:0

=== GENERATE METHOD SIGNATURE ===
(question, passages)

=== GENERATE METHOD SOURCE ===
    def generate(self, question, passages):
        """
        Generate an answer using retrieved evidence.
        """

        # Step 1: Construct prompt
        prompt = self.build_prompt(
            question,
            passages
        )

        # Step 2: Tokenize WITHOUT truncation
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=False
        )

        input_token_count = inputs["input_ids"].shape[1]

        # Step 3: Enforce input token budget
        if input_token_count > self.max_input_tokens:
            raise ValueError(
                f"Prompt requires {input_token_count} tokens, "
                f"but ma

In [74]:
# ============================================================
# Stage 6.4.6 — Execute Reusable Fusion Methods
# ============================================================

from rag.fusion import (
    concatenate_evidence,
    generate_from_evidence
)

print("=== REUSABLE FUSION VERIFICATION ===")

QUESTION = "Who is Diana Richburg?"
TOP_K = 3
MAX_EVIDENCE_TOKENS = 80

retrieved_passages = retriever.retrieve(
    QUESTION,
    top_k=TOP_K
)

# ------------------------------------------------------------
# Method 1 — Concatenation Fusion
# ------------------------------------------------------------

print("\n=== METHOD 1: CONCATENATION FUSION ===")

prepared, combined = concatenate_evidence(
    passages=retrieved_passages,
    tokenizer=tokenizer,
    max_tokens=MAX_EVIDENCE_TOKENS
)

concat_result = generator.generate(
    question=QUESTION,
    passages=combined
)

print("Generated answer:", concat_result["answer"])
print("Input tokens:", concat_result["input_tokens"])
print("Generated tokens:", concat_result["generated_tokens"])
print(
    "Generation time:",
    round(concat_result["generation_time_seconds"], 4),
    "seconds"
)

# ------------------------------------------------------------
# Method 2 — Generate-from-Evidence Fusion
# ------------------------------------------------------------

print("\n=== METHOD 2: GENERATE-FROM-EVIDENCE ===")

fusion_result = generate_from_evidence(
    question=QUESTION,
    passages=retrieved_passages,
    generator=generator,
    tokenizer=tokenizer,
    max_tokens=MAX_EVIDENCE_TOKENS
)

print("\nIndividual candidate answers:")

for candidate in fusion_result["candidates"]:
    print(
        f"Rank {candidate['rank']} "
        f"({candidate['title']}): "
        f"{candidate['answer']}"
    )

print("\nFinal answer:", fusion_result["answer"])

print("\n=== FUSION METRICS ===")
print("Model calls:", fusion_result["model_calls"])
print("Total input tokens:", fusion_result["total_input_tokens"])
print("Total generated tokens:", fusion_result["total_generated_tokens"])
print(
    "Combined generation time:",
    round(fusion_result["generation_time_seconds"], 4),
    "seconds"
)

assert fusion_result["model_calls"] == TOP_K + 1

print("\nReusable fusion execution verification passed.")

=== REUSABLE FUSION VERIFICATION ===

=== METHOD 1: CONCATENATION FUSION ===
Generated answer: Moldovan chess player
Input tokens: 279
Generated tokens: 8
Generation time: 0.1704 seconds

=== METHOD 2: GENERATE-FROM-EVIDENCE ===

Individual candidate answers:
Rank 1 (Diana Richburg): middle-distance runner
Rank 2 (Diana Richburg): Insufficient evidence
Rank 3 (Diana Baciu): Moldovan chess player

Final answer: Moldovan chess player

=== FUSION METRICS ===
Model calls: 4
Total input tokens: 445
Total generated tokens: 30
Combined generation time: 0.2903 seconds

Reusable fusion execution verification passed.


### Stage 6.4.6 — Reusable Fusion Verification

**Objective:** Verify the implementation of two evidence-fusion
strategies using the reusable Python project.

Two strategies were evaluated using K=3 retrieved passages:

1. **Concatenation Fusion:** Combines retrieved passages before
   generating a single answer.
2. **Generate-from-Evidence Fusion:** Generates an individual answer
   from each passage and synthesizes the candidate answers.

#### Results

| Method                 | Final Answer          | Correct | Model Calls |
| ---------------------- | --------------------- | ------- | ----------- |
| Concatenation          | Moldovan chess player | No      | 1           |
| Generate-from-evidence | Moldovan chess player | No      | 4           |

#### Observations

- Both reusable fusion implementations executed successfully.
- The top-ranked passage contained the correct answer.
- An irrelevant retrieved passage introduced information about
  another individual, Diana Baciu.
- Both methods produced incorrect final answers.
- Generate-from-evidence produced the correct individual answer
  from the first passage but failed to preserve it during synthesis.
- The reusable synthesis prompt differed slightly from the earlier
  notebook experiment, resulting in a different final answer.

#### Conclusion

Successful execution does not guarantee factual correctness.

Retrieval quality, irrelevant evidence, and answer synthesis can
all affect RAG performance.

These preliminary findings motivate systematic evaluation across
the complete 20-question dataset.


In [75]:
# ============================================================
# Stage 6.4.7 — Inspect Generate-from-Evidence Synthesis Prompt
# ============================================================

print("=== FINAL SYNTHESIS PROMPT INSPECTION ===")

candidate_text = "\n".join(
    f"Rank {item['rank']} "
    f"(Source: {item['title']}): {item['answer']}"
    for item in fusion_result["candidates"]
)

synthesis_passages = [{"text": candidate_text}]

synthesis_prompt = generator.build_prompt(
    QUESTION,
    synthesis_passages
)

print("\n=== CANDIDATE EVIDENCE ===")
print(candidate_text)

print("\n=== COMPLETE SYNTHESIS PROMPT ===")
print(synthesis_prompt)

print("\n=== TOKEN COUNT ===")
print(
    len(
        tokenizer(
            synthesis_prompt,
            add_special_tokens=True
        )["input_ids"]
    )
)

print("\nSynthesis prompt inspection completed.")

=== FINAL SYNTHESIS PROMPT INSPECTION ===

=== CANDIDATE EVIDENCE ===
Rank 1 (Source: Diana Richburg): middle-distance runner
Rank 2 (Source: Diana Richburg): Insufficient evidence
Rank 3 (Source: Diana Baciu): Moldovan chess player

=== COMPLETE SYNTHESIS PROMPT ===
Answer the question using only the evidence below. If the evidence does not contain the answer, say 'Insufficient evidence'.

Evidence:
Rank 1 (Source: Diana Richburg): middle-distance runner
Rank 2 (Source: Diana Richburg): Insufficient evidence
Rank 3 (Source: Diana Baciu): Moldovan chess player

Question: Who is Diana Richburg?
Answer:

=== TOKEN COUNT ===
88

Synthesis prompt inspection completed.


In [76]:
# ============================================================
# Stage 6.4.8 — Source-Aware Synthesis Prototype
# ============================================================

print("=== SOURCE-AWARE SYNTHESIS EXPERIMENT ===")

QUESTION = "Who is Diana Richburg?"

candidate_text = "\n".join(
    f"Rank {item['rank']} "
    f"(Source: {item['title']}): {item['answer']}"
    for item in fusion_result["candidates"]
)

source_aware_prompt = f"""
You are answering a question using candidate answers from
retrieved Wikipedia passages.

Use only candidate answers from sources relevant to the
person or subject mentioned in the question.

Ignore answers from unrelated subjects.

If no relevant source provides an answer, respond:
Insufficient evidence.

Candidate answers:
{candidate_text}

Question: {QUESTION}
Final answer:
""".strip()

print("\n=== SOURCE-AWARE PROMPT ===")
print(source_aware_prompt)

# Tokenize prompt
inputs = tokenizer(
    source_aware_prompt,
    return_tensors="pt",
    truncation=False
)

input_tokens = inputs["input_ids"].shape[1]

print("\nInput tokens:", input_tokens)

assert input_tokens <= generator.max_input_tokens, \
    "Prompt exceeds model input limit!"

# Move inputs to the existing GPU
inputs = {
    key: value.to(generator.device)
    for key, value in inputs.items()
}

# Generate without loading a new model
import torch

generator.model.eval()

with torch.inference_mode():

    output_ids = generator.model.generate(
        **inputs,
        max_new_tokens=generator.max_new_tokens,
        do_sample=False,
        num_beams=1
    )

source_aware_answer = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("\n=== RESULTS ===")
print("Original synthesis answer:", fusion_result["answer"])
print("Source-aware synthesis answer:", source_aware_answer)

print("\nSource-aware synthesis experiment completed.")

=== SOURCE-AWARE SYNTHESIS EXPERIMENT ===

=== SOURCE-AWARE PROMPT ===
You are answering a question using candidate answers from
retrieved Wikipedia passages.

Use only candidate answers from sources relevant to the
person or subject mentioned in the question.

Ignore answers from unrelated subjects.

If no relevant source provides an answer, respond:
Insufficient evidence.

Candidate answers:
Rank 1 (Source: Diana Richburg): middle-distance runner
Rank 2 (Source: Diana Richburg): Insufficient evidence
Rank 3 (Source: Diana Baciu): Moldovan chess player

Question: Who is Diana Richburg?
Final answer:

Input tokens: 122

=== RESULTS ===
Original synthesis answer: Moldovan chess player
Source-aware synthesis answer: Rank 1

Source-aware synthesis experiment completed.


### Stage 6.4.8 — Source-Aware Synthesis Experiment

**Objective:** Investigate whether explicit instructions about
source relevance improve answer synthesis.

#### Results

| Strategy               | Generated Output      | Correct |
| ---------------------- | --------------------- | ------- |
| Original synthesis     | Moldovan chess player | No      |
| Source-aware synthesis | Rank 1                | No      |

#### Observations

The source-aware prompt returned "Rank 1", which refers to
the candidate containing the correct answer.

However, the model failed to return the actual answer.

This indicates that identifying a relevant candidate does not
necessarily guarantee successful answer generation.

#### Conclusion

Prompt engineering alone did not resolve the synthesis problem
in this experiment.

A possible future improvement is to separate candidate selection
from answer extraction.

The original fusion strategies will remain unchanged for the
main evaluation to preserve experimental comparability.


In [77]:
# ============================================================
# Stage 6.5.1 — Evaluation Configuration and Planning
# ============================================================

from pathlib import Path
import pandas as pd

print("=== RAG SYSTEMATIC EVALUATION CONFIGURATION ===")

# ------------------------------------------------------------
# 1. Define evaluation settings
# ------------------------------------------------------------

EVALUATION_FILE = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "rag_evaluation_20.csv"
)

EVALUATION_K_VALUES = [1, 3, 5]

EVALUATION_METHODS = [
    "concatenation",
    "generate_from_evidence"
]

MAX_EVIDENCE_TOKENS = 80
MAX_INPUT_TOKENS = 512

# ------------------------------------------------------------
# 2. Load evaluation questions
# ------------------------------------------------------------

evaluation_df = pd.read_csv(EVALUATION_FILE)

required_columns = [
    "question_id",
    "question",
    "reference_answer",
    "source_article"
]

assert all(
    column in evaluation_df.columns
    for column in required_columns
), "Missing required evaluation columns!"

assert len(evaluation_df) == 20, \
    "Expected exactly 20 evaluation questions!"

assert evaluation_df["question_id"].is_unique, \
    "Duplicate question IDs detected!"

assert not evaluation_df[required_columns].isna().any().any(), \
    "Missing evaluation values detected!"

print("\n=== DATASET INFORMATION ===")

print("Evaluation questions:", len(evaluation_df))

print(
    "Unique source articles:",
    evaluation_df["source_article"].nunique()
)

print("Retrieval sizes:", EVALUATION_K_VALUES)
print("Fusion methods:", EVALUATION_METHODS)
print("Evidence token limit:", MAX_EVIDENCE_TOKENS)
print("Model input token limit:", MAX_INPUT_TOKENS)

# ------------------------------------------------------------
# 3. Calculate experiment size
# ------------------------------------------------------------

num_questions = len(evaluation_df)

total_evaluations = (
    num_questions
    * len(EVALUATION_K_VALUES)
    * len(EVALUATION_METHODS)
)

concatenation_calls = (
    num_questions * len(EVALUATION_K_VALUES)
)

generate_from_evidence_calls = sum(
    num_questions * (k + 1)
    for k in EVALUATION_K_VALUES
)

total_model_calls = (
    concatenation_calls
    + generate_from_evidence_calls
)

print("\n=== EXPERIMENT EXECUTION PLAN ===")

print("Total evaluations:", total_evaluations)

print(
    "Concatenation model calls:",
    concatenation_calls
)

print(
    "Generate-from-evidence model calls:",
    generate_from_evidence_calls
)

print("Total model calls:", total_model_calls)

# ------------------------------------------------------------
# 4. Verify evaluation configuration
# ------------------------------------------------------------

assert total_evaluations == 120
assert total_model_calls == 300

print("\n=== CONFIGURATION CHECKS ===")

print("Evaluation dataset: Passed")
print("Question uniqueness: Passed")
print("Experiment combinations: Passed")
print("Model call calculations: Passed")

print("\nEvaluation configuration verification passed.")

=== RAG SYSTEMATIC EVALUATION CONFIGURATION ===

=== DATASET INFORMATION ===
Evaluation questions: 20
Unique source articles: 14
Retrieval sizes: [1, 3, 5]
Fusion methods: ['concatenation', 'generate_from_evidence']
Evidence token limit: 80
Model input token limit: 512

=== EXPERIMENT EXECUTION PLAN ===
Total evaluations: 120
Concatenation model calls: 60
Generate-from-evidence model calls: 240
Total model calls: 300

=== CONFIGURATION CHECKS ===
Evaluation dataset: Passed
Question uniqueness: Passed
Experiment combinations: Passed
Model call calculations: Passed

Evaluation configuration verification passed.


In [81]:
# ============================================================
# Stage 6.5.2 — Define Answer Evaluation Metrics
# ============================================================

import re
import string
from collections import Counter


def normalize_answer(text):
    """
    Normalize answers for Exact Match and Token F1.

    1. Lowercase
    2. Replace punctuation with spaces
    3. Remove English articles
    4. Normalize whitespace
    """

    text = str(text).lower()

    # Replace punctuation with spaces instead of deleting it
    text = "".join(
        " " if char in string.punctuation else char
        for char in text
    )

    # Remove articles
    text = re.sub(r"\b(a|an|the)\b", " ", text)

    # Remove extra spaces
    text = " ".join(text.split())

    return text


def exact_match_score(prediction, reference):
    """
    Return 1 if normalized answers match, otherwise 0.
    """

    return int(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1_score(prediction, reference):
    """
    Calculate token-level precision, recall and F1.

    Uses word-token overlap with frequency counts.
    """

    predicted_tokens = normalize_answer(prediction).split()
    reference_tokens = normalize_answer(reference).split()

    if not predicted_tokens or not reference_tokens:
        return float(predicted_tokens == reference_tokens)

    predicted_counts = Counter(predicted_tokens)
    reference_counts = Counter(reference_tokens)

    common = sum(
        (predicted_counts & reference_counts).values()
    )

    if common == 0:
        return 0.0

    precision = common / len(predicted_tokens)
    recall = common / len(reference_tokens)

    return 2 * precision * recall / (precision + recall)


print("=== ANSWER EVALUATION METRICS VERIFICATION ===")

test_cases = [
    {
        "prediction": "1922",
        "reference": "1922",
        "expected_em": 1,
        "expected_f1": 1.0
    },
    {
        "prediction": "middle-distance runner",
        "reference": "An American former middle-distance runner",
        "expected_em": 0,
        "expected_f1": 0.75
    },
    {
        "prediction": "Moldovan chess player",
        "reference": "An American former middle-distance runner",
        "expected_em": 0,
        "expected_f1": 0.0
    }
]

for index, case in enumerate(test_cases, start=1):

    em = exact_match_score(
        case["prediction"],
        case["reference"]
    )

    f1 = token_f1_score(
        case["prediction"],
        case["reference"]
    )

    print(f"\nTest {index}")
    print("Prediction:", case["prediction"])
    print("Reference:", case["reference"])
    print("Exact Match:", em)
    print("Token F1:", round(f1, 4))

    assert em == case["expected_em"]
    assert abs(f1 - case["expected_f1"]) < 1e-9

print("\nAll answer metric verification checks passed.")

=== ANSWER EVALUATION METRICS VERIFICATION ===

Test 1
Prediction: 1922
Reference: 1922
Exact Match: 1
Token F1: 1.0

Test 2
Prediction: middle-distance runner
Reference: An American former middle-distance runner
Exact Match: 0
Token F1: 0.75

Test 3
Prediction: Moldovan chess player
Reference: An American former middle-distance runner
Exact Match: 0
Token F1: 0.0

All answer metric verification checks passed.


In [82]:
# ============================================================
# Stage 6.5.3 — Retrieval Evaluation Metrics
# ============================================================

print("=== RETRIEVAL EVALUATION METRICS ===")


def retrieval_hit_at_k(retrieved_passages, expected_article):
    """
    Return 1 if the expected source article appears
    among the retrieved passages.
    """

    return int(
        any(
            passage["title"] == expected_article
            for passage in retrieved_passages
        )
    )


def evidence_support_at_k(retrieved_passages, reference_answer):
    """
    Return 1 if the normalized reference answer appears
    in at least one retrieved passage.

    This is a lexical evidence-support proxy.
    """

    reference = normalize_answer(reference_answer)

    if not reference:
        return 0

    for passage in retrieved_passages:

        passage_text = normalize_answer(
            passage["text"]
        )

        # Require a whole normalized phrase, not a substring
        # inside an unrelated word.
        if f" {reference} " in f" {passage_text} ":
            return 1

    return 0


# ------------------------------------------------------------
# Evaluate selected questions
# ------------------------------------------------------------

TEST_QUESTIONS = ["Q001", "Q014"]
K_VALUES = [1, 3, 5]

results = []

for question_id in TEST_QUESTIONS:

    row = evaluation_df.loc[
        evaluation_df["question_id"] == question_id
    ].iloc[0]

    question = row["question"]
    reference = row["reference_answer"]
    expected_article = row["source_article"]

    print("\n" + "=" * 65)
    print("Question ID:", question_id)
    print("Question:", question)
    print("Expected article:", expected_article)
    print("Reference answer:", reference)

    for k in K_VALUES:

        retrieved = retriever.retrieve(
            question,
            top_k=k
        )

        hit = retrieval_hit_at_k(
            retrieved,
            expected_article
        )

        support = evidence_support_at_k(
            retrieved,
            reference
        )

        results.append({
            "question_id": question_id,
            "k": k,
            "retrieval_hit": hit,
            "evidence_support": support
        })

        print(
            f"K={k} | "
            f"Hit@K={hit} | "
            f"Evidence Support@K={support}"
        )


# ------------------------------------------------------------
# Display comparison
# ------------------------------------------------------------

retrieval_metrics_df = pd.DataFrame(results)

print("\n=== RETRIEVAL METRICS SUMMARY ===")

display(retrieval_metrics_df)

assert len(retrieval_metrics_df) == 6

assert retrieval_metrics_df["retrieval_hit"].isin([0, 1]).all()
assert retrieval_metrics_df["evidence_support"].isin([0, 1]).all()

print("\nRetrieval metrics verification passed.")

=== RETRIEVAL EVALUATION METRICS ===

Question ID: Q001
Question: Who is Diana Richburg?
Expected article: Diana Richburg
Reference answer: An American former middle-distance runner
K=1 | Hit@K=1 | Evidence Support@K=1
K=3 | Hit@K=1 | Evidence Support@K=1
K=5 | Hit@K=1 | Evidence Support@K=1

Question ID: Q014
Question: Who portrays Kate Beckett in the television series Castle?
Expected article: Kate Beckett
Reference answer: Stana Katic
K=1 | Hit@K=1 | Evidence Support@K=0
K=3 | Hit@K=1 | Evidence Support@K=0
K=5 | Hit@K=1 | Evidence Support@K=0

=== RETRIEVAL METRICS SUMMARY ===


,question_id,k,retrieval_hit,evidence_support
0,Q001,1,1,1
1,Q001,3,1,1
2,Q001,5,1,1
3,Q014,1,1,0
4,Q014,3,1,0
5,Q014,5,1,0



Retrieval metrics verification passed.


### Stage 6.5.3 — Retrieval Evaluation Metrics

**Objective:** Evaluate whether TF-IDF retrieves the correct source
article and whether the retrieved passages contain the reference answer.

Two metrics were implemented:

1. **Retrieval Hit@K:** Measures whether the expected Wikipedia article
   appears among the top K retrieved passages.

2. **Evidence Support@K:** Checks whether at least one retrieved passage
   contains the normalized reference answer.

#### Experimental Results

| Question              | K       | Hit@K | Evidence Support@K |
| --------------------- | ------- | ----- | ------------------ |
| Q001 — Diana Richburg | 1, 3, 5 | 1     | 1                  |
| Q014 — Kate Beckett   | 1, 3, 5 | 1     | 0                  |

#### Observations

- The expected article was retrieved for both questions.
- Q001 successfully retrieved evidence containing the reference answer.
- Q014 retrieved the correct article but failed to retrieve an
  answer-bearing passage within the top five results.
- Increasing K from 1 to 5 did not resolve the Q014 failure.

#### Conclusion

Article-level retrieval success does not guarantee passage-level
evidence availability.

This distinction is important because a RAG generator can produce
an incorrect or unsupported answer even when the correct article
has been retrieved.

Evidence Support@K is implemented as a normalized text-presence
check and does not measure full semantic entailment.


In [83]:
# ============================================================
# Stage 6.5.5 — Verify Reusable Evaluation Module
# ============================================================

import importlib
import numpy as np

import rag.evaluation as evaluation_module

# Reload saved Python changes
importlib.invalidate_caches()
evaluation_module = importlib.reload(evaluation_module)

from rag.evaluation import (
    normalize_answer as modular_normalize_answer,
    exact_match_score as modular_exact_match,
    token_f1_score as modular_token_f1,
    retrieval_hit_at_k as modular_retrieval_hit,
    evidence_support_at_k as modular_evidence_support
)

print("=== REUSABLE EVALUATION MODULE VERIFICATION ===")

# ------------------------------------------------------------
# 1. Answer metric consistency
# ------------------------------------------------------------

test_cases = [
    ("1922", "1922"),
    (
        "middle-distance runner",
        "An American former middle-distance runner"
    ),
    (
        "Moldovan chess player",
        "An American former middle-distance runner"
    )
]

print("\n=== ANSWER METRIC CONSISTENCY ===")

for prediction, reference in test_cases:

    notebook_em = exact_match_score(
        prediction,
        reference
    )

    module_em = modular_exact_match(
        prediction,
        reference
    )

    notebook_f1 = token_f1_score(
        prediction,
        reference
    )

    module_f1 = modular_token_f1(
        prediction,
        reference
    )

    assert notebook_em == module_em

    assert np.isclose(
        notebook_f1,
        module_f1,
        atol=1e-10
    )

    print(
        f"Prediction: {prediction} | "
        f"EM={module_em} | "
        f"F1={module_f1:.4f}"
    )

print("\nAnswer metric consistency: Passed")

# ------------------------------------------------------------
# 2. Retrieval metric consistency
# ------------------------------------------------------------

print("\n=== RETRIEVAL METRIC CONSISTENCY ===")

for question_id in ["Q001", "Q014"]:

    row = evaluation_df.loc[
        evaluation_df["question_id"] == question_id
    ].iloc[0]

    retrieved = retriever.retrieve(
        row["question"],
        top_k=3
    )

    notebook_hit = retrieval_hit_at_k(
        retrieved,
        row["source_article"]
    )

    module_hit = modular_retrieval_hit(
        retrieved,
        row["source_article"]
    )

    notebook_support = evidence_support_at_k(
        retrieved,
        row["reference_answer"]
    )

    module_support = modular_evidence_support(
        retrieved,
        row["reference_answer"]
    )

    assert notebook_hit == module_hit
    assert notebook_support == module_support

    print(
        f"{question_id} | "
        f"Hit@3={module_hit} | "
        f"Evidence Support@3={module_support}"
    )

print("\nRetrieval metric consistency: Passed")

# ------------------------------------------------------------
# 3. Normalization consistency
# ------------------------------------------------------------

sample_text = "An American middle-distance runner!"

assert (
    normalize_answer(sample_text)
    == modular_normalize_answer(sample_text)
)

print("Normalization consistency: Passed")

print("\nAll reusable evaluation module checks passed.")

=== REUSABLE EVALUATION MODULE VERIFICATION ===

=== ANSWER METRIC CONSISTENCY ===
Prediction: 1922 | EM=1 | F1=1.0000
Prediction: middle-distance runner | EM=0 | F1=0.7500
Prediction: Moldovan chess player | EM=0 | F1=0.0000

Answer metric consistency: Passed

=== RETRIEVAL METRIC CONSISTENCY ===
Q001 | Hit@3=1 | Evidence Support@3=1
Q014 | Hit@3=1 | Evidence Support@3=0

Retrieval metric consistency: Passed
Normalization consistency: Passed

All reusable evaluation module checks passed.


In [84]:
# ============================================================
# Stage 6.5.6 — Small-Batch RAG Evaluation
# ============================================================

import pandas as pd
import time

from rag.evidence import prepare_evidence

from rag.fusion import (
    concatenate_evidence,
    generate_from_evidence
)

from rag.evaluation import (
    exact_match_score,
    token_f1_score,
    retrieval_hit_at_k,
    evidence_support_at_k
)

print("=== SMALL-BATCH RAG EVALUATION ===")

# ------------------------------------------------------------
# 1. Experiment configuration
# ------------------------------------------------------------

PILOT_QUESTION_IDS = ["Q001", "Q014"]
PILOT_K_VALUES = [1, 3]

MAX_EVIDENCE_TOKENS = 80

pilot_results = []

# ------------------------------------------------------------
# 2. Evaluate each question
# ------------------------------------------------------------

for question_id in PILOT_QUESTION_IDS:

    row = evaluation_df.loc[
        evaluation_df["question_id"] == question_id
    ].iloc[0]

    question = row["question"]
    reference = row["reference_answer"]
    expected_article = row["source_article"]

    print("\n" + "=" * 70)
    print("Question ID:", question_id)
    print("Question:", question)
    print("Reference:", reference)

    for k in PILOT_K_VALUES:

        # ----------------------------------------------------
        # Retrieve passages
        # ----------------------------------------------------

        retrieval_start = time.perf_counter()

        retrieved = retriever.retrieve(
            question,
            top_k=k
        )

        retrieval_time = (
            time.perf_counter() - retrieval_start
        )

        # ----------------------------------------------------
        # Calculate retrieval metrics
        # ----------------------------------------------------

        hit = retrieval_hit_at_k(
            retrieved,
            expected_article
        )

        original_support = evidence_support_at_k(
            retrieved,
            reference
        )

        prepared = prepare_evidence(
            passages=retrieved,
            tokenizer=tokenizer,
            max_tokens=MAX_EVIDENCE_TOKENS
        )

        prepared_support = evidence_support_at_k(
            prepared,
            reference
        )

        # ----------------------------------------------------
        # Method 1: Concatenation
        # ----------------------------------------------------

        _, combined = concatenate_evidence(
            passages=retrieved,
            tokenizer=tokenizer,
            max_tokens=MAX_EVIDENCE_TOKENS
        )

        concat_result = generator.generate(
            question=question,
            passages=combined
        )

        pilot_results.append({
            "question_id": question_id,
            "question": question,
            "reference_answer": reference,
            "method": "concatenation",
            "k": k,
            "answer": concat_result["answer"],
            "exact_match": exact_match_score(
                concat_result["answer"],
                reference
            ),
            "token_f1": token_f1_score(
                concat_result["answer"],
                reference
            ),
            "retrieval_hit": hit,
            "original_evidence_support": original_support,
            "prepared_evidence_support": prepared_support,
            "retrieval_time_seconds": retrieval_time,
            "generation_time_seconds":
                concat_result["generation_time_seconds"],
            "input_tokens": concat_result["input_tokens"],
            "generated_tokens": concat_result["generated_tokens"],
            "model_calls": 1
        })

        # ----------------------------------------------------
        # Method 2: Generate-from-Evidence
        # ----------------------------------------------------

        fusion_result = generate_from_evidence(
            question=question,
            passages=retrieved,
            generator=generator,
            tokenizer=tokenizer,
            max_tokens=MAX_EVIDENCE_TOKENS
        )

        pilot_results.append({
            "question_id": question_id,
            "question": question,
            "reference_answer": reference,
            "method": "generate_from_evidence",
            "k": k,
            "answer": fusion_result["answer"],
            "exact_match": exact_match_score(
                fusion_result["answer"],
                reference
            ),
            "token_f1": token_f1_score(
                fusion_result["answer"],
                reference
            ),
            "retrieval_hit": hit,
            "original_evidence_support": original_support,
            "prepared_evidence_support": prepared_support,
            "retrieval_time_seconds": retrieval_time,
            "generation_time_seconds":
                fusion_result["generation_time_seconds"],
            "input_tokens": fusion_result["total_input_tokens"],
            "generated_tokens":
                fusion_result["total_generated_tokens"],
            "model_calls": fusion_result["model_calls"]
        })

        print(f"\nK={k}")
        print("Retrieval Hit:", hit)
        print("Original evidence support:", original_support)
        print("Prepared evidence support:", prepared_support)
        print("Concatenation:", concat_result["answer"])
        print("Generate-from-evidence:", fusion_result["answer"])


# ------------------------------------------------------------
# 3. Results table
# ------------------------------------------------------------

pilot_df = pd.DataFrame(pilot_results)

print("\n=== PILOT EVALUATION RESULTS ===")

display(
    pilot_df[
        [
            "question_id",
            "method",
            "k",
            "answer",
            "exact_match",
            "token_f1",
            "retrieval_hit",
            "original_evidence_support",
            "prepared_evidence_support",
            "model_calls"
        ]
    ]
)

# ------------------------------------------------------------
# 4. Verification
# ------------------------------------------------------------

assert len(pilot_df) == 8

assert pilot_df["model_calls"].sum() == 16

assert pilot_df["exact_match"].between(0, 1).all()

assert pilot_df["token_f1"].between(0, 1).all()

print("\n=== VERIFICATION ===")
print("Total evaluations:", len(pilot_df))
print("Total model calls:", pilot_df["model_calls"].sum())

print("\nSmall-batch RAG evaluation completed.")

=== SMALL-BATCH RAG EVALUATION ===

Question ID: Q001
Question: Who is Diana Richburg?
Reference: An American former middle-distance runner

K=1
Retrieval Hit: 1
Original evidence support: 1
Prepared evidence support: 1
Concatenation: middle-distance runner
Generate-from-evidence: ranked 1

K=3
Retrieval Hit: 1
Original evidence support: 1
Prepared evidence support: 1
Concatenation: Moldovan chess player
Generate-from-evidence: Moldovan chess player

Question ID: Q014
Question: Who portrays Kate Beckett in the television series Castle?
Reference: Stana Katic

K=1
Retrieval Hit: 1
Original evidence support: 0
Prepared evidence support: 0
Concatenation: Insufficient evidence
Generate-from-evidence: Insufficient evidence

K=3
Retrieval Hit: 1
Original evidence support: 0
Prepared evidence support: 0
Concatenation: Ryan
Generate-from-evidence: Ryan

=== PILOT EVALUATION RESULTS ===


,question_id,method,k,answer,exact_match,token_f1,retrieval_hit,original_evidence_support,prepared_evidence_support,model_calls
0,Q001,concatenation,1,middle-distance runner,0,0.75,1,1,1,1
1,Q001,generate_from_evidence,1,ranked 1,0,0.00,1,1,1,2
2,Q001,concatenation,3,Moldovan chess player,0,0.00,1,1,1,1
3,Q001,generate_from_evidence,3,Moldovan chess player,0,0.00,1,1,1,4
4,Q014,concatenation,1,Insufficient evidence,0,0.00,1,0,0,1
5,Q014,generate_from_evidence,1,Insufficient evidence,0,0.00,1,0,0,2
6,Q014,concatenation,3,Ryan,0,0.00,1,0,0,1
7,Q014,generate_from_evidence,3,Ryan,0,0.00,1,0,0,4



=== VERIFICATION ===
Total evaluations: 8
Total model calls: 16

Small-batch RAG evaluation completed.


### Stage 6.5.6 — Small-Batch RAG Evaluation

**Objective:** Validate the complete RAG evaluation workflow before
executing the full 20-question experiment.

The pilot evaluated two questions using:

- TF-IDF retrieval
- Flan-T5-small generation
- Concatenation and generate-from-evidence fusion
- K = 1 and K = 3
- An 80-token limit per retrieved passage

A total of eight evaluations and sixteen generation calls were completed.

#### Key Findings

**1. Increasing retrieval size does not guarantee better accuracy.**

For Q001, concatenation produced a correct answer with K=1,
but an incorrect answer with K=3 because unrelated evidence
was included.

**2. Correct evidence does not guarantee correct synthesis.**

Generate-from-evidence returned "ranked 1" for Q001 at K=1,
despite the correct candidate answer being available.

**3. Passage-level evidence availability matters.**

For Q014, the correct Wikipedia article was retrieved,
but the reference answer was absent from the selected passages.

At K=1, both fusion methods appropriately returned
"Insufficient evidence".

At K=3, both generated the unsupported answer "Ryan".

**4. Exact Match does not capture all factual correctness.**

The answer "middle-distance runner" received:

- Exact Match: 0
- Token F1: 0.75

Although this answer was factually correct, it did not exactly
match the longer reference answer.

#### Conclusion

The pilot successfully verified the end-to-end evaluation workflow.

The results demonstrate that retrieval quality, evidence selection,
fusion strategy, and answer generation can independently influence
RAG performance.

The pilot contains only two deliberately selected diagnostic
questions; its results are not representative of overall accuracy.

A systematic 20-question evaluation is required before drawing
broader conclusions.


In [85]:
# ============================================================
# Stage 6.5.7 — Save Small-Batch Evaluation Results
# ============================================================

from pathlib import Path
import json

print("=== PILOT EVALUATION PERSISTENCE ===")

# Results directory
PILOT_RESULTS_DIR = PROJECT_ROOT / "results" / "evaluation"

PILOT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Output paths
PILOT_CSV_PATH = (
    PILOT_RESULTS_DIR / "rag_pilot_results.csv"
)

PILOT_JSON_PATH = (
    PILOT_RESULTS_DIR / "rag_pilot_results.json"
)

# Protect existing results
for path in [PILOT_CSV_PATH, PILOT_JSON_PATH]:
    if path.exists():
        raise FileExistsError(
            f"Existing evaluation artifact will not be overwritten: {path}"
        )

# Save CSV
pilot_df.to_csv(
    PILOT_CSV_PATH,
    index=False
)

# Save JSON
pilot_df.to_json(
    PILOT_JSON_PATH,
    orient="records",
    indent=4,
    force_ascii=False
)

# Reload and verify
loaded_csv = pd.read_csv(PILOT_CSV_PATH)

with open(
    PILOT_JSON_PATH,
    "r",
    encoding="utf-8"
) as f:
    loaded_json = json.load(f)

assert len(loaded_csv) == len(pilot_df)
assert len(loaded_json) == len(pilot_df)

assert loaded_csv["model_calls"].sum() == 16

print("\n=== SAVED FILES ===")
print("CSV:", PILOT_CSV_PATH)
print("JSON:", PILOT_JSON_PATH)

print("\n=== VERIFICATION ===")
print("Original evaluations:", len(pilot_df))
print("Reloaded CSV evaluations:", len(loaded_csv))
print("Reloaded JSON evaluations:", len(loaded_json))

print("\nPilot evaluation persistence: PASSED")

=== PILOT EVALUATION PERSISTENCE ===

=== SAVED FILES ===
CSV: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\results\evaluation\rag_pilot_results.csv
JSON: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\results\evaluation\rag_pilot_results.json

=== VERIFICATION ===
Original evaluations: 8
Reloaded CSV evaluations: 8
Reloaded JSON evaluations: 8

Pilot evaluation persistence: PASSED


In [86]:
# ============================================================
# Stage 6.5.8 — Prepare Full Evaluation Execution Plan
# ============================================================

from pathlib import Path
import pandas as pd

print("=== FULL RAG EVALUATION EXECUTION PLAN ===")

# ------------------------------------------------------------
# 1. Experiment settings
# ------------------------------------------------------------

FULL_K_VALUES = [1, 3, 5]

FULL_METHODS = [
    "concatenation",
    "generate_from_evidence"
]

FULL_MAX_EVIDENCE_TOKENS = 80

FULL_RESULTS_DIR = (
    PROJECT_ROOT / "results" / "evaluation"
)

FULL_RESULTS_PATH = (
    FULL_RESULTS_DIR / "rag_full_evaluation.jsonl"
)

# ------------------------------------------------------------
# 2. Build evaluation plan
# ------------------------------------------------------------

evaluation_plan = []

for _, row in evaluation_df.iterrows():

    for k in FULL_K_VALUES:

        for method in FULL_METHODS:

            evaluation_plan.append({
                "question_id": row["question_id"],
                "question": row["question"],
                "reference_answer": row["reference_answer"],
                "source_article": row["source_article"],
                "k": k,
                "method": method
            })

evaluation_plan_df = pd.DataFrame(evaluation_plan)

# ------------------------------------------------------------
# 3. Verify uniqueness
# ------------------------------------------------------------

experiment_keys = [
    "question_id",
    "k",
    "method"
]

assert len(evaluation_plan_df) == 120

assert not evaluation_plan_df.duplicated(
    subset=experiment_keys
).any()

# ------------------------------------------------------------
# 4. Check existing checkpoints
# ------------------------------------------------------------

if FULL_RESULTS_PATH.exists():

    existing_results = pd.read_json(
        FULL_RESULTS_PATH,
        lines=True
    )

    print(
        "\nExisting checkpoint records:",
        len(existing_results)
    )

else:

    print("\nNo existing full-evaluation checkpoint.")

# ------------------------------------------------------------
# 5. Display execution plan
# ------------------------------------------------------------

print("\n=== CONFIGURATION ===")

print("Questions:", evaluation_df.shape[0])
print("Retrieval sizes:", FULL_K_VALUES)
print("Fusion methods:", FULL_METHODS)
print("Total evaluations:", len(evaluation_plan_df))
print("Evidence tokens per passage:", FULL_MAX_EVIDENCE_TOKENS)

print("\n=== FIRST 12 EXPERIMENTS ===")

display(
    evaluation_plan_df[
        ["question_id", "k", "method"]
    ].head(12)
)

print("\nCheckpoint location:")
print(FULL_RESULTS_PATH)

print("\nFull evaluation execution plan verification passed.")

=== FULL RAG EVALUATION EXECUTION PLAN ===

No existing full-evaluation checkpoint.

=== CONFIGURATION ===
Questions: 20
Retrieval sizes: [1, 3, 5]
Fusion methods: ['concatenation', 'generate_from_evidence']
Total evaluations: 120
Evidence tokens per passage: 80

=== FIRST 12 EXPERIMENTS ===


,question_id,k,method
0,Q001,1,concatenation
1,Q001,1,generate_from_evidence
2,Q001,3,concatenation
3,Q001,3,generate_from_evidence
4,Q001,5,concatenation
5,Q001,5,generate_from_evidence
6,Q002,1,concatenation
7,Q002,1,generate_from_evidence
8,Q002,3,concatenation
9,Q002,3,generate_from_evidence



Checkpoint location:
c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\results\evaluation\rag_full_evaluation.jsonl

Full evaluation execution plan verification passed.


In [95]:
# ============================================================
# Stage 6.5.9 — Checkpointed Full RAG Evaluation Runner
# ============================================================

import json
import os
import time
import pandas as pd

from rag.evidence import prepare_evidence

from rag.fusion import (
    concatenate_evidence,
    generate_from_evidence
)

from rag.evaluation import (
    exact_match_score,
    token_f1_score,
    retrieval_hit_at_k,
    evidence_support_at_k
)

print("=== CHECKPOINTED RAG EVALUATION ===")

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

BATCH_SIZE = 18

FULL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Load existing completed evaluations
# ------------------------------------------------------------

completed_keys = set()

if FULL_RESULTS_PATH.exists():

    with open(
        FULL_RESULTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                continue

            record = json.loads(line)

            key = (
                record["question_id"],
                int(record["k"]),
                record["method"]
            )

            if key in completed_keys:
                raise ValueError(
                    f"Duplicate checkpoint key on line {line_number}: {key}"
                )

            completed_keys.add(key)

print("Previously completed evaluations:", len(completed_keys))

# ------------------------------------------------------------
# 3. Select pending experiments
# ------------------------------------------------------------

pending_experiments = []

for experiment in evaluation_plan:

    key = (
        experiment["question_id"],
        int(experiment["k"]),
        experiment["method"]
    )

    if key not in completed_keys:
        pending_experiments.append(experiment)

batch = pending_experiments[:BATCH_SIZE]

print("Evaluations in current batch:", len(batch))

# ------------------------------------------------------------
# 4. Execute the batch
# ------------------------------------------------------------

for experiment in batch:

    question_id = experiment["question_id"]
    question = experiment["question"]
    reference = experiment["reference_answer"]
    expected_article = experiment["source_article"]

    k = int(experiment["k"])
    method = experiment["method"]

    print("\n" + "=" * 65)

    print("Question:", question_id)
    print("K:", k)
    print("Method:", method)

    # --------------------------------------------------------
    # Retrieve passages
    # --------------------------------------------------------

    retrieval_start = time.perf_counter()

    retrieved = retriever.retrieve(
        question,
        top_k=k
    )

    retrieval_time = (
        time.perf_counter() - retrieval_start
    )

    # --------------------------------------------------------
    # Calculate retrieval metrics
    # --------------------------------------------------------

    hit = retrieval_hit_at_k(
        retrieved,
        expected_article
    )

    original_support = evidence_support_at_k(
        retrieved,
        reference
    )

    prepared = prepare_evidence(
        passages=retrieved,
        tokenizer=tokenizer,
        max_tokens=FULL_MAX_EVIDENCE_TOKENS
    )

    prepared_support = evidence_support_at_k(
        prepared,
        reference
    )

    # --------------------------------------------------------
    # Execute selected fusion method
    # --------------------------------------------------------

    if method == "concatenation":

        _, combined = concatenate_evidence(
            passages=retrieved,
            tokenizer=tokenizer,
            max_tokens=FULL_MAX_EVIDENCE_TOKENS
        )

        result = generator.generate(
            question=question,
            passages=combined
        )

        answer = result["answer"]

        input_tokens = result["input_tokens"]
        generated_tokens = result["generated_tokens"]

        generation_time = result[
            "generation_time_seconds"
        ]

        model_calls = 1

    elif method == "generate_from_evidence":

        result = generate_from_evidence(
            question=question,
            passages=retrieved,
            generator=generator,
            tokenizer=tokenizer,
            max_tokens=FULL_MAX_EVIDENCE_TOKENS
        )

        answer = result["answer"]

        input_tokens = result["total_input_tokens"]
        generated_tokens = result["total_generated_tokens"]

        generation_time = result[
            "generation_time_seconds"
        ]

        model_calls = result["model_calls"]

    else:
        raise ValueError(
            f"Unknown fusion method: {method}"
        )

    # --------------------------------------------------------
    # Evaluate generated answer
    # --------------------------------------------------------

    em = exact_match_score(
        answer,
        reference
    )

    f1 = token_f1_score(
        answer,
        reference
    )

    abstained = (
        " ".join(answer.lower().split())
        == "insufficient evidence"
    )

    # --------------------------------------------------------
    # Create result record
    # --------------------------------------------------------

    record = {
        "question_id": question_id,
        "question": question,
        "reference_answer": reference,
        "source_article": expected_article,
        "method": method,
        "k": k,
        "answer": answer,
        "exact_match": em,
        "token_f1": f1,
        "retrieval_hit": hit,
        "original_evidence_support": original_support,
        "prepared_evidence_support": prepared_support,
        "abstained": abstained,
        "retrieval_time_seconds": retrieval_time,
        "generation_time_seconds": generation_time,
        "input_tokens": input_tokens,
        "generated_tokens": generated_tokens,
        "model_calls": model_calls,
        "max_evidence_tokens": FULL_MAX_EVIDENCE_TOKENS
    }

    # --------------------------------------------------------
    # Save checkpoint immediately after each evaluation
    # --------------------------------------------------------

    with open(
        FULL_RESULTS_PATH,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

        f.flush()
        os.fsync(f.fileno())

    completed_keys.add(
        (question_id, k, method)
    )

    print("Generated answer:", answer)
    print("Exact Match:", em)
    print("Token F1:", round(f1, 4))
    print("Checkpoint saved: Yes")

# ------------------------------------------------------------
# 5. Read current progress
# ------------------------------------------------------------

if FULL_RESULTS_PATH.exists():

    full_results_df = pd.read_json(
        FULL_RESULTS_PATH,
        lines=True
    )

    total_completed = len(full_results_df)

else:

    total_completed = 0

print("\n=== EVALUATION PROGRESS ===")

print("Completed evaluations:", total_completed)
print("Remaining evaluations:", 120 - total_completed)

print(
    "Progress:",
    f"{total_completed / 120 * 100:.2f}%"
)

print("\nCheckpointed evaluation batch completed.")

=== CHECKPOINTED RAG EVALUATION ===
Previously completed evaluations: 102
Evaluations in current batch: 18

Question: Q018
K: 1
Method: concatenation
Generated answer: Crisler Arena
Exact Match: 1
Token F1: 1.0
Checkpoint saved: Yes

Question: Q018
K: 1
Method: generate_from_evidence
Generated answer: Crisler Arena
Exact Match: 1
Token F1: 1.0
Checkpoint saved: Yes

Question: Q018
K: 3
Method: concatenation
Generated answer: Crisler Arena in Ann Arbor, Michigan
Exact Match: 0
Token F1: 0.5
Checkpoint saved: Yes

Question: Q018
K: 3
Method: generate_from_evidence
Generated answer: Crisler Arena
Exact Match: 1
Token F1: 1.0
Checkpoint saved: Yes

Question: Q018
K: 5
Method: concatenation
Generated answer: Crisler Arena in Ann Arbor, Michigan
Exact Match: 0
Token F1: 0.5
Checkpoint saved: Yes

Question: Q018
K: 5
Method: generate_from_evidence
Generated answer: Mediterranean Games
Exact Match: 0
Token F1: 0.0
Checkpoint saved: Yes

Question: Q019
K: 1
Method: concatenation
Generated answe

In [88]:
# ============================================================
# Stage 6.5.10 — Verify Evaluation Checkpoint
# ============================================================

import pandas as pd

print("=== EVALUATION CHECKPOINT VERIFICATION ===")

checkpoint_df = pd.read_json(
    FULL_RESULTS_PATH,
    lines=True
)

print("Saved evaluations:", len(checkpoint_df))

# Check that experiment keys are unique
assert not checkpoint_df.duplicated(
    subset=["question_id", "k", "method"]
).any()

# Verify the first batch
assert len(checkpoint_df) == 6

assert checkpoint_df["question_id"].eq("Q001").all()

assert set(checkpoint_df["k"]) == {1, 3, 5}

assert set(checkpoint_df["method"]) == {
    "concatenation",
    "generate_from_evidence"
}

print("\nCheckpoint records:")
display(
    checkpoint_df[
        [
            "question_id",
            "k",
            "method",
            "answer",
            "exact_match",
            "token_f1"
        ]
    ]
)

print("\nCheckpoint uniqueness: Passed")
print("Checkpoint completeness: Passed")

print("\nAll checkpoint verification checks passed.")

=== EVALUATION CHECKPOINT VERIFICATION ===
Saved evaluations: 6

Checkpoint records:


,question_id,k,method,answer,exact_match,token_f1
0,Q001,1,concatenation,middle-distance runner,0,0.75
1,Q001,1,generate_from_evidence,ranked 1,0,0.00
2,Q001,3,concatenation,Moldovan chess player,0,0.00
3,Q001,3,generate_from_evidence,Moldovan chess player,0,0.00
4,Q001,5,concatenation,a Moldovan chess player,0,0.00
5,Q001,5,generate_from_evidence,Moldovan chess player,0,0.00



Checkpoint uniqueness: Passed
Checkpoint completeness: Passed

All checkpoint verification checks passed.


In [96]:
# ============================================================
# Stage 6.6.1 — Aggregate RAG Evaluation Results
# ============================================================

import pandas as pd

print("=== FINAL RAG EVALUATION ANALYSIS ===")

# ------------------------------------------------------------
# 1. Load saved results
# ------------------------------------------------------------

full_results_df = pd.read_json(
    FULL_RESULTS_PATH,
    lines=True
)

print("\n=== DATASET VERIFICATION ===")

print("Total evaluation records:", len(full_results_df))

print(
    "Unique questions:",
    full_results_df["question_id"].nunique()
)

print(
    "Fusion methods:",
    full_results_df["method"].unique().tolist()
)

print(
    "K values:",
    sorted(full_results_df["k"].unique().tolist())
)

# ------------------------------------------------------------
# 2. Verify all experiments are present
# ------------------------------------------------------------

assert len(full_results_df) == 120

assert full_results_df["question_id"].nunique() == 20

assert not full_results_df.duplicated(
    subset=["question_id", "k", "method"]
).any()

assert set(full_results_df["k"]) == {1, 3, 5}

assert set(full_results_df["method"]) == {
    "concatenation",
    "generate_from_evidence"
}

# Verify each method/K combination contains 20 questions
combination_counts = full_results_df.groupby(
    ["method", "k"]
).size()

assert (combination_counts == 20).all()

print("\nDataset integrity: PASSED")

# ------------------------------------------------------------
# 3. Aggregate evaluation metrics
# ------------------------------------------------------------

evaluation_summary = (
    full_results_df
    .groupby(["method", "k"])
    .agg(
        questions=("question_id", "count"),
        exact_match=("exact_match", "mean"),
        token_f1=("token_f1", "mean"),
        retrieval_hit=("retrieval_hit", "mean"),
        original_evidence_support=(
            "original_evidence_support", "mean"
        ),
        prepared_evidence_support=(
            "prepared_evidence_support", "mean"
        ),
        abstention_rate=("abstained", "mean"),
        mean_retrieval_time=(
            "retrieval_time_seconds", "mean"
        ),
        mean_generation_time=(
            "generation_time_seconds", "mean"
        ),
        mean_input_tokens=("input_tokens", "mean"),
        mean_generated_tokens=("generated_tokens", "mean"),
        total_model_calls=("model_calls", "sum")
    )
    .reset_index()
)

# Convert rates to percentages
percentage_columns = [
    "exact_match",
    "token_f1",
    "retrieval_hit",
    "original_evidence_support",
    "prepared_evidence_support",
    "abstention_rate"
]

for column in percentage_columns:
    evaluation_summary[column + "_pct"] = (
        evaluation_summary[column] * 100
    )

# ------------------------------------------------------------
# 4. Accuracy comparison
# ------------------------------------------------------------

print("\n=== ACCURACY COMPARISON ===")

accuracy_columns = [
    "method",
    "k",
    "questions",
    "exact_match_pct",
    "token_f1_pct",
    "retrieval_hit_pct",
    "original_evidence_support_pct",
    "prepared_evidence_support_pct",
    "abstention_rate_pct"
]

display(
    evaluation_summary[accuracy_columns].round(2)
)

# ------------------------------------------------------------
# 5. Computational cost comparison
# ------------------------------------------------------------

print("\n=== COMPUTATIONAL COST COMPARISON ===")

cost_columns = [
    "method",
    "k",
    "mean_retrieval_time",
    "mean_generation_time",
    "mean_input_tokens",
    "mean_generated_tokens",
    "total_model_calls"
]

display(
    evaluation_summary[cost_columns].round(4)
)

# ------------------------------------------------------------
# 6. Overall method comparison
# ------------------------------------------------------------

print("\n=== OVERALL FUSION METHOD COMPARISON ===")

overall_summary = (
    full_results_df
    .groupby("method")
    .agg(
        evaluations=("question_id", "count"),
        mean_exact_match=("exact_match", "mean"),
        mean_token_f1=("token_f1", "mean"),
        mean_generation_time=(
            "generation_time_seconds", "mean"
        ),
        total_input_tokens=("input_tokens", "sum"),
        total_generated_tokens=("generated_tokens", "sum"),
        total_model_calls=("model_calls", "sum")
    )
    .reset_index()
)

display(overall_summary.round(4))

# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

assert len(evaluation_summary) == 6

assert full_results_df["model_calls"].sum() == 300

print("\n=== FINAL VERIFICATION ===")

print("Evaluation combinations:", len(evaluation_summary))

print(
    "Total model calls:",
    full_results_df["model_calls"].sum()
)

print("\nEvaluation aggregation completed successfully.")

=== FINAL RAG EVALUATION ANALYSIS ===

=== DATASET VERIFICATION ===
Total evaluation records: 120
Unique questions: 20
Fusion methods: ['concatenation', 'generate_from_evidence']
K values: [1, 3, 5]

Dataset integrity: PASSED

=== ACCURACY COMPARISON ===


,method,k,questions,exact_match_pct,token_f1_pct,retrieval_hit_pct,original_evidence_support_pct,prepared_evidence_support_pct,abstention_rate_pct
0,concatenation,1,20,75.0,85.62,100.0,90.0,85.0,10.0
1,concatenation,3,20,70.0,78.88,100.0,90.0,85.0,5.0
2,concatenation,5,20,65.0,73.88,100.0,90.0,85.0,0.0
3,generate_from_evidence,1,20,50.0,57.54,100.0,90.0,85.0,20.0
4,generate_from_evidence,3,20,35.0,43.66,100.0,90.0,85.0,25.0
5,generate_from_evidence,5,20,20.0,24.38,100.0,90.0,85.0,15.0



=== COMPUTATIONAL COST COMPARISON ===


,method,k,mean_retrieval_time,mean_generation_time,mean_input_tokens,mean_generated_tokens,total_model_calls
0,concatenation,1,0.0057,0.0722,125.10,5.50,20
1,concatenation,3,0.0052,0.0570,282.55,5.65,20
2,concatenation,5,0.0052,0.0564,436.90,5.70,20
3,generate_from_evidence,1,0.0055,0.1052,188.30,10.30,40
4,generate_from_evidence,3,0.0053,0.2003,471.20,20.55,80
5,generate_from_evidence,5,0.0054,0.2931,751.00,30.75,120



=== OVERALL FUSION METHOD COMPARISON ===


,method,evaluations,mean_exact_match,mean_token_f1,mean_generation_time,total_input_tokens,total_generated_tokens,total_model_calls
0,concatenation,60,0.70,0.7946,0.0618,16891,337,60
1,generate_from_evidence,60,0.35,0.4186,0.1995,28210,1232,240



=== FINAL VERIFICATION ===
Evaluation combinations: 6
Total model calls: 300

Evaluation aggregation completed successfully.


### Stage 6.6.1 — Systematic RAG Evaluation Results

#### Objective

Evaluate the effectiveness and computational cost of two
evidence-fusion strategies:

1. Concatenation Fusion
2. Generate-from-Evidence Fusion

Each strategy was evaluated using retrieval sizes K = 1, 3, 5
across 20 manually constructed, evidence-verified questions.

A total of 120 evaluations and 300 Flan-T5 generation calls
were completed.

#### Evaluation Metrics

- **Exact Match (EM):** Exact agreement after answer normalization.
- **Token F1:** Word-level overlap between prediction and reference.
- **Retrieval Hit@K:** Presence of expected article in top-K passages.
- **Evidence Support@K:** Normalized reference-answer presence.
- **Abstention Rate:** Frequency of "Insufficient evidence".
- **Computational Cost:** Model calls, generation time and token usage.

#### Accuracy Results

| Method                 | K   | EM (%) | F1 (%) |
| ---------------------- | --- | -----: | -----: |
| Concatenation          | 1   |     75 |  85.62 |
| Concatenation          | 3   |     70 |  78.88 |
| Concatenation          | 5   |     65 |  73.88 |
| Generate-from-Evidence | 1   |     50 |  57.54 |
| Generate-from-Evidence | 3   |     35 |  43.66 |
| Generate-from-Evidence | 5   |     20 |  24.38 |

#### Key Findings

**1. Best-performing configuration**

Concatenation with K=1 achieved the highest Exact Match
(75%) and Token F1 (85.62%).

**2. Impact of retrieval size**

Increasing K reduced Exact Match for both fusion strategies.

Concatenation declined from 75% at K=1 to 65% at K=5.

Generate-from-evidence declined from 50% at K=1 to
20% at K=5.

**3. Fusion method comparison**

Concatenation achieved higher accuracy with fewer
model calls and lower recorded generation time.

Across all 60 evaluations per method:

- Concatenation: EM = 70%, mean F1 = 0.7946.
- Generate-from-evidence: EM = 35%, mean F1 = 0.4186.

**4. Retrieval versus evidence availability**

Article-level Retrieval Hit@K was 100%.

However, only 90% of questions had reference answers
present in the original retrieved passages, and 85%
retained reference answers after evidence shortening.

Thus, correct article retrieval did not guarantee
answer-bearing evidence.

**5. Computational cost**

Concatenation required 60 generation calls.

Generate-from-evidence required 240 generation calls.

Average recorded generation times were 0.0618 seconds
and 0.1995 seconds per evaluation, respectively.

#### Limitations

- The evaluation dataset contains only 20 questions.
- Questions were constructed from the indexed corpus.
- Some evaluation examples informed development decisions.
- Exact Match can penalize factually correct paraphrases.
- Token F1 can award partial credit to factually incorrect answers.
- Evidence support uses normalized text matching rather than
  semantic entailment.
- Results reflect the selected Wikipedia slice, Flan-T5-small,
  TF-IDF retrieval and experimental prompt configurations.

#### Conclusion

Concatenation Fusion with K=1 provided the strongest
accuracy-efficiency tradeoff on the evaluated dataset.

Increasing the number of retrieved passages introduced
additional noise and, in several cases, reduced answer quality.

These findings motivate improved passage ranking, evidence
selection and source-aware answer synthesis as possible
future enhancements.


In [97]:
# ============================================================
# Stage 6.6.2 — Best Configuration Failure Analysis
# ============================================================

print("=== BEST CONFIGURATION FAILURE ANALYSIS ===")

best_results = full_results_df[
    (full_results_df["method"] == "concatenation")
    & (full_results_df["k"] == 1)
].copy()

assert len(best_results) == 20

print("\n=== PERFORMANCE SUMMARY ===")

print(
    "Exact matches:",
    int(best_results["exact_match"].sum()),
    "out of",
    len(best_results)
)

print(
    "Mean Token F1:",
    round(best_results["token_f1"].mean(), 4)
)

print(
    "Abstentions:",
    int(best_results["abstained"].sum())
)

# ------------------------------------------------------------
# Identify non-exact answers
# ------------------------------------------------------------

failures = best_results[
    best_results["exact_match"] == 0
].copy()

print("\n=== NON-EXACT ANSWERS ===")

display(
    failures[
        [
            "question_id",
            "question",
            "reference_answer",
            "answer",
            "token_f1",
            "retrieval_hit",
            "original_evidence_support",
            "prepared_evidence_support",
            "abstained"
        ]
    ]
)

# ------------------------------------------------------------
# Separate potential retrieval/evidence failures
# ------------------------------------------------------------

print("\n=== FAILURE DIAGNOSTICS ===")

print("Non-exact answers:", len(failures))

print(
    "Failures without original reference evidence:",
    int(
        (failures["original_evidence_support"] == 0).sum()
    )
)

print(
    "Failures without prepared reference evidence:",
    int(
        (failures["prepared_evidence_support"] == 0).sum()
    )
)

print(
    "Failures despite prepared reference evidence:",
    int(
        (failures["prepared_evidence_support"] == 1).sum()
    )
)

print("\nFailure analysis completed.")

=== BEST CONFIGURATION FAILURE ANALYSIS ===

=== PERFORMANCE SUMMARY ===
Exact matches: 15 out of 20
Mean Token F1: 0.8562
Abstentions: 2

=== NON-EXACT ANSWERS ===


,question_id,question,reference_answer,answer,token_f1,retrieval_hit,original_evidence_support,prepared_evidence_support,abstained
0,Q001,Who is Diana Richburg?,An American former middle-distance runner,middle-distance runner,0.750,1,1,1,False
42,Q008,In which month and year was Kavbiuro established?,April 1920,8 April,0.500,1,1,1,False
66,Q012,What was the role of an agonothetes in ancient...,President or superintendent of the sacred Panh...,president or superintendent of one of the sacr...,0.875,1,0,0,False
78,Q014,Who portrays Kate Beckett in the television se...,Stana Katic,Insufficient evidence,0.000,1,0,0,True
84,Q015,Who was John Hubley's second wife?,Faith Hubley,Insufficient evidence,0.000,1,1,0,True



=== FAILURE DIAGNOSTICS ===
Non-exact answers: 5
Failures without original reference evidence: 2
Failures without prepared reference evidence: 3
Failures despite prepared reference evidence: 2

Failure analysis completed.


In [98]:
# ============================================================
# Stage 6.6.3 — Manual Answer Quality Assessment
# ============================================================

import pandas as pd

print("=== MANUAL ANSWER QUALITY ASSESSMENT ===")

manual_assessments = [
    {
        "question_id": "Q001",
        "assessment": "factually_correct",
        "reason": "Correct occupation; shorter than reference."
    },
    {
        "question_id": "Q008",
        "assessment": "incomplete",
        "reason": "Correct month/day, but missing the requested year."
    },
    {
        "question_id": "Q012",
        "assessment": "factually_correct",
        "reason": "Equivalent meaning with different wording."
    },
    {
        "question_id": "Q014",
        "assessment": "appropriate_abstention",
        "reason": "Answer absent from retrieved evidence."
    },
    {
        "question_id": "Q015",
        "assessment": "appropriate_abstention",
        "reason": "Answer removed during 80-token shortening."
    }
]

manual_df = pd.DataFrame(manual_assessments)

# Verify exactly the five non-exact question IDs are covered
assert set(manual_df["question_id"]) == set(
    failures["question_id"]
)

assert manual_df["question_id"].is_unique

print("\n=== MANUAL ASSESSMENTS ===")

display(manual_df)

# ------------------------------------------------------------
# Calculate descriptive counts
# ------------------------------------------------------------

assessment_counts = (
    manual_df["assessment"]
    .value_counts()
    .rename_axis("assessment")
    .reset_index(name="count")
)

print("\n=== ASSESSMENT SUMMARY ===")

display(assessment_counts)

exact_matches = int(best_results["exact_match"].sum())

additional_correct = int(
    (manual_df["assessment"] == "factually_correct").sum()
)

manual_correct_total = exact_matches + additional_correct

print("\nExact matches:", exact_matches)

print(
    "Additional factually correct answers:",
    additional_correct
)

print(
    "Preliminary factually correct answers:",
    manual_correct_total,
    "out of",
    len(best_results)
)

print(
    "Preliminary factual correctness:",
    f"{manual_correct_total / len(best_results) * 100:.2f}%"
)

print("\nManual assessment preparation completed.")

=== MANUAL ANSWER QUALITY ASSESSMENT ===

=== MANUAL ASSESSMENTS ===


,question_id,assessment,reason
0,Q001,factually_correct,Correct occupation; shorter than reference.
1,Q008,incomplete,"Correct month/day, but missing the requested y..."
2,Q012,factually_correct,Equivalent meaning with different wording.
3,Q014,appropriate_abstention,Answer absent from retrieved evidence.
4,Q015,appropriate_abstention,Answer removed during 80-token shortening.



=== ASSESSMENT SUMMARY ===


,assessment,count
0,factually_correct,2
1,appropriate_abstention,2
2,incomplete,1



Exact matches: 15
Additional factually correct answers: 2
Preliminary factually correct answers: 17 out of 20
Preliminary factual correctness: 85.00%

Manual assessment preparation completed.


In [99]:
# ============================================================
# Stage 6.6.4 — Save Final Evaluation Analysis
# ============================================================

from pathlib import Path
import pandas as pd

print("=== FINAL EVALUATION ANALYSIS PERSISTENCE ===")

# ------------------------------------------------------------
# 1. Define output paths
# ------------------------------------------------------------

ANALYSIS_DIR = PROJECT_ROOT / "results" / "evaluation"

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FULL_CSV_PATH = (
    ANALYSIS_DIR / "rag_full_evaluation.csv"
)

SUMMARY_CSV_PATH = (
    ANALYSIS_DIR / "rag_evaluation_summary.csv"
)

OVERALL_CSV_PATH = (
    ANALYSIS_DIR / "rag_fusion_comparison.csv"
)

MANUAL_CSV_PATH = (
    ANALYSIS_DIR / "rag_manual_assessment.csv"
)

output_paths = [
    FULL_CSV_PATH,
    SUMMARY_CSV_PATH,
    OVERALL_CSV_PATH,
    MANUAL_CSV_PATH
]

# ------------------------------------------------------------
# 2. Protect existing results
# ------------------------------------------------------------

existing_paths = [
    path for path in output_paths
    if path.exists()
]

if existing_paths:
    raise FileExistsError(
        f"Analysis files already exist: {existing_paths}"
    )

# ------------------------------------------------------------
# 3. Save evaluation artifacts
# ------------------------------------------------------------

full_results_df.to_csv(
    FULL_CSV_PATH,
    index=False
)

evaluation_summary.to_csv(
    SUMMARY_CSV_PATH,
    index=False
)

overall_summary.to_csv(
    OVERALL_CSV_PATH,
    index=False
)

manual_df.to_csv(
    MANUAL_CSV_PATH,
    index=False
)

# ------------------------------------------------------------
# 4. Reload and verify
# ------------------------------------------------------------

loaded_full = pd.read_csv(FULL_CSV_PATH)
loaded_summary = pd.read_csv(SUMMARY_CSV_PATH)
loaded_overall = pd.read_csv(OVERALL_CSV_PATH)
loaded_manual = pd.read_csv(MANUAL_CSV_PATH)

assert len(loaded_full) == 120
assert len(loaded_summary) == 6
assert len(loaded_overall) == 2
assert len(loaded_manual) == 5

assert loaded_full["model_calls"].sum() == 300

print("\n=== SAVED ARTIFACTS ===")

for path in output_paths:
    print(path.name, ":", path.exists())

print("\n=== RELOAD VERIFICATION ===")

print("Individual evaluations:", len(loaded_full))
print("Method/K combinations:", len(loaded_summary))
print("Fusion methods:", len(loaded_overall))
print("Manual assessments:", len(loaded_manual))

print("\nFinal evaluation analysis persistence: PASSED")

=== FINAL EVALUATION ANALYSIS PERSISTENCE ===

=== SAVED ARTIFACTS ===
rag_full_evaluation.csv : True
rag_evaluation_summary.csv : True
rag_fusion_comparison.csv : True
rag_manual_assessment.csv : True

=== RELOAD VERIFICATION ===
Individual evaluations: 120
Method/K combinations: 6
Fusion methods: 2
Manual assessments: 5

Final evaluation analysis persistence: PASSED


# Stage 6 — RAG Evaluation and Ablation Study

## 6.1 Objective

This stage evaluates the quality, efficiency, and limitations of
the implemented Retrieval-Augmented Generation (RAG) system.

The system combines:

- A Wikipedia-based document corpus
- Word-based passage chunking
- TF-IDF passage indexing
- Cosine-similarity retrieval
- Flan-T5-small answer generation

The experiments investigate:

1. The effect of retrieval size K.
2. The effectiveness of evidence selection strategies.
3. The comparison of two evidence-fusion methods.
4. The relationship between retrieval quality and answer quality.
5. The computational cost of answer generation.

---

## 6.2 Evaluation Dataset

A manually constructed evaluation dataset containing 20
question-answer pairs was prepared.

The questions cover 14 Wikipedia articles from the indexed corpus.

Each evaluation record includes:

- Question ID
- Question
- Reference answer
- Expected source article

Reference answers were checked against supporting source passages.

The dataset was saved in CSV and JSON formats.

---

## 6.3 Evidence Selection Experiments

The Flan-T5-small model has a maximum input length of 512 tokens.

Experiments using complete retrieved passages showed that:

- K=1: All 20 prompts fit within the input limit.
- K=3: Three questions exceeded the input limit.
- K=5: All 20 questions exceeded the input limit.

To address this limitation, evidence-shortening strategies were explored.

### Compared Strategies

1. Original retrieved passages
2. Prefix-based token shortening
3. Query-aware sentence selection

### Evidence Retention

Across the 20 questions:

| Method                         | Questions retaining reference text |
| ------------------------------ | ---------------------------------: |
| Original retrieved passages    |                                 19 |
| Prefix shortening (80 tokens)  |                                 18 |
| Query-aware sentence selection |                                 13 |

Prefix-based shortening was selected as the controlled experimental
baseline because it preserved more reference answers than the tested
query-aware sentence-selection method.

However, prefix shortening could remove information appearing
later in a passage.

The John Hubley example demonstrated this limitation:
the reference answer "Faith Hubley" appeared in the original
passage but was removed by the 80-token cutoff.

---

## 6.4 Evidence Fusion Strategies

Two fusion approaches were evaluated.

### Method A — Concatenation Fusion

The top-K retrieved passages are shortened and concatenated.

The combined evidence is supplied to Flan-T5-small in one
generation call.

**Model calls per question:** 1

### Method B — Generate-from-Evidence Fusion

Flan-T5-small independently generates candidate answers from
each of the K retrieved passages.

A final generation call synthesizes the candidate answers.

**Model calls per question:** K + 1

Both strategies were implemented in the reusable Python project.

---

## 6.5 Systematic Evaluation

The evaluation used:

- 20 questions
- K = 1, 3, 5
- Two fusion methods
- Maximum 80 evidence tokens per retrieved passage
- Flan-T5-small as the generator

A total of:

20 × 3 × 2 = 120 evaluations

were conducted.

These required 300 model-generation calls.

Evaluation results were checkpointed to disk to support
interruption recovery.

### Evaluation Metrics

**Exact Match (EM)**

Measures whether normalized prediction and reference are identical.

**Token F1**

Measures word-token overlap using precision and recall.

F1 = 2 × Precision × Recall / (Precision + Recall)

**Retrieval Hit@K**

Measures whether the expected Wikipedia article appears among
the top-K retrieved passages.

**Evidence Support@K**

Checks whether the normalized reference answer appears in at
least one retrieved passage.

This is a lexical proxy, not a semantic entailment measure.

**Abstention Rate**

Measures how frequently the model responds with
"Insufficient evidence".

**Computational Cost**

Measured using model calls, token counts and generation time.

---

## 6.6 Quantitative Results

### Accuracy Comparison

| Fusion Method          |   K | Exact Match (%) | Token F1 (%) |
| ---------------------- | --: | --------------: | -----------: |
| Concatenation          |   1 |              75 |        85.62 |
| Concatenation          |   3 |              70 |        78.88 |
| Concatenation          |   5 |              65 |        73.88 |
| Generate-from-Evidence |   1 |              50 |        57.54 |
| Generate-from-Evidence |   3 |              35 |        43.66 |
| Generate-from-Evidence |   5 |              20 |        24.38 |

### Overall Fusion Comparison

| Metric               | Concatenation | Generate-from-Evidence |
| -------------------- | ------------: | ---------------------: |
| Evaluations          |            60 |                     60 |
| Mean Exact Match     |           70% |                    35% |
| Mean Token F1        |        79.46% |                 41.86% |
| Mean generation time |      0.0618 s |               0.1995 s |
| Total model calls    |            60 |                    240 |
| Total input tokens   |        16,891 |                 28,210 |

The recorded generation time represents measured model-generation
time rather than complete end-to-end application latency.

### Retrieval and Evidence Support

Across the evaluated configurations:

- Article Retrieval Hit@K: 100%
- Original evidence support: 90%
- Prepared evidence support: 85%

These findings demonstrate that correct article retrieval
does not necessarily guarantee answer-bearing passages.

---

## 6.7 Failure Analysis

The best-performing configuration was:

**Concatenation Fusion with K=1**

This configuration achieved:

- Exact Match: 75%
- Mean Token F1: 0.8562
- Exact matches: 15/20
- Abstentions: 2/20

Five answers did not exactly match their references.

Manual inspection suggested:

- Two additional answers were factually correct.
- One answer was incomplete.
- Two abstentions were appropriate given the supplied evidence.

The resulting preliminary factual-correctness estimate was
17/20 = 85%.

This manually assessed figure is separate from the
automated Exact Match metric.

### Representative Failure Cases

**Q001 — Diana Richburg**

The answer "middle-distance runner" was factually correct
but shorter than the reference answer.

**Q008 — Kavbiuro**

The answer "8 April" omitted the year 1920.

**Q012 — Agonothetes**

The generated answer was semantically equivalent to the
reference but differed in wording.

**Q014 — Kate Beckett**

The correct article was retrieved, but the answer-bearing
passage was absent from the retrieved top-K evidence.

**Q015 — John Hubley**

The answer appeared in the original retrieved passage but
was removed by prefix-based evidence shortening.

---

## 6.8 Main Findings

### Finding 1: Concatenation performed better

Concatenation achieved higher Exact Match and Token F1
than generate-from-evidence at every evaluated K.

### Finding 2: Increasing K did not improve accuracy

Both fusion approaches exhibited declining Exact Match
as retrieval size increased.

Additional passages sometimes introduced unrelated information,
which contaminated generated answers.

### Finding 3: More generation calls did not improve results

Generate-from-evidence required substantially more model calls
but achieved lower accuracy.

### Finding 4: Retrieval and generation failures are different

An incorrect RAG answer may result from:

- Failure to retrieve an answer-bearing passage
- Loss of evidence during shortening
- Conflicting or irrelevant evidence
- Incorrect synthesis of candidate answers
- Incomplete or unsupported generation

### Finding 5: Multiple evaluation metrics are necessary

Exact Match may penalize factually correct paraphrases.

Token F1 may award partial credit to factually incorrect answers.

Qualitative assessment is therefore useful alongside
automated metrics.

---

## 6.9 Limitations

1. The evaluation dataset contains only 20 questions.
2. Questions were manually constructed from the indexed corpus.
3. Some evaluation examples informed development decisions.
4. The results may not generalize to arbitrary Wikipedia questions.
5. Evidence support was measured using lexical text matching.
6. A fixed 80-token cutoff may remove important evidence.
7. Only one generator model was evaluated.
8. The reported timings depend on the local execution environment.

---

## 6.10 Stage 6 Conclusion

The systematic evaluation identified Concatenation Fusion with
K=1 as the strongest configuration for the selected
Wikipedia-based RAG system.

It achieved:

- 75% Exact Match
- 85.62% Token F1

It also required fewer generation calls and lower computational
cost than generate-from-evidence fusion.

The experiments demonstrated that increasing retrieval size
does not necessarily improve answer quality.

The results also highlighted limitations in passage ranking,
evidence shortening, and answer synthesis.

These findings provide the experimental basis for selecting
the final RAG configuration and documenting potential
future improvements.


In [100]:
# ============================================================
# Stage 7.1.1 — Inspect Existing RAG Pipeline Interfaces
# ============================================================

import inspect

from rag.pipeline import RAGPipeline
from rag.retriever import TfidfRetriever
from rag.generator import FlanT5Generator
from rag.indexer import load_tfidf_index

print("=== FINAL RAG PIPELINE INTERFACE INSPECTION ===")

print("\n=== PIPELINE CONSTRUCTOR ===")
print(inspect.signature(RAGPipeline.__init__))

print("\n=== PIPELINE IMPLEMENTATION ===")
print(inspect.getsource(RAGPipeline))

print("\n=== RETRIEVER CONSTRUCTOR ===")
print(inspect.signature(TfidfRetriever.__init__))

print("\n=== GENERATOR CONSTRUCTOR ===")
print(inspect.signature(FlanT5Generator.__init__))

print("\n=== INDEX LOADER SIGNATURE ===")
print(inspect.signature(load_tfidf_index))

print("\n=== CURRENT GENERATOR DEVICE ===")
print(next(generator.model.parameters()).device)

print("\n=== CURRENT PIPELINE COMPONENTS ===")
print("Retriever:", type(retriever).__name__)
print("Generator:", type(generator).__name__)

print("\nPipeline interface inspection completed.")

=== FINAL RAG PIPELINE INTERFACE INSPECTION ===

=== PIPELINE CONSTRUCTOR ===
(self, retriever, generator)

=== PIPELINE IMPLEMENTATION ===
class RAGPipeline:
    """
    Coordinate retrieval and evidence-grounded answer generation.

    The retriever and generator are supplied during initialization
    so existing model and index instances can be reused.
    """

    def __init__(self, retriever, generator):

        self.retriever = retriever
        self.generator = generator

    def answer(self, question, top_k=1):
        """
        Retrieve Wikipedia evidence and generate an answer.
        """

        # ----------------------------------------------------
        # Step 1: Start pipeline timer
        # ----------------------------------------------------

        pipeline_start = time.perf_counter()

        # ----------------------------------------------------
        # Step 2: Retrieve relevant Wikipedia passages
        # -------------------------------------------------

In [101]:
# ============================================================
# Stage 7.1.3 — Final Integrated RAG Pipeline Verification
# ============================================================

import importlib

import rag.pipeline as pipeline_module

importlib.invalidate_caches()
pipeline_module = importlib.reload(pipeline_module)

RAGPipeline = pipeline_module.RAGPipeline

print("=== FINAL INTEGRATED RAG PIPELINE VERIFICATION ===")

# ------------------------------------------------------------
# 1. Create final pipeline using existing components
# ------------------------------------------------------------

final_rag_pipeline = RAGPipeline(
    retriever=retriever,
    generator=generator,
    max_evidence_tokens=80,
    fusion_method="concatenation"
)

print("\n=== PIPELINE CONFIGURATION ===")

print("Retriever:", type(final_rag_pipeline.retriever).__name__)
print("Generator:", type(final_rag_pipeline.generator).__name__)
print("Fusion:", final_rag_pipeline.fusion_method)
print("Evidence token limit:", final_rag_pipeline.max_evidence_tokens)

# Verify existing objects are reused
assert final_rag_pipeline.retriever is retriever
assert final_rag_pipeline.generator is generator

print("Existing component reuse: Passed")

# ------------------------------------------------------------
# 2. Execute final RAG pipeline
# ------------------------------------------------------------

QUESTION = "Who is Diana Richburg?"

result = final_rag_pipeline.answer(
    question=QUESTION
)

print("\n=== GENERATED ANSWER ===")

print("Question:", result["question"])
print("Answer:", result["answer"])

print("\n=== RETRIEVAL INFORMATION ===")

print("Top-K:", result["top_k"])
print("Retrieved sources:", len(result["sources"]))
print("Prepared sources:", len(result["prepared_sources"]))

# ------------------------------------------------------------
# 3. Verify evidence preparation
# ------------------------------------------------------------

prepared_text = result["prepared_sources"][0]["text"]

prepared_token_count = len(
    generator.tokenizer(
        prepared_text,
        add_special_tokens=False
    )["input_ids"]
)

print("\n=== EVIDENCE PREPARATION ===")

print("Prepared evidence tokens:", prepared_token_count)
print("Maximum allowed:", result["max_evidence_tokens"])

assert prepared_token_count <= 80

assert len(result["sources"]) == 1
assert len(result["prepared_sources"]) == 1

print("Evidence token limit: Passed")
print("Source count verification: Passed")

# ------------------------------------------------------------
# 4. Verify generated answer
# ------------------------------------------------------------

assert result["answer"].strip(), \
    "Generated answer is empty!"

print("\n=== PERFORMANCE METRICS ===")

print(
    "Retrieval time:",
    round(result["retrieval_time_seconds"], 4),
    "seconds"
)

print(
    "Generation time:",
    round(result["generation_time_seconds"], 4),
    "seconds"
)

print(
    "Total pipeline time:",
    round(result["total_time_seconds"], 4),
    "seconds"
)

print("Input tokens:", result["input_tokens"])
print("Generated tokens:", result["generated_tokens"])
print("Device:", result["device"])

# ------------------------------------------------------------
# 5. Final verification
# ------------------------------------------------------------

assert result["top_k"] == 1

assert result["fusion_method"] == "concatenation"

assert result["max_evidence_tokens"] == 80

assert result["device"] == "cuda"

print("\n=== FINAL VERIFICATION ===")

print("Configuration consistency: Passed")
print("Generator reuse: Passed")
print("Evidence preparation: Passed")
print("Answer generation: Passed")

print("\nAll final RAG pipeline integration checks passed.")

=== FINAL INTEGRATED RAG PIPELINE VERIFICATION ===

=== PIPELINE CONFIGURATION ===
Retriever: TfidfRetriever
Generator: FlanT5Generator
Fusion: concatenation
Evidence token limit: 80
Existing component reuse: Passed

=== GENERATED ANSWER ===
Question: Who is Diana Richburg?
Answer: middle-distance runner

=== RETRIEVAL INFORMATION ===
Top-K: 1
Retrieved sources: 1
Prepared sources: 1

=== EVIDENCE PREPARATION ===
Prepared evidence tokens: 80
Maximum allowed: 80
Evidence token limit: Passed
Source count verification: Passed

=== PERFORMANCE METRICS ===
Retrieval time: 0.0079 seconds
Generation time: 0.1263 seconds
Total pipeline time: 0.1373 seconds
Input tokens: 119
Generated tokens: 9
Device: cuda

=== FINAL VERIFICATION ===
Configuration consistency: Passed
Generator reuse: Passed
Evidence preparation: Passed
Answer generation: Passed

All final RAG pipeline integration checks passed.


In [102]:
# ============================================================
# Stage 7.2.1 — Inspect Saved RAG Artifacts and Loaders
# ============================================================

from pathlib import Path
import inspect

from rag.indexer import load_tfidf_index
from rag.retriever import TfidfRetriever

print("=== STANDALONE RAG ARTIFACT INSPECTION ===")

# ------------------------------------------------------------
# 1. Inspect saved passage directory
# ------------------------------------------------------------

PASSAGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "wiki_chunks_100w_20overlap"
)

print("\n=== SAVED PASSAGE FILES ===")

print("Passage directory:", PASSAGE_DIR)

if PASSAGE_DIR.exists():

    passage_files = sorted(
        path
        for path in PASSAGE_DIR.iterdir()
        if path.is_file()
    )

    for path in passage_files:
        print(
            path.name,
            "| Size:",
            path.stat().st_size,
            "bytes"
        )

else:
    print("Passage directory not found!")

# ------------------------------------------------------------
# 2. Inspect TF-IDF index files
# ------------------------------------------------------------

INDEX_DIR = PROJECT_ROOT / "data" / "tfidf_index"

print("\n=== SAVED INDEX FILES ===")

for path in sorted(INDEX_DIR.iterdir()):

    if path.is_file():

        print(
            path.name,
            "| Size:",
            path.stat().st_size,
            "bytes"
        )

# ------------------------------------------------------------
# 3. Inspect existing loader implementation
# ------------------------------------------------------------

print("\n=== TF-IDF INDEX LOADER ===")

print(inspect.getsource(load_tfidf_index))

# ------------------------------------------------------------
# 4. Inspect retriever constructor
# ------------------------------------------------------------

print("\n=== RETRIEVER INITIALIZATION ===")

print(inspect.getsource(TfidfRetriever.__init__))

# ------------------------------------------------------------
# 5. Inspect passage structure
# ------------------------------------------------------------

print("\n=== PASSAGE RECORD STRUCTURE ===")

print("Number of loaded passages:", len(chunks_local))

print("First passage fields:")
print(list(chunks_local[0].keys()))

print("\nFirst passage metadata:")

for key, value in chunks_local[0].items():

    if key != "text":
        print(key, ":", value)

print("\nStandalone artifact inspection completed.")

=== STANDALONE RAG ARTIFACT INSPECTION ===

=== SAVED PASSAGE FILES ===
Passage directory: c:\AshutoshLocal\explore\Code\local-workspaces\IIT-B\Sem-2\DeepLearning-Prof-Biplab\code\Mid-Sem-Project-RAG\data\wiki_chunks_100w_20overlap
data-00000-of-00001.arrow | Size: 6196200 bytes
dataset_info.json | Size: 666 bytes
state.json | Size: 259 bytes

=== SAVED INDEX FILES ===
index_metadata.json | Size: 348 bytes
tfidf_matrix.npz | Size: 3106058 bytes
vectorizer.joblib | Size: 1115589 bytes

=== TF-IDF INDEX LOADER ===
def load_tfidf_index(index_dir: str | Path):
    """
    Load a previously saved TF-IDF index from local storage.

    Parameters
    ----------
    index_dir : str or Path
        Directory containing the saved index artifacts.

    Returns
    -------
    tuple
        Fitted vectorizer, sparse matrix, and metadata dictionary.

    Raises
    ------
    FileNotFoundError
        If an expected artifact is missing.

    ValueError
        If the index dimensions disagree with 

In [103]:
# ============================================================
# Stage 7.2.2 — Verify Standalone Data Loading
# ============================================================

from datasets import load_from_disk
from rag.indexer import load_tfidf_index

print("=== STANDALONE DATA LOADING VERIFICATION ===")

# ------------------------------------------------------------
# 1. Load saved passages independently
# ------------------------------------------------------------

standalone_passages = load_from_disk(
    str(PASSAGE_DIR)
)

print("\n=== LOADED PASSAGES ===")

print("Dataset type:", type(standalone_passages).__name__)
print("Number of passages:", len(standalone_passages))

# ------------------------------------------------------------
# 2. Load saved TF-IDF index independently
# ------------------------------------------------------------

standalone_vectorizer, standalone_matrix, standalone_metadata = (
    load_tfidf_index(INDEX_DIR)
)

print("\n=== LOADED TF-IDF INDEX ===")

print("Matrix shape:", standalone_matrix.shape)
print("Vocabulary size:", len(standalone_vectorizer.vocabulary_))

# ------------------------------------------------------------
# 3. Verify passage counts
# ------------------------------------------------------------

assert len(standalone_passages) == 8191

assert standalone_matrix.shape == (8191, 30000)

assert len(standalone_passages) == standalone_matrix.shape[0]

print("\nPassage count alignment: Passed")

# ------------------------------------------------------------
# 4. Verify passage order
# ------------------------------------------------------------

sample_indices = [
    0,
    len(standalone_passages) // 2,
    len(standalone_passages) - 1
]

print("\n=== PASSAGE ALIGNMENT CHECK ===")

for index in sample_indices:

    saved_passage = standalone_passages[index]
    notebook_passage = chunks_local[index]

    assert (
        saved_passage["chunk_id"]
        == notebook_passage["chunk_id"]
    )

    assert (
        saved_passage["text"]
        == notebook_passage["text"]
    )

    print(
        f"Row {index}: "
        f"{saved_passage['chunk_id']} — Passed"
    )

print("\nPassage order verification: Passed")

# ------------------------------------------------------------
# 5. Verify index metadata
# ------------------------------------------------------------

assert (
    standalone_metadata["num_passages"]
    == len(standalone_passages)
)

assert (
    standalone_metadata["vocabulary_size"]
    == standalone_matrix.shape[1]
)

print("Index metadata consistency: Passed")

print("\nAll standalone data loading checks passed.")

=== STANDALONE DATA LOADING VERIFICATION ===

=== LOADED PASSAGES ===
Dataset type: Dataset
Number of passages: 8191

=== LOADED TF-IDF INDEX ===
Matrix shape: (8191, 30000)
Vocabulary size: 30000

Passage count alignment: Passed

=== PASSAGE ALIGNMENT CHECK ===
Row 0: 55617566::0 — Passed
Row 4095: 28821362::25 — Passed
Row 8190: 28820436::15 — Passed

Passage order verification: Passed
Index metadata consistency: Passed

All standalone data loading checks passed.


In [104]:
# ============================================================
# Stage 7.3.1 — Reproducibility Environment Inspection
# ============================================================

import sys
import platform
import importlib.metadata

import torch
import transformers
import datasets
import sklearn

print("=== RAG PROJECT REPRODUCIBILITY INFORMATION ===")

# ------------------------------------------------------------
# 1. Python environment
# ------------------------------------------------------------

print("\n=== PYTHON ENVIRONMENT ===")

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Operating system:", platform.platform())

# ------------------------------------------------------------
# 2. Library versions
# ------------------------------------------------------------

print("\n=== LIBRARY VERSIONS ===")

libraries = [
    "torch",
    "transformers",
    "datasets",
    "scikit-learn",
    "scipy",
    "numpy",
    "pandas",
    "joblib",
    "sentencepiece",
    "accelerate"
]

for library in libraries:

    try:
        version = importlib.metadata.version(library)
        print(f"{library}: {version}")

    except importlib.metadata.PackageNotFoundError:
        print(f"{library}: Not installed")

# ------------------------------------------------------------
# 3. GPU information
# ------------------------------------------------------------

print("\n=== GPU INFORMATION ===")

print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    total_memory = torch.cuda.get_device_properties(
        0
    ).total_memory

    print(
        "Total GPU memory:",
        round(total_memory / (1024 ** 3), 2),
        "GB"
    )

# ------------------------------------------------------------
# 4. RAG configuration
# ------------------------------------------------------------

print("\n=== FINAL RAG CONFIGURATION ===")

print("Dataset: wikimedia/wikipedia")
print("Dataset configuration: 20231101.en")
print("Indexed passages: 8191")
print("TF-IDF vocabulary size: 30000")
print("Chunk size: 100 words")
print("Chunk overlap: 20 words")
print("Retrieval K: 1")
print("Evidence token limit: 80")
print("Fusion method: concatenation")
print("Generator: google/flan-t5-small")
print("Maximum input tokens: 512")

print("\nReproducibility environment inspection completed.")

=== RAG PROJECT REPRODUCIBILITY INFORMATION ===

=== PYTHON ENVIRONMENT ===
Python version: 3.11.14
Python executable: c:\Users\Ashutosh Chittora\anaconda3\envs\.cvenv\python.exe
Operating system: Windows-10-10.0.26200-SP0

=== LIBRARY VERSIONS ===
torch: 2.10.0+cu128
transformers: 5.21.0
datasets: 5.1.0
scikit-learn: 1.8.0
scipy: 1.17.1
numpy: 2.1.3
pandas: 3.0.1
joblib: 1.5.3
sentencepiece: 0.2.2
accelerate: 1.15.0

=== GPU INFORMATION ===
CUDA available: True
PyTorch CUDA version: 12.8
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Total GPU memory: 7.96 GB

=== FINAL RAG CONFIGURATION ===
Dataset: wikimedia/wikipedia
Dataset configuration: 20231101.en
Indexed passages: 8191
TF-IDF vocabulary size: 30000
Chunk size: 100 words
Chunk overlap: 20 words
Retrieval K: 1
Evidence token limit: 80
Fusion method: concatenation
Generator: google/flan-t5-small
Maximum input tokens: 512

Reproducibility environment inspection completed.


In [105]:
# ============================================================
# Stage 8.2.2 — Documentation Consistency Verification
# ============================================================

import json
import pandas as pd

print("=== PROJECT DOCUMENTATION CONSISTENCY CHECK ===")

# ------------------------------------------------------------
# 1. Load final configuration
# ------------------------------------------------------------

config_path = PROJECT_ROOT / "config" / "rag_config.json"

with open(config_path, "r", encoding="utf-8") as f:
    final_config = json.load(f)

# ------------------------------------------------------------
# 2. Load saved index metadata
# ------------------------------------------------------------

metadata_path = (
    PROJECT_ROOT
    / "data"
    / "tfidf_index"
    / "index_metadata.json"
)

with open(metadata_path, "r", encoding="utf-8") as f:
    index_metadata = json.load(f)

# ------------------------------------------------------------
# 3. Verify configuration against index
# ------------------------------------------------------------

print("\n=== INDEX CONFIGURATION ===")

checks = {
    "Passage count": (
        final_config["data"]["num_passages"]
        == index_metadata["num_passages"]
    ),

    "TF-IDF vocabulary": (
        final_config["retrieval"]["max_features"]
        == index_metadata["vocabulary_size"]
    ),

    "Chunk size": (
        final_config["chunking"]["chunk_size_words"]
        == index_metadata["chunk_size_words"]
    ),

    "Chunk overlap": (
        final_config["chunking"]["chunk_overlap_words"]
        == index_metadata["chunk_overlap_words"]
    ),

    "Final K": (
        final_config["retrieval"]["top_k"] == 1
    ),

    "Evidence token limit": (
        final_config["evidence"]["max_tokens_per_passage"] == 80
    ),

    "Fusion method": (
        final_config["evidence"]["fusion_method"]
        == "concatenation"
    ),

    "Generator": (
        final_config["generator"]["model_name"]
        == "google/flan-t5-small"
    )
}

for name, passed in checks.items():
    print(f"{name}: {'Passed' if passed else 'FAILED'}")

assert all(checks.values()), \
    "Configuration consistency check failed!"

# ------------------------------------------------------------
# 4. Verify evaluation results
# ------------------------------------------------------------

print("\n=== EVALUATION RESULT CONSISTENCY ===")

results_path = (
    PROJECT_ROOT
    / "results"
    / "evaluation"
    / "rag_full_evaluation.csv"
)

saved_results = pd.read_csv(results_path)

assert len(saved_results) == 120

best_results = saved_results[
    (saved_results["method"] == "concatenation")
    & (saved_results["k"] == 1)
]

assert len(best_results) == 20

best_em = best_results["exact_match"].mean()

best_f1 = best_results["token_f1"].mean()

print("Best configuration evaluations:", len(best_results))
print("Exact Match:", round(best_em * 100, 2), "%")
print("Token F1:", round(best_f1 * 100, 2), "%")

assert abs(best_em - 0.75) < 1e-10

assert abs(best_f1 - 0.8562) < 0.0001

print("Evaluation result consistency: Passed")

# ------------------------------------------------------------
# 5. Verify required project files
# ------------------------------------------------------------

print("\n=== REQUIRED PROJECT FILES ===")

required_files = [
    "README.md",
    "requirements.txt",
    "config/rag_config.json",
    "docs/architecture.md",
    "src/run_rag.py",
    "src/rag/indexer.py",
    "src/rag/retriever.py",
    "src/rag/generator.py",
    "src/rag/evidence.py",
    "src/rag/fusion.py",
    "src/rag/pipeline.py",
    "src/rag/evaluation.py"
]

for relative_path in required_files:

    path = PROJECT_ROOT / relative_path

    exists = path.is_file()

    print(
        f"{relative_path}:",
        "Present" if exists else "MISSING"
    )

    assert exists, f"Missing project file: {relative_path}"

print("\nAll documentation consistency checks passed.")

=== PROJECT DOCUMENTATION CONSISTENCY CHECK ===

=== INDEX CONFIGURATION ===
Passage count: Passed
TF-IDF vocabulary: Passed
Chunk size: Passed
Chunk overlap: Passed
Final K: Passed
Evidence token limit: Passed
Fusion method: Passed
Generator: Passed

=== EVALUATION RESULT CONSISTENCY ===
Best configuration evaluations: 20
Exact Match: 75.0 %
Token F1: 85.62 %
Evaluation result consistency: Passed

=== REQUIRED PROJECT FILES ===
README.md: Present
requirements.txt: Present
config/rag_config.json: Present
docs/architecture.md: Present
src/run_rag.py: Present
src/rag/indexer.py: Present
src/rag/retriever.py: Present
src/rag/generator.py: Present
src/rag/evidence.py: Present
src/rag/fusion.py: Present
src/rag/pipeline.py: Present
src/rag/evaluation.py: Present

All documentation consistency checks passed.


In [106]:
# ============================================================
# Stage 9.1.1 — Final RAG Project File Audit
# ============================================================

from pathlib import Path
import pandas as pd

print("=== FINAL RAG PROJECT FILE AUDIT ===")

# ------------------------------------------------------------
# 1. Required files
# ------------------------------------------------------------

required_files = {
    "Main notebook": "Project-Offline-RAG-Wikipedia.ipynb",

    "Application": "src/run_rag.py",

    "Data loader": "src/rag/data_loader.py",
    "Chunking": "src/rag/chunking.py",
    "Indexer": "src/rag/indexer.py",
    "Retriever": "src/rag/retriever.py",
    "Generator": "src/rag/generator.py",
    "Pipeline": "src/rag/pipeline.py",
    "Evidence": "src/rag/evidence.py",
    "Fusion": "src/rag/fusion.py",
    "Evaluation": "src/rag/evaluation.py",

    "Configuration": "config/rag_config.json",
    "Dependencies": "requirements.txt",
    "README": "README.md",
    "Architecture": "docs/architecture.md",
    "Report source": "docs/project_report.md",

    "Index vectorizer": "data/tfidf_index/vectorizer.joblib",
    "TF-IDF matrix": "data/tfidf_index/tfidf_matrix.npz",
    "Index metadata": "data/tfidf_index/index_metadata.json",

    "Evaluation questions": "data/evaluation/rag_evaluation_20.csv",

    "Full evaluation": "results/evaluation/rag_full_evaluation.csv",
    "Evaluation summary": "results/evaluation/rag_evaluation_summary.csv",
    "Fusion comparison": "results/evaluation/rag_fusion_comparison.csv",
    "Manual assessment": "results/evaluation/rag_manual_assessment.csv"
}

audit_results = []

for description, relative_path in required_files.items():

    path = PROJECT_ROOT / relative_path

    exists = path.is_file()

    size_kb = (
        round(path.stat().st_size / 1024, 2)
        if exists
        else 0
    )

    audit_results.append({
        "component": description,
        "file": relative_path,
        "exists": exists,
        "size_kb": size_kb
    })

audit_df = pd.DataFrame(audit_results)

print("\n=== FILE AUDIT RESULTS ===")

display(audit_df)

# ------------------------------------------------------------
# 2. Verify required directories
# ------------------------------------------------------------

required_directories = {
    "Wikipedia articles": "data/wiki_slice_1000",
    "Wikipedia passages": "data/wiki_chunks_100w_20overlap",
    "TF-IDF index": "data/tfidf_index",
    "Evaluation dataset": "data/evaluation",
    "Evaluation results": "results/evaluation",
    "Python modules": "src/rag"
}

print("\n=== DIRECTORY AUDIT ===")

directory_results = []

for description, relative_path in required_directories.items():

    path = PROJECT_ROOT / relative_path

    exists = path.is_dir()

    directory_results.append({
        "component": description,
        "directory": relative_path,
        "exists": exists
    })

directory_df = pd.DataFrame(directory_results)

display(directory_df)

# ------------------------------------------------------------
# 3. Final summary
# ------------------------------------------------------------

missing_files = audit_df[
    audit_df["exists"] == False
]

missing_directories = directory_df[
    directory_df["exists"] == False
]

print("\n=== AUDIT SUMMARY ===")

print("Required files:", len(audit_df))
print("Files present:", int(audit_df["exists"].sum()))
print("Missing files:", len(missing_files))

print("Required directories:", len(directory_df))
print("Directories present:", int(directory_df["exists"].sum()))
print("Missing directories:", len(missing_directories))

if missing_files.empty and missing_directories.empty:

    print("\nFinal project file audit: PASSED")

else:

    print("\nFinal project file audit: REQUIRES ATTENTION")

    if not missing_files.empty:
        print("\nMissing files:")
        display(missing_files)

    if not missing_directories.empty:
        print("\nMissing directories:")
        display(missing_directories)

=== FINAL RAG PROJECT FILE AUDIT ===

=== FILE AUDIT RESULTS ===


,component,file,exists,size_kb
0,Main notebook,Project-Offline-RAG-Wikipedia.ipynb,True,594.37
1,Application,src/run_rag.py,True,7.02
2,Data loader,src/rag/data_loader.py,True,1.54
3,Chunking,src/rag/chunking.py,True,1.68
4,Indexer,src/rag/indexer.py,True,3.07
5,Retriever,src/rag/retriever.py,True,2.68
6,Generator,src/rag/generator.py,True,4.57
7,Pipeline,src/rag/pipeline.py,True,3.77
8,Evidence,src/rag/evidence.py,True,1.56
9,Fusion,src/rag/fusion.py,True,2.71



=== DIRECTORY AUDIT ===


,component,directory,exists
0,Wikipedia articles,data/wiki_slice_1000,True
1,Wikipedia passages,data/wiki_chunks_100w_20overlap,True
2,TF-IDF index,data/tfidf_index,True
3,Evaluation dataset,data/evaluation,True
4,Evaluation results,results/evaluation,True
5,Python modules,src/rag,True



=== AUDIT SUMMARY ===
Required files: 24
Files present: 24
Missing files: 0
Required directories: 6
Directories present: 6
Missing directories: 0

Final project file audit: PASSED


In [107]:
# ============================================================
# Stage 9.2.1 — Final Jupyter Notebook Integrity Audit
# ============================================================

import json
import pandas as pd

from pathlib import Path

print("=== FINAL JUPYTER NOTEBOOK AUDIT ===")

# ------------------------------------------------------------
# 1. Locate notebook
# ------------------------------------------------------------

NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "Project-Offline-RAG-Wikipedia.ipynb"
)

assert NOTEBOOK_PATH.is_file(), \
    "Main notebook file not found!"

print("\nNotebook:", NOTEBOOK_PATH.name)

# ------------------------------------------------------------
# 2. Load saved notebook
# ------------------------------------------------------------

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    notebook = json.load(f)

cells = notebook["cells"]

print("\n=== NOTEBOOK STRUCTURE ===")

print("Total cells:", len(cells))

code_cells = [
    cell for cell in cells
    if cell["cell_type"] == "code"
]

markdown_cells = [
    cell for cell in cells
    if cell["cell_type"] == "markdown"
]

print("Code cells:", len(code_cells))
print("Markdown cells:", len(markdown_cells))

# ------------------------------------------------------------
# 3. Identify empty cells
# ------------------------------------------------------------

empty_cells = []

for index, cell in enumerate(cells):

    source = "".join(
        cell.get("source", [])
    ).strip()

    if not source:

        empty_cells.append({
            "cell_index": index,
            "cell_type": cell["cell_type"]
        })

print("\n=== EMPTY CELLS ===")

print("Empty cells:", len(empty_cells))

if empty_cells:
    display(pd.DataFrame(empty_cells))

# ------------------------------------------------------------
# 4. Identify saved execution errors
# ------------------------------------------------------------

error_records = []

for index, cell in enumerate(cells):

    if cell["cell_type"] != "code":
        continue

    for output in cell.get("outputs", []):

        if output.get("output_type") == "error":

            error_records.append({
                "cell_index": index,
                "execution_count": cell.get(
                    "execution_count"
                ),
                "error_type": output.get("ename"),
                "message": output.get("evalue")
            })

print("\n=== SAVED EXECUTION ERRORS ===")

print("Error outputs:", len(error_records))

if error_records:
    display(pd.DataFrame(error_records))

# ------------------------------------------------------------
# 5. Identify code cells without execution count
# ------------------------------------------------------------

unexecuted_cells = []

for index, cell in enumerate(cells):

    if cell["cell_type"] != "code":
        continue

    source = "".join(
        cell.get("source", [])
    ).strip()

    if source and cell.get("execution_count") is None:

        unexecuted_cells.append({
            "cell_index": index,
            "source_preview": source[:100]
        })

print("\n=== UNEXECUTED CODE CELLS ===")

print("Unexecuted code cells:", len(unexecuted_cells))

if unexecuted_cells:
    display(pd.DataFrame(unexecuted_cells))

# ------------------------------------------------------------
# 6. Identify important stage references
# ------------------------------------------------------------

notebook_text = "\n".join(
    "".join(cell.get("source", []))
    for cell in cells
)

print("\n=== STAGE HEADING CHECK ===")

for stage in range(1, 8):

    marker = f"Stage {stage}"

    found = marker.lower() in notebook_text.lower()

    print(
        marker,
        ":",
        "Found" if found else "Not found"
    )

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print("\n=== NOTEBOOK AUDIT SUMMARY ===")

print("Total cells:", len(cells))
print("Code cells:", len(code_cells))
print("Markdown cells:", len(markdown_cells))

print("Empty cells:", len(empty_cells))
print("Saved error outputs:", len(error_records))
print("Unexecuted code cells:", len(unexecuted_cells))

print("\nNotebook integrity inspection completed.")

=== FINAL JUPYTER NOTEBOOK AUDIT ===

Notebook: Project-Offline-RAG-Wikipedia.ipynb

=== NOTEBOOK STRUCTURE ===
Total cells: 111
Code cells: 79
Markdown cells: 32

=== EMPTY CELLS ===
Empty cells: 1


,cell_index,cell_type
0,110,code



=== SAVED EXECUTION ERRORS ===
Error outputs: 3


,cell_index,execution_count,error_type,message
0,8,6,FileNotFoundError,Dataset directory already exists:c:\AshutoshLo...
1,17,11,FileExistsError,Chunk dataset already exists: c:\AshutoshLocal...
2,27,18,FileExistsError,Index artifacts already exist: [WindowsPath('c...



=== UNEXECUTED CODE CELLS ===
Unexecuted code cells: 8


,cell_index,source_preview
0,43,# ============================================...
1,45,# ============================================...
2,46,# How to release GPU memory manually . When yo...
3,49,# ============================================...
4,52,# ============================================...
5,56,# ============================================...
6,58,# ============================================...
7,69,# ============================================...



=== STAGE HEADING CHECK ===
Stage 1 : Found
Stage 2 : Found
Stage 3 : Found
Stage 4 : Found
Stage 5 : Found
Stage 6 : Found
Stage 7 : Found

=== NOTEBOOK AUDIT SUMMARY ===
Total cells: 111
Code cells: 79
Markdown cells: 32
Empty cells: 1
Saved error outputs: 3
Unexecuted code cells: 8

Notebook integrity inspection completed.


In [108]:
# ============================================================
# Stage 9.2.2 — Inspect Flagged Notebook Cells
# ============================================================

print("=== FLAGGED NOTEBOOK CELL INSPECTION ===")

# ------------------------------------------------------------
# 1. Inspect cells containing saved errors
# ------------------------------------------------------------

ERROR_CELL_INDICES = [8, 17, 27]

print("\n=== CELLS WITH SAVED ERRORS ===")

for index in ERROR_CELL_INDICES:

    cell = cells[index]

    source = "".join(
        cell.get("source", [])
    )

    print("\n" + "=" * 70)
    print("Cell index:", index)
    print("Execution count:", cell.get("execution_count"))

    print("\nCELL SOURCE:")
    print(source[:2500])

    print("\nSAVED ERRORS:")

    for output in cell.get("outputs", []):

        if output.get("output_type") == "error":

            print(
                output.get("ename"),
                ":",
                output.get("evalue")
            )

# ------------------------------------------------------------
# 2. Inspect unexecuted code cells
# ------------------------------------------------------------

UNEXECUTED_CELL_INDICES = [
    43, 45, 46, 49, 52, 56, 58, 69
]

print("\n=== UNEXECUTED CODE CELLS ===")

for index in UNEXECUTED_CELL_INDICES:

    cell = cells[index]

    source = "".join(
        cell.get("source", [])
    )

    print("\n" + "=" * 70)
    print("Cell index:", index)

    print("\nCELL SOURCE:")
    print(source[:1800])

print("\nFlagged notebook cell inspection completed.")

=== FLAGGED NOTEBOOK CELL INSPECTION ===

=== CELLS WITH SAVED ERRORS ===

Cell index: 8
Execution count: 6

CELL SOURCE:
# Stage 2.3.2 — Save Wikipedia corpus locally

from pathlib import Path
from datasets import Dataset

# Use the notebook's currnt working directory 
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "wiki_slice_1000"

print("Project working directory:", PROJECT_ROOT)
print("Target corpus directory:", CORPUS_DIR)

# Protect an existing saved dataset from accidental overwriting

if CORPUS_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory already exists:{CORPUS_DIR}"
    )

# Ensure the parent data directpry exists 
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Convert selected records into a Hugging face dataset
wiki_dataset = Dataset.from_list(selected_articles)

# Persist the complete selected dataset
wiki_dataset.save_to_disk(str(CORPUS_DIR))

print("\n=== CORPUS SAVED ===")
print("Articles saved:", len(wiki_da

In [ ]:
# ============================================================
# Stage 9.2.3 — Prepare Clean Submission Notebook
# ============================================================

import json
import copy
import hashlib
from pathlib import Path

print("=== FINAL NOTEBOOK PREPARATION ===")

PROJECT_ROOT = Path.cwd()

SOURCE_NOTEBOOK = (
    PROJECT_ROOT / "Project-Offline-RAG-Wikipedia.ipynb"
)

SUBMISSION_NOTEBOOK = (
    PROJECT_ROOT / "Project-Offline-RAG-Wikipedia-Submission.ipynb"
)

# ------------------------------------------------------------
# Step 1: Safety checks
# ------------------------------------------------------------

if not SOURCE_NOTEBOOK.is_file():
    raise FileNotFoundError(
        f"Original notebook not found: {SOURCE_NOTEBOOK}"
    )

if SUBMISSION_NOTEBOOK.exists():
    raise FileExistsError(
        f"Submission notebook already exists: {SUBMISSION_NOTEBOOK}"
    )

original_bytes = SOURCE_NOTEBOOK.read_bytes()

original_hash = hashlib.sha256(
    original_bytes
).hexdigest()

# ------------------------------------------------------------
# Step 2: Load and copy notebook
# ------------------------------------------------------------

original_notebook = json.loads(
    original_bytes.decode("utf-8")
)

submission_notebook = copy.deepcopy(
    original_notebook
)

cells = submission_notebook["cells"]

print("Original cells:", len(cells))

# ------------------------------------------------------------
# Step 3: Remove saved error outputs
# ------------------------------------------------------------

removed_errors = 0

for cell in cells:

    if cell["cell_type"] != "code":
        continue

    outputs = cell.get("outputs", [])

    error_outputs = [
        output
        for output in outputs
        if output.get("output_type") == "error"
    ]

    removed_errors += len(error_outputs)

    cell["outputs"] = [
        output
        for output in outputs
        if output.get("output_type") != "error"
    ]

    # Do not imply successful execution when only
    # an error was recorded.
    if error_outputs and not cell["outputs"]:
        cell["execution_count"] = None

# ------------------------------------------------------------
# Step 4: Remove all empty cells from submission copy
# ------------------------------------------------------------

original_cell_count = len(cells)

cells[:] = [
    cell
    for cell in cells
    if (
        "".join(cell.get("source", [])).strip()
        if isinstance(cell.get("source", ""), list)
        else cell.get("source", "").strip()
    )
]

removed_empty = original_cell_count - len(cells)

# ------------------------------------------------------------
# Step 5: Correct the original dataset-protection cell only
# ------------------------------------------------------------

corrected_exceptions = 0

for cell in cells:

    if cell["cell_type"] != "code":
        continue

    source = cell.get("source", "")

    was_list = isinstance(source, list)

    source_text = (
        "".join(source)
        if was_list
        else source
    )

    # Restrict correction to the actual dataset creation cell.
    # Exclude this notebook-cleanup code.

    if (
        "if CORPUS_DIR.exists():" in source_text
        and "Dataset directory already exists" in source_text
        and "raise FileNotFoundError(" in source_text
        and "corrected_exceptions" not in source_text
    ):

        corrected_text = source_text.replace(
            "raise FileNotFoundError(",
            "raise FileExistsError(",
            1
        )

        cell["source"] = (
            corrected_text.splitlines(keepends=True)
            if was_list
            else corrected_text
        )

        corrected_exceptions += 1

# ------------------------------------------------------------
# Step 6: Verify cleanup dynamically
# ------------------------------------------------------------

remaining_errors = sum(
    1
    for cell in cells
    for output in cell.get("outputs", [])
    if output.get("output_type") == "error"
)

remaining_empty = sum(
    1
    for cell in cells
    if not (
        "".join(cell.get("source", []))
        if isinstance(cell.get("source", ""), list)
        else cell.get("source", "")
    ).strip()
)

print("\n=== CLEANUP VERIFICATION ===")

print("Original cells:", original_cell_count)
print("Saved errors removed:", removed_errors)
print("Empty cells removed:", removed_empty)
print("Exception corrections:", corrected_exceptions)

print("Remaining saved errors:", remaining_errors)
print("Remaining empty cells:", remaining_empty)

assert remaining_errors == 0
assert remaining_empty == 0
assert corrected_exceptions == 1
assert len(cells) == original_cell_count - removed_empty

print("\nCleanup verification: PASSED")

# ------------------------------------------------------------
# Step 7: Save new notebook without overwriting
# ------------------------------------------------------------

with SUBMISSION_NOTEBOOK.open(
    "x",
    encoding="utf-8"
) as file:

    json.dump(
        submission_notebook,
        file,
        ensure_ascii=False,
        indent=1
    )

# ------------------------------------------------------------
# Step 8: Verify saved notebook and original
# ------------------------------------------------------------

with SUBMISSION_NOTEBOOK.open(
    "r",
    encoding="utf-8"
) as file:

    reloaded = json.load(file)

assert len(reloaded["cells"]) == original_cell_count - removed_empty

current_original_hash = hashlib.sha256(
    SOURCE_NOTEBOOK.read_bytes()
).hexdigest()

assert current_original_hash == original_hash

print("\n=== CLEANUP SUMMARY ===")

print("Original cells:", len(original_notebook["cells"]))
print("Submission cells:", len(reloaded["cells"]))

print("Saved errors removed:", removed_errors)
print("Empty cells removed:", removed_empty)
print("Exception corrections:", corrected_exceptions)

print("\nOriginal notebook unchanged: True")

print("\nSubmission notebook:")
print(SUBMISSION_NOTEBOOK)

print("\nNotebook preparation: PASSED")

=== FINAL NOTEBOOK PREPARATION ===
Original cells: 115

=== CLEANUP VERIFICATION ===
Original cells: 115
Saved errors removed: 4
Empty cells removed: 1
Exception corrections: 2
Remaining saved errors: 0
Remaining empty cells: 0


In [110]:
# ============================================================
# Stage 9.2.3 — Notebook Cleanup Diagnostic
# ============================================================

import json
from pathlib import Path

notebook_path = Path.cwd() / "Project-Offline-RAG-Wikipedia.ipynb"

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook["cells"]

empty_cells = [
    i
    for i, cell in enumerate(cells)
    if not "".join(cell.get("source", [])).strip()
]

print("=== NOTEBOOK CLEANUP DIAGNOSTIC ===")

print("Total cells:", len(cells))
print("Empty cell indices:", empty_cells)

print("\nLast cell:")
print("Type:", cells[-1]["cell_type"])

last_source = "".join(cells[-1].get("source", []))

print("Source preview:", last_source[:150])

print("\nSubmission notebook exists:",
      (Path.cwd() / "Project-Offline-RAG-Wikipedia-Submission.ipynb").exists())

print("\nDiagnostic completed.")

=== NOTEBOOK CLEANUP DIAGNOSTIC ===
Total cells: 114
Empty cell indices: [0]

Last cell:
Type: code
Source preview: # ============================================================
# Stage 9.2.3 — Prepare Clean Submission Notebook
# ===================================

Submission notebook exists: False

Diagnostic completed.
